# Subject-level validation of independent voice, spiral-drawing and T1-MRI classifiers for Parkinson's disease

**Analysis notebook v21 (final)** — accompanies the revised manuscript.

Three public cohorts are analysed **independently** (no patient pairing): UCI Parkinson's voice (195 recordings / 32 identifiers), UCI spiral drawing (40 files + 37 PD-only files) and NEUROCON T1-weighted MRI (43 subjects, downloaded automatically).

### How to run (≈ 3–4 h)
1. **Add Input**: the voice and spiral datasets (any names — paths are found automatically).
2. **Settings**: *Internet ON*, *Accelerator GPU T4*.
3. **Save Version → Save & Run All (Commit)**. The run continues on Kaggle's servers even if your computer or power goes off.
4. Download `PD_Unpaired_Research_v9/model_comparison/PD_revision_package.zip` (results + figures + app) from the version's *Output* tab. `revision/paper_numbers.md` contains every number for the manuscript.

### Structure
- **Part A (cells 1–80)** — data verification, feature extraction, MRI preprocessing (deterministic), single-seed nested selection, retrospective holdout, acquisition control, QC figures.
- **Part R** — revision analyses: repeated nested CV, parallel full-pipeline permutation test, uncertainty, confounding audits, sensitivity analyses, leakage demonstration, publication figures and tables.

Set `MODE='FAST'` in cell **R0** for a ~1.5 h check run.

## Environment and failure reporting

Install only missing dependencies; keep Kaggle PyTorch/CUDA intact. Each section reports success only after completion.


In [7]:
import os
for r,d,f in os.walk('/kaggle/input'):
    if 'parkinsons.data' in f or 'hw_dataset' in d: print(r)

/kaggle/input/datasets/mdnazmulhasan07/voice-dataset/Voice
/kaggle/input/datasets/mdnazmulhasan07/spiral-datasset/Spiral


In [8]:
import sys,os,json,time,subprocess,importlib.util,traceback
from pathlib import Path
RUN_STARTED=time.time()
OUT=Path('/kaggle/working/PD_Unpaired_Research_v9')
for folder in ['tables','models','figures','tools','logs','mri_cache','raw_local_cache']:(OUT/folder).mkdir(parents=True,exist_ok=True)
CELL_STATUS=[]
def complete(cell):
    record=dict(timestamp_utc=__import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),cell=cell,status='SUCCESSFULLY COMPLETED',elapsed_minutes=round((time.time()-RUN_STARTED)/60,2))
    CELL_STATUS.append(record);(OUT/'run_status.json').write_text(json.dumps(CELL_STATUS,indent=2))
    print(f'[{cell}] SUCCESSFULLY COMPLETED | elapsed={record["elapsed_minutes"]:.2f} min',flush=True)
ip=get_ipython()
for callback in list(ip.events.callbacks.get('post_run_cell',[])):
    if getattr(callback,'__name__','') in ['record_cell_failure','v2_record_failure','v3_record_failure','v4_record_failure']:ip.events.unregister('post_run_cell',callback)
def v3_record_failure(result):
    error=result.error_before_exec or result.error_in_exec
    if error is not None:
        if type(error).__name__=='MRIReviewRequired':
            CELL_STATUS.append(dict(cell='MRI_REVIEW_GATE',status='REVIEW_REQUIRED',reason=str(error)))
            (OUT/'run_status.json').write_text(json.dumps(CELL_STATUS,indent=2))
            print('[MRI_REVIEW_GATE] REVIEW_REQUIRED: diagnostic packet is ready; no MRI training has run.',flush=True)
            return
        CELL_STATUS.append(dict(cell='FAILED_CELL',status='FAILED',error=str(error)))
        (OUT/'run_status.json').write_text(json.dumps(CELL_STATUS,indent=2))
        (OUT/'logs/last_error.txt').write_text(str(error))
        print('[FAILED_CELL] FAILED:',str(error),flush=True)
ip.events.register('post_run_cell',v3_record_failure)
# বাংলা: ইনস্টল করা torch/CUDA আপগ্রেড নয়; শুধু অনুপস্থিত dependency।
for module,package in [('nibabel','nibabel'),('SimpleITK','SimpleITK'),('nilearn','nilearn'),('surfa','surfa')]:
    if importlib.util.find_spec(module) is None:
        print('Installing missing package:',package,flush=True)
        subprocess.run([sys.executable,'-m','pip','install','--quiet',package],check=True)
import numpy as np,pandas as pd,torch,matplotlib.pyplot as plt,joblib
print('Accelerator:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU fallback (slower)')
print('Dependency versions:', {n:__import__(n).__version__ for n in ['numpy','pandas','sklearn','torch']})

complete('V9-00')


Accelerator: Tesla T4
Dependency versions: {'numpy': '2.1.3', 'pandas': '2.3.3', 'sklearn': '1.6.1', 'torch': '2.11.0+cu130'}
[V9-00] SUCCESSFULLY COMPLETED | elapsed=0.00 min


## Configuration

Edit DATASET_ROOT only if your Kaggle dataset is mounted elsewhere. Compute limit is a guard, not a runtime guarantee. If the limit is reached, completed checkpoints remain saved; no candidate is silently skipped.


In [9]:
DATASET_ROOT=Path('/kaggle/input/datasets/romanarahman28')
SEED=42
COMPUTE_BUDGET_HOURS=48.0
CNN_MAX_EPOCHS=12
CNN_PATIENCE=3
CNN_SLICES_PER_VIEW=3
FROZEN_SLICES_PER_VIEW=8
BOOTSTRAP_REPEATS=1000
SYNTHETIC_REPEATS=50
class MRIReviewRequired(RuntimeError):
    pass
def require_mri_review():
    if REQUIRE_APPROVED_MRI_QC and not globals().get('QC_APPROVED',False):
        raise MRIReviewRequired('MRI review is pending. Complete the generated diagnostic review before running MRI-dependent cells.')
MRI_MODEL_SCOPE='compact'  # compact=4 prespecified families; expanded keeps legacy candidates.
QC_REVIEW_CSV=None  # Completed template path, e.g. '/kaggle/input/my-review/mri_qc_review.csv'
REQUIRE_APPROVED_MRI_QC=True  # True: enforce human anatomical review before training.
SPIRAL_PERSON_SPLIT_CSV=None  # Optional person_id,split for original cohort; predeclare before training.
SPIRAL_IDENTITY_CSV=None  # Optional CSV: file_subject_id, person_id; one row per source file.
# If repeats are known, provide this mapping. Mapped files are aggregated by person before CV.
SPIRAL_IDENTITY_VERIFIED=False  # Set True only with independent dataset/person documentation.
SPIRAL_IDENTITY_EVIDENCE=''  # Citation/manifest explaining file-to-person mapping.
RESAMPLING_POINTS=256
PERMUTATION_REPEATS=20
REPEATED_CV_REPEATS=5
MRI_SPLIT_OVERRIDE=None
# বাংলা: পুরোনো output প্রয়োজন নেই; একই নতুন project-এর ভেতর preprocessing ও models।
PREVIOUS_OUTPUT=OUT
OUTPUT=OUT/'model_comparison'
for folder in ['tables','models','figures','fit_cache','deployment','logs']:(OUTPUT/folder).mkdir(parents=True,exist_ok=True)
np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
DEVICE='cuda' if torch.cuda.is_available() else 'cpu';N_SLICES_PER_VIEW=FROZEN_SLICES_PER_VIEW
assert COMPUTE_BUDGET_HOURS>0 and CNN_MAX_EPOCHS>=1 and CNN_SLICES_PER_VIEW>=1

def deadline_check():
    if time.time()>RUN_STARTED+COMPUTE_BUDGET_HOURS*3600:
        raise TimeoutError('Compute guard reached. Preserve outputs and resume from checkpoints; no forced success.')
print('Whole-run compute guard:',COMPUTE_BUDGET_HOURS,'hours')
print('No guaranteed accuracy. Retrospective exploratory evaluation only.')
# বাংলা: near-duplicate review-এর প্রমাণিত সিদ্ধান্ত ছাড়া training এগোবে না।
DUPLICATE_REVIEW_CSV=None
NEAR_VECTOR_RELATIVE_TOLERANCE=1e-4
MRI_NEAR_CORRELATION=.9999
MRI_NEAR_NRMSE=.002
RUN_SYNTHETIC_FUSION=False
ALIGNMENT_EPOCHS=120
ALIGNMENT_LAMBDAS=(0.,.01,.1)
print('Unpaired architecture protocol: fixed epochs; nested lambda selection; no clinical fusion claim.')

complete('V9-01')


Whole-run compute guard: 48.0 hours
No guaranteed accuracy. Retrospective exploratory evaluation only.
Unpaired architecture protocol: fixed epochs; nested lambda selection; no clinical fusion claim.
[V9-01] SUCCESSFULLY COMPLETED | elapsed=0.06 min


## Embedded preprocessing and model modules

Both portable modules are included inside this notebook; no separate source file is required.


In [10]:
VALIDATION_SOURCE='"""Input/provenance validation, independent of torch and Streamlit."""\nimport hashlib\nimport numpy as np\nimport pandas as pd\n\ndef validate_qc_review(review,current):\n    fields=[\'subject_id\',\'input_hash\',\'array_hash\',\'protocol_hash\',\'native_mask_hash\',\'registered_mask_hash\',\'native_corrected_hash\']\n    needed=set(fields+[\'decision\',\'reviewer\',\'evidence\'])\n    if not needed<=set(review) or not review.subject_id.is_unique:raise ValueError(\'Invalid QC review schema.\')\n    if set(review.subject_id)!=set(current.subject_id):raise ValueError(\'Review must cover every current subject.\')\n    joined=current[fields].merge(review,on=\'subject_id\',validate=\'one_to_one\',suffixes=(\'_current\',\'_review\'))\n    for key in fields[1:]:\n        if not joined[key+\'_current\'].eq(joined[key+\'_review\']).all():raise ValueError(\'Stale QC review: \'+key+\' does not match current preprocessing.\')\n    d=review.decision.astype(str).str.strip().str.lower()\n    if not d.isin([\'accept\',\'reject\']).all():raise ValueError(\'Each QC decision must be accept/reject.\')\n    for key in [\'reviewer\',\'evidence\']:\n        if review[key].isna().any() or review[key].astype(str).str.strip().eq(\'\').any():raise ValueError(\'Nonempty QC \'+key+\' required.\')\n    if d.eq(\'reject\').any():raise ValueError(\'Rejected scan: correct preprocessing and obtain a revised review; no automatic deletion.\')\n    return True\n\ndef validate_mri_metadata(metadata,payload,expected_protocol_hash):\n    if metadata.get(\'protocol_hash\')!=expected_protocol_hash:raise ValueError(\'MRI preprocessing protocol mismatch.\')\n    if metadata.get(\'array_hash\')!=hashlib.sha256(payload).hexdigest():raise ValueError(\'MRI NPY bytes do not match metadata.\')\n    if metadata.get(\'array_order\')!=\'z,y,x\':raise ValueError(\'Expected trained z,y,x array ordering.\')\n    return True\n\ndef validate_feature_frame(frame,features,name):\n    if frame.empty or frame.columns.duplicated().any():raise ValueError(\'Nonempty input with unique columns required.\')\n    if not set(features)<=set(frame):raise ValueError(\'Missing required feature columns.\')\n    if name==\'voice\':\n        if \'subject_id\' not in frame or frame.subject_id.isna().any() or frame.subject_id.astype(str).str.strip().eq(\'\').any() or frame.subject_id.nunique()!=1:raise ValueError(\'Exactly one nonempty voice subject_id is required.\')\n        frame=frame.drop_duplicates(subset=[\'subject_id\']+list(features))\n    elif len(frame)!=1:raise ValueError(\'One extracted Spiral subject row is required.\')\n    values=frame[list(features)].apply(pd.to_numeric,errors=\'raise\').to_numpy(float)\n    if np.isinf(values).any():raise ValueError(\'Infinite features are invalid.\')\n    if name==\'voice\' and not np.isfinite(values).all():raise ValueError(\'Voice features must be finite.\')\n    if np.isnan(values).all():raise ValueError(\'ABSTAIN: no usable input features.\')\n    return frame,values\n'
SPIRAL_SOURCE='"""Segment-aware spiral features. Resampling is a sensitivity choice, not proof of deconfounding."""\nimport numpy as np\nSHAPE_FEATURE_NAMES=[\'path_efficiency\',\'step_cv\',\'abs_turn_median\',\'abs_turn_p90\',\'radius_cv\',\'turn_consistency\']\ndef shape_features(frame,n_points=256):\n    result={};quality=[]\n    for task in (0,1):\n        values={k:np.nan for k in SHAPE_FEATURE_NAMES}\n        # Preserve original order; never join across another task or a backwards timestamp.\n        ids=np.flatnonzero(frame.TestID.to_numpy(int)==task)\n        blocks=[]\n        if len(ids):\n            ts=frame.Timestamp.to_numpy(float)\n            cut=np.flatnonzero((np.diff(ids)!=1)|(np.diff(ts[ids])<0))+1\n            for index in np.split(ids,cut):\n                xy=frame.iloc[index][[\'X\',\'Y\']].to_numpy(float)\n                if len(xy)<5:continue\n                keep=np.r_[True,np.linalg.norm(np.diff(xy,axis=0),axis=1)>1e-10];xy=xy[keep]\n                if len(xy)<5:continue\n                arc=np.r_[0,np.cumsum(np.linalg.norm(np.diff(xy,axis=0),axis=1))]\n                if arc[-1]<=1e-8:continue\n                grid=np.linspace(0,arc[-1],n_points)\n                xy=np.column_stack([np.interp(grid,arc,xy[:,k]) for k in (0,1)])\n                xy-=xy.mean(0);scale=np.sqrt(np.mean(xy**2))\n                if scale<=1e-8:continue\n                xy/=scale;delta=np.diff(xy,axis=0);lengths=np.linalg.norm(delta,axis=1)\n                delta=delta[lengths>1e-8];lengths=lengths[lengths>1e-8]\n                if len(delta)<5:continue\n                turns=np.arctan2(delta[:-1,0]*delta[1:,1]-delta[:-1,1]*delta[1:,0],(delta[:-1]*delta[1:]).sum(1))\n                radius=np.linalg.norm(xy,axis=1)\n                v=[np.linalg.norm(xy[-1]-xy[0])/lengths.sum(),lengths.std()/max(lengths.mean(),1e-8),np.median(np.abs(turns)),np.quantile(np.abs(turns),.9),radius.std()/max(radius.mean(),1e-8),abs(turns.sum())/max(np.abs(turns).sum(),1e-8)]\n                if np.isfinite(v).all():blocks.append(v)\n        if blocks:values=dict(zip(SHAPE_FEATURE_NAMES,np.mean(blocks,axis=0)))\n        result.update({f\'shape_{task}_{k}\':float(v) for k,v in values.items()})\n        quality.append(dict(task=task,point_count=len(ids),usable_segments=len(blocks),status=\'VALID\' if blocks else \'INSUFFICIENT_USABLE_GEOMETRY\'))\n    return result,quality\n'
PREPROCESSING_SOURCE="from pathlib import Path\nimport json, hashlib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\nCOLUMNS=['X','Y','Z','Pressure','GripAngle','Timestamp','TestID']\nFEATURE_NAMES=['n_points','x_range','y_range','path_length','median_step','median_speed','p90_speed','pressure_mean','pressure_std','pressure_median','grip_mean','grip_std']\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, 'rb') as f:\n        for b in iter(lambda: f.read(2 ** 20), b''):\n            h.update(b)\n    return h.hexdigest()\n\ndef write_json(path, obj):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(obj, indent=2, default=lambda v: v.item() if hasattr(v, 'item') else str(v)), encoding='utf-8')\n\ndef assign_split(df, name, out, seed=42):\n    known = {'voice': ['phon_R01_S05', 'phon_R01_S16', 'phon_R01_S17', 'phon_R01_S18', 'phon_R01_S21', 'phon_R01_S33', 'phon_R01_S37', 'phon_R01_S49'], 'spiral': ['Healthy/C_0004', 'Healthy/C_0007', 'Healthy/C_0013', 'Parkinson/P_05060003', 'Parkinson/P_11120003', 'Parkinson/P_26060007', 'Parkinson/P_27110001', 'Parkinson/P_30100001']}\n    df = df.sort_values('subject_id').reset_index(drop=True).copy()\n    if name in known:\n        assert set(known[name]) <= set(df.subject_id), 'Known holdout IDs missing; inspect dataset before changing protocol.'\n        test_ids = known[name]\n        status = 'Previously evaluated holdout; retrospective reuse; no fresh-test claim'\n    else:\n        _, te = train_test_split(np.arange(len(df)), test_size=11, random_state=seed, stratify=df.label)\n        test_ids = df.subject_id.iloc[te].tolist()\n        status = 'New retrospective label-stratified MRI split; not the original label-plus-QC split'\n    df['split'] = np.where(df.subject_id.isin(test_ids), 'test', 'development')\n    df[['subject_id', 'label', 'split']].to_csv(out / f'{name}_split.csv', index=False)\n    write_json(out / f'{name}_evaluation_status.json', {'status': status, 'external_validation': False})\n    return df\n\ndef empty_features():\n    return {feature: np.nan for feature in FEATURE_NAMES}\n\ndef load_trajectory(filepath):\n    data = pd.read_csv(filepath, sep=';', header=None, encoding='utf-8-sig')\n    if data.empty:\n        raise ValueError(f'Empty trajectory: {filepath.name}')\n    if data.shape[1] != 7:\n        raise ValueError(f'Unexpected columns: {filepath.name}')\n    data.columns = COLUMNS\n    for col in COLUMNS:\n        data[col] = pd.to_numeric(data[col], errors='raise')\n    if not np.isfinite(data[COLUMNS].to_numpy(dtype=float)).all():\n        raise ValueError(f'Invalid numeric data: {filepath.name}')\n    if not data['TestID'].isin([0, 1, 2]).all():\n        raise ValueError(f'Unexpected Test ID: {filepath.name}')\n    data['TestID'] = data['TestID'].astype(int)\n    return data\n\ndef extract_test_features(data, test_id):\n    subset = data[data['TestID'] == test_id]\n    if subset.empty:\n        return empty_features()\n    x = subset['X'].to_numpy(dtype=float)\n    y = subset['Y'].to_numpy(dtype=float)\n    pressure = subset['Pressure'].to_numpy(dtype=float)\n    grip = subset['GripAngle'].to_numpy(dtype=float)\n    all_x = data['X'].to_numpy(dtype=float)\n    all_y = data['Y'].to_numpy(dtype=float)\n    all_t = data['Timestamp'].to_numpy(dtype=float)\n    all_test = data['TestID'].to_numpy(dtype=int)\n    dx = np.diff(all_x)\n    dy = np.diff(all_y)\n    dt = np.diff(all_t)\n    distances = np.hypot(dx, dy)\n    same_test = (all_test[:-1] == test_id) & (all_test[1:] == test_id)\n    valid_time = same_test & (dt > 0)\n    valid_distances = distances[valid_time]\n    valid_dt = dt[valid_time]\n    speeds = valid_distances / valid_dt * 1000.0 if len(valid_dt) > 0 else np.array([])\n    features = {'n_points': int(len(subset)), 'x_range': float(np.ptp(x)), 'y_range': float(np.ptp(y)), 'path_length': float(np.sum(valid_distances)), 'median_step': float(np.median(valid_distances)) if len(valid_distances) > 0 else np.nan, 'median_speed': float(np.median(speeds)) if len(speeds) > 0 else np.nan, 'p90_speed': float(np.percentile(speeds, 90)) if len(speeds) > 0 else np.nan, 'pressure_mean': float(np.mean(pressure)), 'pressure_std': float(np.std(pressure)), 'pressure_median': float(np.median(pressure)), 'grip_mean': float(np.mean(grip)), 'grip_std': float(np.std(grip))}\n    return features\n\ndef timestamp_quality(data):\n    timestamps = data['Timestamp'].to_numpy(dtype=float)\n    test_ids = data['TestID'].to_numpy(dtype=int)\n    dt = np.diff(timestamps)\n    same_test = test_ids[:-1] == test_ids[1:]\n    return {'negative_within_test': int(np.sum((dt < 0) & same_test)), 'negative_between_tests': int(np.sum((dt < 0) & ~same_test)), 'zero_within_test': int(np.sum((dt == 0) & same_test)), 'valid_time_intervals': int(np.sum((dt > 0) & same_test))}\n\ndef trajectory_features(path):\n    data = load_trajectory(path)\n    record = {}\n    for test_id in [0, 1, 2]:\n        record.update({f'test_{test_id}_{k}': v for k, v in extract_test_features(data, test_id).items()})\n    t = data.Timestamp.to_numpy(float)\n    dt = np.diff(t)\n    same = np.diff(data.TestID.to_numpy(int)) == 0\n    positive = dt[(dt > 0) & same]\n    audit = dict(n_points=len(data), median_dt=float(np.median(positive)) if len(positive) else np.nan, zero_dt_fraction=float(np.mean(dt == 0)), negative_dt_count=int((dt < 0).sum()))\n    return (record, audit)\n\ndef slice_images(volume, n_slices=8):\n    import torch\n    import torch.nn.functional as F\n    a = np.asarray(volume, np.float32)\n    assert a.ndim == 3 and np.isfinite(a).all()\n    images = []\n    for axis in range(3):\n        for frac in np.linspace(0.25, 0.75, n_slices):\n            i = int(round((a.shape[axis] - 1) * frac))\n            s = np.take(a, i, axis=axis).copy()\n            t = torch.from_numpy(s)[None, None]\n            t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)[0].repeat(3, 1, 1)\n            images.append(t)\n    x = torch.stack(images)\n    return (x - torch.tensor([0.485, 0.456, 0.406])[None, :, None, None]) / torch.tensor([0.229, 0.224, 0.225])[None, :, None, None]\n\ndef encode_volume(volume, encoder, device, n_slices=8):\n    import torch\n    slices = slice_images(volume, n_slices).to(device)\n    features = []\n    encoder.eval()\n    with torch.inference_mode():\n        for batch in slices.split(16):\n            features.append(encoder(batch).float().cpu().numpy())\n    z = np.concatenate(features).reshape(3, n_slices, -1)\n    return np.concatenate([z.mean(axis=1).ravel(), z.std(axis=1).ravel()]).astype(np.float32)"
MODEL_SOURCE='_MRI_MODEL_SCOPE=\'compact\'\n\nfrom pathlib import Path\nimport time,json,hashlib,gc,copy\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom scipy.ndimage import zoom\nfrom scipy.special import expit\nfrom scipy.stats import beta\nfrom sklearn.base import clone,BaseEstimator,TransformerMixin\nfrom sklearn.feature_selection import SelectPercentile,f_classif\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,VotingClassifier\nfrom sklearn.model_selection import StratifiedKFold,train_test_split\nfrom sklearn.calibration import calibration_curve\nfrom sklearn.metrics import (accuracy_score,balanced_accuracy_score,confusion_matrix,\n precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,\n matthews_corrcoef,brier_score_loss,log_loss,roc_curve,precision_recall_curve)\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for block in iter(lambda:f.read(2**20),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef write_json(path,value):\n    Path(path).parent.mkdir(parents=True,exist_ok=True)\n    Path(path).write_text(json.dumps(value,indent=2,default=lambda v:v.item() if hasattr(v,\'item\') else str(v)))\n\ndef guard(cfg):\n    if time.time()>cfg[\'deadline_unix\']:\n        raise TimeoutError(\'Compute budget reached. Completed fits are cached. Resume with the same configuration and output folder.\')\n\ndef grids(name):\n    if name==\'mri\' and _MRI_MODEL_SCOPE==\'compact\':\n        return {\'Voxel_LR\':[dict(C=c) for c in (.01,.1,1.)],\n                \'Voxel_SVM\':[dict(C=c) for c in (.1,1.)],\n                \'MedicalNet_Global_LR\':[dict(C=c,percentile=100) for c in (.01,.1,1.)],\n                \'MedicalNet_Regional_LR\':[dict(C=c,percentile=100) for c in (.01,.1,1.)]}\n    if name in [\'voice\',\'spiral\']:\n        base={\n            \'LogisticRegression\':[dict(C=c,percentile=p) for c in (.1,1.,10.) for p in (50,100)],\n            \'RBF_SVM\':[dict(C=c,percentile=p) for c in (1.,10.) for p in (50,100)],\n            \'RandomForest\':[dict(max_depth=d) for d in (3,5)],\n            \'GradientBoosting\':[dict(max_depth=d,learning_rate=.05) for d in (1,2)],\n            \'SoftVoting\':[dict(C=1.)]}\n        if name==\'spiral\':\n            base={family:[dict(param,columns=cols,task_mode=mode) for param in params for mode,cols in [(\'both\',None),(\'static\',list(range(6))),(\'dynamic\',list(range(6,12)))]] for family,params in base.items()}\n        return base\n    standard = {\n        \'Voxel_SVM\':[dict(C=c) for c in (.1,1.)],\n        \'Frozen_ResNet_LR\':[dict(C=c) for c in (.1,1.)],\n        \'Frozen_ResNet_SVM\':[dict(C=c) for c in (.1,1.)],\n        \'Spatial_LR\':[dict(C=c,percentile=p) for c in (.1,1.,10.) for p in (50,100)],\n        \'Spatial_SVM\':[dict(C=c,percentile=p) for c in (1.,10.) for p in (50,100)],\n        \'FineTune_ResNet18\':[dict(lr=.0001)]}\n    standard.update({family:[dict(C=c,percentile=p) for c in (.01,.1,1.) for p in (50,100)] for family in [\'MedicalNet_Global_LR\',\'MedicalNet_Global_SVM\',\'MedicalNet_Regional_LR\',\'MedicalNet_Regional_SVM\']})\n    return standard\n\ndef make_pipeline(family,param,seed):\n    if family.startswith(\'MedicalNet_\'):family=\'Frozen_ResNet_LR\' if family.endswith(\'_LR\') else \'Frozen_ResNet_SVM\'\n    if family in [\'LogisticRegression\',\'Frozen_ResNet_LR\',\'Spatial_LR\',\'Voxel_LR\']:\n        model=LogisticRegression(C=param.get(\'C\',1.),max_iter=4000,random_state=seed)\n    elif family in [\'RBF_SVM\',\'Voxel_SVM\',\'Frozen_ResNet_SVM\',\'Spatial_SVM\']:\n        model=SVC(C=param.get(\'C\',1.),kernel=\'rbf\',gamma=\'scale\',probability=False,random_state=seed)\n    elif family==\'RandomForest\':\n        model=RandomForestClassifier(n_estimators=200,max_depth=param[\'max_depth\'],min_samples_leaf=2,n_jobs=2,random_state=seed)\n    elif family==\'GradientBoosting\':\n        model=GradientBoostingClassifier(n_estimators=100,max_depth=param[\'max_depth\'],learning_rate=param[\'learning_rate\'],random_state=seed)\n    elif family==\'SoftVoting\':\n        model=VotingClassifier(estimators=[(\'lr\',LogisticRegression(C=param.get(\'C\',1.),max_iter=4000,random_state=seed)),(\'rf\',RandomForestClassifier(n_estimators=200,max_depth=5,min_samples_leaf=2,n_jobs=2,random_state=seed)),(\'gb\',GradientBoostingClassifier(n_estimators=100,max_depth=1,learning_rate=.05,random_state=seed))],voting=\'soft\',n_jobs=1)\n    else:raise ValueError(\'Unknown family \'+family)\n    return Pipeline([\n        (\'columns\',FeatureColumns(param.get(\'columns\'))),\n        (\'impute\',SimpleImputer(strategy=\'median\',keep_empty_features=True)),\n        (\'scale\',StandardScaler()),\n        (\'selection\',SelectPercentile(safe_f_classif,percentile=param.get(\'percentile\',100))),\n        (\'pca\',PCA(n_components=.95,svd_solver=\'full\') if family.startswith((\'Voxel_\',\'Frozen_\')) else \'passthrough\'),\n        (\'model\',model)])\n\ndef model_score(model,X):\n    classes=np.asarray(model.classes_)\n    if not np.array_equal(classes,[0,1]):raise ValueError(\'Expected classes [0, 1]; PD=1.\')\n    if hasattr(model,\'predict_proba\'):\n        return model.predict_proba(X)[:,int(np.flatnonzero(classes==1)[0])]\n    # বাংলা: sigmoid শুধু bounded score; calibrated probability দাবি করা হবে না।\n    return expit(model.decision_function(X))\n\ndef voxel_vector(volume,shape=(12,14,12)):\n    a=np.asarray(volume,np.float32)\n    assert a.ndim==3 and np.isfinite(a).all()\n    b=zoom(a,np.array(shape)/np.array(a.shape),order=1,prefilter=False)\n    assert b.shape==tuple(shape)\n    return b.ravel().astype(np.float32)\n\nclass FineTuneMRI:\n    """Patient-level multi-view fine-tuning with train-only early stopping and refit."""\n    def __init__(self,checkpoint,cfg,lr=.0001,seed=42):\n        self.checkpoint=str(checkpoint);self.cfg=dict(cfg);self.lr=lr;self.seed=seed\n        self.classes_=np.array([0,1]);self.history_=[];self.selected_epochs_=None\n    def _init(self):\n        import torch\n        from torchvision.models import resnet18\n        torch.manual_seed(self.seed)\n        if torch.cuda.is_available():torch.cuda.manual_seed_all(self.seed)\n        self.encoder=resnet18(weights=None);self.encoder.fc=torch.nn.Identity()\n        self.encoder.load_state_dict(torch.load(self.checkpoint,map_location=\'cpu\',weights_only=True))\n        for p in self.encoder.parameters():p.requires_grad=False\n        for p in self.encoder.layer4.parameters():p.requires_grad=True\n        self.head=torch.nn.Sequential(torch.nn.Dropout(.3),torch.nn.Linear(512,1))\n    def _batch(self,paths,device,training=False):\n        import torch\n        import torch.nn.functional as F\n        xs=[]\n        for path in paths:\n            a=np.load(str(path),allow_pickle=False)\n            # বাংলা: সব view/slice একই patient-এর; একই split-এ থাকবে।\n            x=slice_images(a,self.cfg[\'cnn_slices_per_view\'])\n            x=F.interpolate(x,size=(160,160),mode=\'bilinear\',align_corners=False)\n            xs.append(x)\n        return torch.stack(xs).to(device)\n    def _forward(self,x):\n        n,s,c,h,w=x.shape\n        z=self.encoder(x.reshape(n*s,c,h,w)).reshape(n,s,-1).mean(dim=1)\n        return self.head(z).flatten()\n    def _train(self,paths,y,epochs,val_paths=None,val_y=None,early=False):\n        import torch\n        device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n        # CPU fallback is valid; expected to be much slower than GPU.\n        self.encoder.to(device);self.head.to(device)\n        params=[p for p in self.encoder.parameters() if p.requires_grad]+list(self.head.parameters())\n        optimizer=torch.optim.AdamW(params,lr=self.lr,weight_decay=.001)\n        n0=max(1,int((y==0).sum()));n1=max(1,int((y==1).sum()))\n        criterion=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(n0/n1,device=device))\n        rng=np.random.default_rng(self.seed);best=float(\'inf\');bad=0;selected=epochs\n        for epoch in range(1,epochs+1):\n            guard(self.cfg);self.encoder.eval();self.encoder.layer4.train();self.head.train()\n            # বাংলা: অল্প patient-এ BatchNorm running statistics freeze থাকবে।\n            for layer in self.encoder.modules():\n                if isinstance(layer,torch.nn.modules.batchnorm._BatchNorm):layer.eval()\n            order=rng.permutation(len(paths));losses=[]\n            for k in range(0,len(order),2):\n                guard(self.cfg);idx=order[k:k+2];x=self._batch(paths[idx],device,True)\n                target=torch.tensor(y[idx],dtype=torch.float32,device=device)\n                optimizer.zero_grad(set_to_none=True);loss=criterion(self._forward(x),target)\n                loss.backward();torch.nn.utils.clip_grad_norm_(params,1.);optimizer.step();losses.extend([float(loss.detach().cpu())]*len(idx))\n            record={\'epoch\':epoch,\'train_loss\':float(np.mean(losses))}\n            if val_paths is not None:\n                self.encoder.eval();self.head.eval();v_losses=[]\n                with torch.inference_mode():\n                    for k in range(0,len(val_paths),2):\n                        x=self._batch(val_paths[k:k+2],device)\n                        target=torch.tensor(val_y[k:k+2],dtype=torch.float32,device=device)\n                        v_losses.extend([float(criterion(self._forward(x),target).cpu())]*len(target))\n                v=float(np.mean(v_losses));record[\'validation_loss\']=v\n                if v<best-1e-4:best=v;selected=epoch;bad=0\n                else:bad+=1\n            self.history_.append(record)\n            print(f\'    CNN epoch {epoch}/{epochs} | loss={record["train_loss"]:.4f}\'+(f\' | validation_loss={record["validation_loss"]:.4f}\' if \'validation_loss\' in record else \'\'),flush=True)\n            if early and bad>=self.cfg[\'cnn_patience\']:break\n        self.encoder.cpu();self.head.cpu();torch.cuda.empty_cache();return selected\n    def fit(self,paths,y):\n        paths=np.asarray(paths,str);y=np.asarray(y,int)\n        assert np.bincount(y,minlength=2).min()>=3,\'Too few MRI training subjects for early stopping.\'\n        tr,va=train_test_split(np.arange(len(y)),test_size=max(2,int(round(len(y)*.25))),random_state=self.seed,stratify=y)\n        self._init()\n        self.selected_epochs_=self._train(paths[tr],y[tr],self.cfg[\'cnn_max_epochs\'],paths[va],y[va],True)\n        self.early_stopping_history_=list(self.history_)\n        # বাংলা: epoch নির্ধারণের পরে সব fit-input patient দিয়ে পুনরায় train।\n        self._init();self.history_=[]\n        self._train(paths,y,self.selected_epochs_)\n        self.refit_history_=list(self.history_);return self\n    def predict_proba(self,paths):\n        import torch\n        device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n        self.encoder.to(device).eval();self.head.to(device).eval();scores=[]\n        paths=np.asarray(paths,str)\n        with torch.inference_mode():\n            for k in range(0,len(paths),2):\n                x=self._batch(paths[k:k+2],device);scores.extend(torch.sigmoid(self._forward(x)).cpu().numpy())\n        self.encoder.cpu();self.head.cpu()\n        if device==\'cuda\':torch.cuda.empty_cache()\n        p=np.asarray(scores);return np.column_stack([1-p,p])\n\ndef training_input(spec,ids,family):\n    ids=list(ids);frame=spec[\'subjects\'].set_index(\'subject_id\').loc[ids]\n    if spec[\'name\']==\'voice\':\n        obs=spec[\'observations\'][spec[\'observations\'].subject_id.isin(ids)].copy()\n        X=obs[spec[\'features\']].to_numpy(float);y=obs.label.to_numpy(int)\n        counts=obs.groupby(\'subject_id\').size();class_counts=frame.label.value_counts()\n        # বাংলা: বেশি recording থাকা patient অতিরিক্ত weight পাবে না।\n        weights=np.array([1/(counts[s]*class_counts[l]) for s,l in zip(obs.subject_id,y)])\n        weights/=weights.mean();return X,y,weights\n    y=frame.label.to_numpy(int)\n    if family==\'FineTune_ResNet18\':return frame.array_path.to_numpy(str),y,None\n    columns=feature_columns(spec,family)\n    X=frame[columns].to_numpy(float)\n    counts=pd.Series(y).value_counts();weights=np.array([len(y)/(2*counts[l]) for l in y])\n    return X,y,weights\n\ndef predict_subjects(spec,ids,bundle):\n    ids=list(ids);family=bundle[\'family\'];model=bundle[\'model\']\n    if spec[\'name\']==\'voice\':\n        obs=spec[\'observations\'][spec[\'observations\'].subject_id.isin(ids)].copy()\n        obs[\'score\']=model_score(model,obs[spec[\'features\']].to_numpy(float))\n        p=obs.groupby(\'subject_id\').score.mean().reindex(ids).to_numpy()\n    else:\n        frame=spec[\'subjects\'].set_index(\'subject_id\').loc[ids]\n        if family==\'FineTune_ResNet18\':p=model.predict_proba(frame.array_path.to_numpy(str))[:,1]\n        else:\n            cols=feature_columns(spec,family)\n            p=model_score(model,frame[cols].to_numpy(float))\n    assert np.isfinite(p).all() and len(p)==len(ids)\n    return p\n\ndef fit_candidate(spec,ids,family,param,cfg,seed):\n    # বাংলা: cache identity-তে training subjects, config ও source signature থাকবে।\n    signature={k:v for k,v in cfg.items() if k not in [\'deadline_unix\',\'max_compute_hours\']}\n    X,y,weights=training_input(spec,ids,family)\n    training_label_hash=hashlib.sha256(np.ascontiguousarray(y).tobytes()).hexdigest()\n    key=hashlib.sha256(json.dumps(dict(training_label_hash=training_label_hash,dataset=spec[\'signature\'],features=spec.get(\'features\'),voxel_features=spec.get(\'voxel_features\'),spatial_features=spec.get(\'spatial_features\'),ids=sorted(ids),family=family,param=param,cfg=signature,seed=seed),sort_keys=True).encode()).hexdigest()\n    cache=Path(cfg[\'output\'])/\'fit_cache\'/f\'{key}.joblib\'\n    if cache.exists():\n        cached=joblib.load(cache)\n        assert set(cached[\'training_subjects\'])==set(ids)\n        return cached\n    guard(cfg)\n    if family==\'FineTune_ResNet18\':\n        model=FineTuneMRI(spec[\'checkpoint\'],cfg,param[\'lr\'],seed).fit(X,y)\n    else:\n        model=make_pipeline(family,param,seed);model.fit(X,y,model__sample_weight=weights)\n    bundle=dict(model=model,family=family,param=param,training_subjects=sorted(ids),dataset=spec[\'name\'],fit_cache_key=key,seed=seed,score_calibrated=False)\n    temporary=cache.with_suffix(\'.partial\');joblib.dump(bundle,temporary);temporary.replace(cache)\n    return bundle\n\ndef choose_key(record):\n    return (record[\'inner_balanced_accuracy\'],record[\'inner_roc_auc\'],-abs(record.get(\'threshold\',.5)-.5))\n\ndef tune_family(spec,train_ids,family,params,cfg,seed):\n    frame=spec[\'subjects\'].set_index(\'subject_id\').loc[list(train_ids)].reset_index()\n    folds=min(3,int(frame.label.value_counts().min()));assert folds>=2\n    splits=list(StratifiedKFold(folds,shuffle=True,random_state=seed).split(frame,frame.label))\n    thresholds=np.asarray(cfg.get(\'mri_threshold_grid\',[.5]) if spec[\'name\']==\'mri\' else cfg.get(\'threshold_grid\',[.3,.35,.4,.45,.5,.55,.6,.65,.7]),float)\n    if not len(thresholds) or not np.isfinite(thresholds).all() or np.any((thresholds<=0)|(thresholds>=1)):raise ValueError(\'Invalid prespecified threshold grid.\')\n    records=[]\n    for param in params:\n        parts=[];aucs=[]\n        for k,(tr,va) in enumerate(splits):\n            trids=frame.subject_id.iloc[tr].tolist();vaids=frame.subject_id.iloc[va].tolist()\n            assert set(trids).isdisjoint(vaids)\n            fit=fit_candidate(spec,trids,family,param,cfg,seed+k+100)\n            p=predict_subjects(spec,vaids,fit);y=frame.label.iloc[va].to_numpy(int)\n            parts.append((y,p));aucs.append(roc_auc_score(y,p))\n        values=[float(np.mean([balanced_accuracy_score(y,p>=threshold) for y,p in parts])) for threshold in thresholds]\n        index=max(range(len(thresholds)),key=lambda j:(values[j],-abs(thresholds[j]-.5)))\n        records.append(dict(family=family,param=param,threshold=float(thresholds[index]),inner_balanced_accuracy=values[index],inner_roc_auc=float(np.mean(aucs)),threshold_candidates=thresholds.tolist(),inner_ba_by_threshold=values,interpretation=\'Joint parameter/threshold tuning; inner score optimistic, outer evaluation required\'))\n    return max(records,key=choose_key),records\n\ndef compare_development(spec,cfg):\n    name=spec[\'name\'];out=Path(cfg[\'output\']);table=out/\'tables\';models=out/\'models\'\n    dev=spec[\'subjects\'].query("split==\'development\'").reset_index(drop=True)\n    assert dev.subject_id.is_unique and set(dev.label)=={0,1}\n    if dev.label.value_counts().min()<6:raise ValueError(\'Need >=6 development subjects per class for this nested protocol.\')\n    candidates=grids(name);fold_records=[];inner_records=[];outer_choices=[]\n    scores={f:np.full(len(dev),np.nan) for f in candidates};cuts={f:np.full(len(dev),np.nan) for f in candidates}\n    selected_p=np.full(len(dev),np.nan);selected_t=np.full(len(dev),np.nan)\n    for fold,(tr,va) in enumerate(StratifiedKFold(3,shuffle=True,random_state=cfg[\'seed\']).split(dev,dev.label),1):\n        trids=dev.subject_id.iloc[tr].tolist();vaids=dev.subject_id.iloc[va].tolist();winners=[]\n        for family,params in candidates.items():\n            print(f\'{name.upper()} | outer {fold}/3 | {family} | feature/config/threshold tuning\',flush=True)\n            winner,records=tune_family(spec,trids,family,params,cfg,cfg[\'seed\']+fold)\n            inner_records.extend(dict(stage=f\'outer_{fold}\',**r) for r in records)\n            bundle=fit_candidate(spec,trids,family,winner[\'param\'],cfg,cfg[\'seed\']+fold+500)\n            p=predict_subjects(spec,vaids,bundle);t=winner[\'threshold\'];scores[family][va]=p;cuts[family][va]=t\n            m=score(dev.label.iloc[va],p,t);train_m=score(dev.label.iloc[tr],predict_subjects(spec,trids,bundle),t)\n            fold_records.append(dict(fold=fold,family=family,selected_param=json.dumps(winner[\'param\'],sort_keys=True),training_accuracy=train_m[\'accuracy\'],training_balanced_accuracy=train_m[\'balanced_accuracy\'],**m));winners.append(winner)\n        choice=max(winners,key=choose_key);selected_p[va]=scores[choice[\'family\']][va];selected_t[va]=choice[\'threshold\'];outer_choices.append(dict(fold=fold,**choice))\n        pd.DataFrame(fold_records).to_csv(table/f\'{name}_candidate_outer_folds.csv\',index=False)\n    pd.DataFrame(inner_records).to_csv(table/f\'{name}_inner_search.csv\',index=False);write_json(table/f\'{name}_outer_selection.json\',outer_choices)\n    comparison=[]\n    for family in candidates:\n        p=scores[family];t=cuts[family];assert np.isfinite(p).all() and np.isfinite(t).all()\n        pred=dev[[\'subject_id\',\'label\']].copy();pred[\'probability_pd\']=p;pred[\'decision_threshold\']=t;pred[\'prediction_pd\']=(p>=t).astype(int)\n        pred.to_csv(table/f\'{name}_{family}_oof.csv\',index=False);comparison.append(dict(family=family,**score(dev.label,p,t)))\n    comparison=pd.DataFrame(comparison).sort_values(\'balanced_accuracy\',ascending=False)\n    comparison.to_csv(table/f\'{name}_four_model_comparison.csv\',index=False)\n    comparison.to_csv(table/f\'{name}_candidate_comparison.csv\',index=False)\n    selected_oof=dev[[\'subject_id\',\'label\']].copy();selected_oof[\'probability_pd\']=selected_p;selected_oof[\'decision_threshold\']=selected_t;selected_oof[\'prediction_pd\']=(selected_p>=selected_t).astype(int)\n    assert np.isfinite(selected_p).all() and np.isfinite(selected_t).all()\n    selected_oof.to_csv(table/f\'{name}_selection_procedure_oof.csv\',index=False)\n    write_json(table/f\'{name}_selection_procedure_metrics.json\',score(dev.label,selected_p,selected_t))\n    final_choices=[];records=[]\n    for family,params in candidates.items():\n        winner,search=tune_family(spec,dev.subject_id.tolist(),family,params,cfg,cfg[\'seed\']+900)\n        final_choices.append(winner);records.extend(search)\n    final_choice=max(final_choices,key=choose_key)\n    pd.DataFrame(records).to_csv(table/f\'{name}_final_development_search.csv\',index=False)\n    final=fit_candidate(spec,dev.subject_id.tolist(),final_choice[\'family\'],final_choice[\'param\'],cfg,cfg[\'seed\']+1000)\n    final=dict(final,features=feature_columns(spec,final_choice[\'family\']),threshold=final_choice[\'threshold\'],score_calibrated=False)\n    joblib.dump(final,models/f\'{name}_selected_bundle.joblib\')\n    write_json(table/f\'{name}_selection_lock.json\',dict(**final_choice,criterion=\'Inner subject balanced accuracy then AUC; threshold tie closest to 0.5\',test_used=False,protocol=\'Retrospective V7; previous test outcomes known\'))\n    print(comparison[[\'family\',\'accuracy\',\'balanced_accuracy\',\'roc_auc\',\'specificity\']].round(4).to_string(index=False))\n    print(name.upper(),\'| LOCKED:\',final_choice[\'family\'],final_choice[\'param\'],\'| threshold:\',final_choice[\'threshold\'])\n    return dict(bundle=final,oof=selected_oof,comparison=comparison,choice=final_choice)\n\ndef holdout_evaluate(spec,result,cfg):\n    frame=spec[\'subjects\'].query("split==\'test\'").reset_index(drop=True);bundle=result[\'bundle\'];t=float(bundle[\'threshold\'])\n    assert set(frame.subject_id).isdisjoint(bundle[\'training_subjects\'])\n    pred=frame[[\'subject_id\',\'label\']].copy();p=predict_subjects(spec,frame.subject_id.tolist(),bundle)\n    pred[\'probability_pd\']=p;pred[\'decision_threshold\']=t;pred[\'prediction_pd\']=(p>=t).astype(int)\n    m=score(pred.label,p,t);m[\'constant_predicted_class\']=bool(pred.prediction_pd.nunique()==1);m[\'score_range\']=float(np.ptp(p))\n    pred.to_csv(Path(cfg[\'output\'])/\'tables\'/f\'{spec["name"]}_selected_retrospective_holdout.csv\',index=False)\n    write_json(Path(cfg[\'output\'])/\'tables\'/f\'{spec["name"]}_selected_retrospective_metrics.json\',m)\n    write_json(Path(cfg[\'output\'])/\'tables\'/f\'{spec["name"]}_fixed_05_holdout_diagnostic.json\',dict(**score(pred.label,p,.5),interpretation=\'Secondary diagnostic only; does not select threshold\'))\n    print(spec[\'name\'].upper(),\'| retrospective holdout:\',json.dumps(m));return pred,m\n\n\ndef score(y, p, threshold=0.5):\n    y = np.asarray(y, int)\n    raw=np.asarray(p,float)\n    if y.ndim!=1 or raw.ndim!=1 or len(y)!=len(raw) or len(y)==0:raise ValueError(\'Nonempty equal-length 1D labels/scores required.\')\n    if not set(y)<= {0,1} or not np.isfinite(raw).all() or np.any((raw<0)|(raw>1)):raise ValueError(\'Invalid labels or scores.\')\n    p = np.clip(raw, 1e-07, 1 - 1e-07)\n    cuts=np.asarray(threshold,float)\n    if cuts.ndim>1 or (cuts.ndim==1 and cuts.shape!=y.shape) or not np.isfinite(cuts).all() or np.any((cuts<=0)|(cuts>=1)):raise ValueError(\'Invalid decision threshold(s).\')\n    pred = (raw >= cuts).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    n = len(y)\n    k = int((pred == y).sum())\n    return dict(n=n, accuracy=accuracy_score(y, pred), balanced_accuracy=balanced_accuracy_score(y, pred), sensitivity=tp / (tp + fn) if tp + fn else None, specificity=tn / (tn + fp) if tn + fp else None, precision=precision_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0), roc_auc=roc_auc_score(y, p) if len(set(y)) == 2 else None, average_precision=average_precision_score(y, p) if len(set(y)) == 2 else None, mcc=matthews_corrcoef(y, pred), brier=brier_score_loss(y, p), log_loss=log_loss(y, p, labels=[0, 1]), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), threshold=float(cuts) if cuts.ndim==0 else None, accuracy_ci_low=float(beta.ppf(0.025, k, n - k + 1)) if k else 0.0, accuracy_ci_high=float(beta.ppf(0.975, k + 1, n - k)) if k < n else 1.0)\n\ndef publication_plots(pred, title, stem, threshold=0.5):\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import ConfusionMatrixDisplay\n    y = pred.label.to_numpy(int)\n    p = pred.probability_pd.to_numpy(float)\n    cuts=decision_thresholds(pred)\n    decision=(p>=cuts).astype(int)\n    plt.rcParams.update({\'font.family\': \'DejaVu Sans\', \'font.size\': 10, \'axes.spines.top\': False, \'axes.spines.right\': False, \'savefig.dpi\': 300})\n    fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)\n    fpr, tpr, _ = roc_curve(y, p)\n    axes[0, 0].plot(fpr, tpr, label=f\'AUC = {roc_auc_score(y, p):.3f}\')\n    axes[0, 0].plot([0, 1], [0, 1], \'--\', color=\'gray\')\n    axes[0, 0].set(xlabel=\'False positive rate\', ylabel=\'True positive rate\', title=\'ROC\')\n    axes[0, 0].legend()\n    precision, recall, _ = precision_recall_curve(y, p)\n    axes[0, 1].step(recall, precision, where=\'post\', label=f\'AP = {average_precision_score(y, p):.3f}\')\n    axes[0, 1].axhline(y.mean(), ls=\'--\', color=\'gray\', label=\'Prevalence\')\n    axes[0, 1].set(xlabel=\'Recall\', ylabel=\'Precision\', title=\'Precision–recall\')\n    axes[0, 1].legend()\n    prob_true, prob_pred = calibration_curve(y, p, n_bins=4, strategy=\'uniform\')\n    axes[0, 2].plot(prob_pred, prob_true, \'o-\')\n    axes[0, 2].plot([0, 1], [0, 1], \'--\', color=\'gray\')\n    axes[0, 2].set(xlabel=\'Mean predicted probability\', ylabel=\'Observed PD fraction\', title=\'Calibration (small sample)\')\n    ConfusionMatrixDisplay.from_predictions(y, decision, display_labels=[\'Control\', \'PD\'], ax=axes[1, 0], colorbar=False, cmap=\'Blues\')\n    axes[1, 0].set_title(\'Confusion matrix (development-selected thresholds)\')\n    for lab, color in [(0, \'#2878b5\'), (1, \'#d9534f\')]:\n        axes[1, 1].hist(p[y == lab], bins=np.linspace(0, 1, 11), alpha=0.65, label=[\'Control\', \'PD\'][lab], color=color)\n    axes[1, 1].set(xlabel=\'Predicted PD probability\', ylabel=\'Subjects\', title=\'Probability distribution\')\n    axes[1, 1].legend()\n    ts = np.linspace(0.05, 0.95, 50)\n    sens = []\n    spec = []\n    for t in ts:\n        m = score(y, p, t)\n        sens.append(m[\'sensitivity\'])\n        spec.append(m[\'specificity\'])\n    axes[1, 2].plot(ts, sens, label=\'Sensitivity\')\n    axes[1, 2].plot(ts, spec, label=\'Specificity\')\n    \n    for cut in np.unique(cuts):axes[1, 2].axvline(cut,ls=\'--\',color=\'gray\',alpha=.4)\n    axes[1, 2].set(xlabel=\'Threshold\', ylabel=\'Rate\', title=\'Threshold analysis (descriptive only)\')\n    axes[1, 2].legend()\n    for ax in [axes[0, 0], axes[0, 1], axes[0, 2], axes[1, 2]]:\n        ax.set_ylim(0, 1.03)\n    fig.suptitle(title)\n    for ext in [\'png\', \'pdf\', \'svg\']:\n        fig.savefig(str(stem) + \'.\' + ext, bbox_inches=\'tight\')\n    plt.show()\n    plt.close(fig)\n\ndef slice_images(volume, n_slices=8):\n    import torch\n    import torch.nn.functional as F\n    a = np.asarray(volume, np.float32)\n    assert a.ndim == 3 and np.isfinite(a).all()\n    images = []\n    for axis in range(3):\n        for frac in np.linspace(0.25, 0.75, n_slices):\n            i = int(round((a.shape[axis] - 1) * frac))\n            s = np.take(a, i, axis=axis).copy()\n            t = torch.from_numpy(s)[None, None]\n            t = F.interpolate(t, size=(224, 224), mode=\'bilinear\', align_corners=False)[0].repeat(3, 1, 1)\n            images.append(t)\n    x = torch.stack(images)\n    return (x - torch.tensor([0.485, 0.456, 0.406])[None, :, None, None]) / torch.tensor([0.229, 0.224, 0.225])[None, :, None, None]\n\n\ndef feature_columns(spec,family):\n    if family.startswith(\'MedicalNet_Global_\'):return spec[\'medical_global_features\']\n    if family.startswith(\'MedicalNet_Regional_\'):return spec[\'medical_regional_features\']\n    if family in [\'Voxel_SVM\',\'Voxel_LR\']:return spec[\'voxel_features\']\n    if family.startswith(\'Spatial_\'):return spec[\'spatial_features\']\n    return spec[\'features\']\n\ndef decision_thresholds(frame):\n    return frame[\'decision_threshold\'].to_numpy(float) if \'decision_threshold\' in frame else np.full(len(frame),.5)\n\ndef predicted_labels(frame):\n    return (frame.probability_pd.to_numpy(float)>=decision_thresholds(frame)).astype(int)\n\ndef score_frame(frame):\n    return score(frame.label,frame.probability_pd,decision_thresholds(frame))\n\n\nclass FeatureColumns(BaseEstimator,TransformerMixin):\n    def __init__(self,columns=None):self.columns=columns\n    def fit(self,X,y=None):\n        X=np.asarray(X);self.n_features_in_=X.shape[1]\n        if self.columns is not None and (not len(self.columns) or len(set(self.columns))!=len(self.columns) or min(self.columns)<0 or max(self.columns)>=X.shape[1]):raise ValueError(\'Invalid prespecified feature subset.\')\n        return self\n    def transform(self,X):\n        X=np.asarray(X)\n        if X.ndim!=2 or X.shape[1]!=self.n_features_in_:raise ValueError(\'Feature schema mismatch.\')\n        return X if self.columns is None else X[:,self.columns]\n\ndef safe_f_classif(X,y):\n    import warnings\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\');scores,pvalues=f_classif(X,y)\n    return np.nan_to_num(scores,nan=0.,posinf=np.finfo(float).max,neginf=0.),np.nan_to_num(pvalues,nan=1.)\n\ndef spatial_vector(volume,grid=(3,4,3)):\n    """Atlas-free intensity/foreground grid summaries; NOT anatomical ROI volumes.\n    Grid indices follow the trained z,y,x reference; foreground is intensity>0.\n    Occupancy/contrast may encode acquisition, age/sex or skull-strip differences.\n    """\n    a=np.asarray(volume,np.float32)\n    if a.ndim!=3 or not np.isfinite(a).all() or a.min()<0 or a.max()>1:raise ValueError(\'Expected finite normalized 3D MRI in [0,1].\')\n    mask=a>0\n    if mask.sum()<100:raise ValueError(\'Insufficient MRI foreground.\')\n    gradients=np.sqrt(sum(g*g for g in np.gradient(a)))\n    values=[]\n    zblocks=np.array_split(np.arange(a.shape[0]),grid[0]);yblocks=np.array_split(np.arange(a.shape[1]),grid[1]);xblocks=np.array_split(np.arange(a.shape[2]),grid[2])\n    for iz in zblocks:\n        for iy in yblocks:\n            for ix in xblocks:\n                q=np.ix_(iz,iy,ix);v=a[q];m=mask[q];brain=v[m]\n                values.append(float(m.mean()))\n                if len(brain):values.extend([float(brain.mean()),float(brain.std()),*np.quantile(brain,[.25,.5,.75]).tolist(),float(gradients[q][m].mean())])\n                else:values.extend([0.]*6)\n    foreground=a[mask];values.extend([float(mask.mean()),float(foreground.mean()),float(foreground.std()),*np.quantile(foreground,[.05,.25,.5,.75,.95]).tolist()])\n    left,right=np.array_split(a,2,axis=2);lm,rm=left>0,right>0\n    values.extend([float(abs(lm.mean()-rm.mean())),float(abs(left[lm].mean()-right[rm].mean())) if lm.any() and rm.any() else 0.])\n    result=np.asarray(values,np.float32)\n    if not np.isfinite(result).all():raise ValueError(\'Invalid spatial MRI descriptor.\')\n    return result\n\ndef usable_input_rows(bundle,X):\n    values=np.asarray(X,float)\n    if values.ndim!=2:raise ValueError(\'Expected a 2D subject-feature matrix.\')\n    columns=bundle.get(\'param\',{}).get(\'columns\')\n    if columns is not None:values=values[:,columns]\n    return np.isfinite(values).any(axis=1)\n'
(OUT/'pd_preprocessing_core.py').write_text(PREPROCESSING_SOURCE,encoding='utf-8')
(OUTPUT/'pd_four_model_core.py').write_text(MODEL_SOURCE,encoding='utf-8')
sys.path.insert(0,str(OUT));sys.path.insert(0,str(OUTPUT))
import importlib
import pd_preprocessing_core
importlib.reload(pd_preprocessing_core)
from pd_preprocessing_core import *
import pd_four_model_core
importlib.reload(pd_four_model_core)
from pd_four_model_core import *
pd_four_model_core._MRI_MODEL_SCOPE=MRI_MODEL_SCOPE
CFG=dict(seed=SEED,cnn_max_epochs=CNN_MAX_EPOCHS,cnn_patience=CNN_PATIENCE,cnn_slices_per_view=CNN_SLICES_PER_VIEW,
 output=str(OUTPUT),max_compute_hours=COMPUTE_BUDGET_HOURS,deadline_unix=RUN_STARTED+COMPUTE_BUDGET_HOURS*3600,
 version='9.0',module_hash=sha256(OUTPUT/'pd_four_model_core.py'))
CFG['software_versions']={p:__import__(p).__version__ for p in ['numpy','pandas','scipy','sklearn','torch','torchvision']}
CFG['mri_threshold_grid']=[.5]
CFG['mri_model_scope']=MRI_MODEL_SCOPE
CFG['threshold_grid']=[.3,.35,.4,.45,.5,.55,.6,.65,.7]
write_json(OUTPUT/'configuration.json',CFG)
INTEGRITY_SOURCE='"""Label-independent duplicate candidates; review never depends on model accuracy."""\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nimport pandas as pd\n\nPAIR_COLUMNS=[\'pair_id\',\'modality\',\'subject_a\',\'subject_b\',\'kind\',\'distance\',\'similarity\',\'label_conflict\']\n\ndef pair_record(name,a,b,kind,distance,similarity,ya,yb):\n    key=hashlib.sha256(f\'{name}|{min(a,b)}|{max(a,b)}|{kind}\'.encode()).hexdigest()[:20]\n    return dict(pair_id=key,modality=name,subject_a=a,subject_b=b,kind=kind,distance=float(distance),similarity=float(similarity),label_conflict=int(ya)!=int(yb))\n\ndef numeric_candidates(name,ids,labels,x,tolerance=1e-4):\n    # বাংলা: repeated scalar value duplicate নয়; সম্পূর্ণ vector-এর relative difference পরীক্ষা।\n    x=np.asarray(x,float);rows=[]\n    for i in range(len(x)):\n        for j in range(i):\n            finite=np.isfinite(x[i])&np.isfinite(x[j])\n            if not finite.any():continue\n            same_missing=np.array_equal(np.isnan(x[i]),np.isnan(x[j]))\n            if not same_missing:continue\n            a=x[i,finite];b=x[j,finite]\n            exact=np.array_equal(a,b)\n            difference=np.max(np.abs(a-b)/np.maximum(np.maximum(np.abs(a),np.abs(b)),1e-8))\n            if exact or difference<=tolerance:\n                rows.append(pair_record(name,str(ids[i]),str(ids[j]),\'exact_feature_vector\' if exact else \'near_feature_vector\',difference,1-difference,labels[i],labels[j]))\n    return pd.DataFrame(rows,columns=PAIR_COLUMNS)\n\ndef enforce_review(pairs,out,review_csv=None):\n    out=Path(out);out.parent.mkdir(parents=True,exist_ok=True);pairs.to_csv(out,index=False)\n    if pairs.empty:\n        print(f\'{out.stem}: no candidates at configured thresholds\');return\n    print(pairs.to_string(index=False))\n    template=pairs[[\'pair_id\']].copy();template[\'decision\']=\'\';template[\'evidence\']=\'\'\n    template.to_csv(out.with_name(out.stem+\'_review_template.csv\'),index=False)\n    if review_csv is None:\n        raise RuntimeError(f\'Duplicate candidates require identity review. Report: {out}. Do not delete subjects based on accuracy. Set DUPLICATE_REVIEW_CSV after documented review.\')\n    review=pd.read_csv(review_csv,dtype=str).fillna(\'\')\n    assert {\'pair_id\',\'decision\',\'evidence\'}<=set(review) and review.pair_id.is_unique\n    selected=pairs.merge(review[[\'pair_id\',\'decision\',\'evidence\']],on=\'pair_id\',how=\'left\',validate=\'one_to_one\')\n    bad=selected.decision.ne(\'verified_distinct\')|selected.evidence.fillna(\'\').str.strip().eq(\'\')\n    if bad.any():raise RuntimeError(\'Unresolved duplicate identities. Correct source data / exclusion manifest before restarting; never mark a confirmed duplicate as distinct.\')\n    selected.to_csv(out.with_name(out.stem+\'_reviewed.csv\'),index=False)\n    print(\'All flagged pairs have documented distinct-identity review; reports retained.\')\n\ndef clean_voice(observations,features,out):\n    # বাংলা: একই patient-এর identical full recording feature row একবার রাখা; audit trail সংরক্ষণ।\n    cols=[\'subject_id\']+list(features)\n    duplicates=observations.duplicated(cols,keep=\'first\')\n    observations.loc[duplicates].to_csv(out,index=False)\n    clean=observations.loc[~duplicates].copy()\n    print(\'Repeated within-subject feature rows removed:\',int(duplicates.sum()))\n    return clean\n\ndef mri_candidates(frame,correlation=.9999,nrmse=.002):\n    # বাংলা: full normalized voxel hash এবং non-background downsample similarity দুটো আলাদা পরীক্ষা।\n    from scipy.ndimage import zoom\n    small=[];digests=[]\n    for row in frame.itertuples(index=False):\n        volume=np.load(row.array_path,allow_pickle=False).astype(\'float32\')\n        digests.append(hashlib.sha256(np.ascontiguousarray(volume).tobytes()).hexdigest())\n        small.append(zoom(volume,np.array([32,32,32])/np.array(volume.shape),order=1,prefilter=False).ravel())\n    rows=[]\n    for i in range(len(frame)):\n        for j in range(i):\n            a,b=small[i],small[j];mask=(a>.05)|(b>.05)\n            if mask.sum()<100:continue\n            a,b=a[mask],b[mask]\n            error=float(np.sqrt(np.mean((a-b)**2)))\n            corr=float(np.corrcoef(a,b)[0,1]) if min(a.std(),b.std())>1e-8 else 0.\n            exact=digests[i]==digests[j]\n            if exact or (corr>=correlation and error<=nrmse):\n                rows.append(pair_record(\'mri\',frame.subject_id.iloc[i],frame.subject_id.iloc[j],\'exact_voxels\' if exact else \'near_normalized_volume\',error,corr,frame.label.iloc[i],frame.label.iloc[j]))\n    return pd.DataFrame(rows,columns=PAIR_COLUMNS)\n\ndef spiral_shape_candidates(spiral_root,load_trajectory):\n    # বাংলা: translation/scale-normalized trajectory signature; temporal order অপরিবর্তিত।\n    ids=[];labels=[];vectors=[]\n    groups=[(\'Healthy\',\'hw_dataset/control\',0),(\'Parkinson\',\'hw_dataset/parkinson\',1),(\'New_Parkinson\',\'new_dataset/parkinson\',1)]\n    for group,relative,label in groups:\n        for p in sorted((Path(spiral_root)/relative).glob(\'*.txt\')):\n            df=load_trajectory(p);chunks=[];valid=True\n            for task in (0,1):\n                xy=df.loc[df.TestID==task,[\'X\',\'Y\']].to_numpy(float)\n                if len(xy)<10 or not np.isfinite(xy).all():valid=False;break\n                t=np.linspace(0,1,len(xy));q=np.linspace(0,1,128)\n                xy=np.column_stack([np.interp(q,t,xy[:,k]) for k in (0,1)])\n                xy-=xy.mean(axis=0);scale=np.sqrt(np.mean(xy**2))\n                if scale<=1e-8:valid=False;break\n                chunks.append((xy/scale).ravel())\n            if valid:ids.append(f\'{group}/{p.stem}\');labels.append(label);vectors.append(np.concatenate(chunks))\n    rows=[]\n    for i in range(len(vectors)):\n        for j in range(i):\n            error=float(np.sqrt(np.mean((vectors[i]-vectors[j])**2)))\n            if error<=.001:rows.append(pair_record(\'spiral\',ids[i],ids[j],\'near_trajectory_shape\',error,1-error,labels[i],labels[j]))\n    return pd.DataFrame(rows,columns=PAIR_COLUMNS)\n'
ALIGNMENT_SOURCE='"""Experimental class-prototype transfer. No patient-level multimodal fusion."""\nfrom pathlib import Path\nimport json, copy, hashlib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.metrics import balanced_accuracy_score, roc_auc_score\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nNAMES=(\'voice\',\'spiral\',\'mri\')\n\ndef subject_inputs(spec):\n    frame=spec[\'subjects\'][[\'subject_id\',\'label\',\'split\']].copy().reset_index(drop=True)\n    if spec[\'name\']==\'voice\':\n        # বাংলা: alignment-এ প্রতি patient একটি row; independent baselines recording aggregate করে।\n        values=spec[\'observations\'].groupby(\'subject_id\')[spec[\'features\']].mean()\n        x=values.reindex(frame.subject_id).to_numpy(float)\n    else:x=spec[\'subjects\'][spec[\'features\']].to_numpy(float)\n    assert len(x)==len(frame) and frame.subject_id.is_unique\n    return dict(frame=frame,x=x)\n\nclass PrototypeNetwork(nn.Module):\n    def __init__(self,dims,isolated=False):\n        super().__init__();self.isolated=isolated\n        self.adapters=nn.ModuleDict({n:nn.Sequential(nn.Linear(d,32),nn.ReLU(),nn.Linear(32,32)) for n,d in dims.items()})\n        self.heads=nn.ModuleDict({n:nn.Linear(32,2) for n in dims})\n        self.shared=nn.Linear(32,2)\n    def forward(self,name,x):\n        z=F.normalize(self.adapters[name](x),dim=1)\n        own=self.heads[name](z)\n        return z,own,self.shared(z)\n\ndef fit_alignment(data,train_ids,lam,seed,epochs=120,isolated=False,device=\'cpu\',cache_dir=None):\n    # বাংলা: scaler/imputer/PCA এবং prototypes শুধু এই fit-এর training subjects থেকে।\n    torch.manual_seed(seed);np.random.seed(seed)\n    transforms={};arrays={};labels={};dims={};raw_signatures={}\n    for n in NAMES:\n        d=data[n];ix=d[\'frame\'].set_index(\'subject_id\').index.get_indexer(train_ids[n]);assert (ix>=0).all()\n        x=d[\'x\'][ix];y=d[\'frame\'].label.to_numpy(int)[ix]\n        raw_signatures[n]=hashlib.sha256(np.ascontiguousarray(x).tobytes()+y.tobytes()).hexdigest()\n        assert set(y)=={0,1}\n        k=max(1,min(16,len(x)-2,x.shape[1]))\n        tr=Pipeline([(\'impute\',SimpleImputer(strategy=\'median\',keep_empty_features=True)),(\'scale\',StandardScaler()),(\'pca\',PCA(n_components=k,svd_solver=\'full\'))])\n        a=tr.fit_transform(x).astype(\'float32\');transforms[n]=tr;dims[n]=a.shape[1]\n        arrays[n]=torch.tensor(a,device=device);labels[n]=torch.tensor(y,dtype=torch.long,device=device)\n    cache_path=None\n    if cache_dir is not None:\n        import joblib\n        digest=hashlib.sha256(Path(__file__).read_bytes())\n        digest.update(json.dumps(dict(ids=train_ids,raw_signatures=raw_signatures,lambda_alignment=lam,seed=seed,epochs=epochs,isolated=isolated),sort_keys=True).encode())\n        for n in NAMES:\n            digest.update(arrays[n].detach().cpu().numpy().tobytes());digest.update(labels[n].detach().cpu().numpy().tobytes())\n        cache_path=Path(cache_dir)/(digest.hexdigest()+\'.joblib\');cache_path.parent.mkdir(parents=True,exist_ok=True)\n        if cache_path.exists():return joblib.load(cache_path)\n    net=PrototypeNetwork(dims,isolated).to(device)\n    optimizer=torch.optim.Adam(net.parameters(),lr=.001,weight_decay=.001)\n    history=[]\n    for epoch in range(1,epochs+1):\n        net.train();optimizer.zero_grad();own_losses=[];shared_losses=[];centres={};details={}\n        for n in NAMES:\n            z,own,shared=net(n,arrays[n]);y=labels[n]\n            count=torch.bincount(y,minlength=2).float();w=len(y)/(2*count)\n            own_loss=F.cross_entropy(own,y,weight=w)\n            shared_loss=F.cross_entropy(shared,y,weight=w)\n            own_losses.append(own_loss);shared_losses.append(shared_loss)\n            centres[n]=torch.stack([F.normalize(z[y==c].mean(0),dim=0) for c in (0,1)])\n            logits=own if isolated else shared\n            details[n+\'_train_accuracy\']=float((logits.argmax(1)==y).float().mean().detach().cpu())\n        classification=torch.stack(own_losses).mean()\n        shared_loss=torch.stack(shared_losses).mean()\n        penalties=[(centres[a]-centres[b]).square().mean() for a,b in [(\'voice\',\'spiral\'),(\'voice\',\'mri\'),(\'spiral\',\'mri\')]]\n        alignment=torch.stack(penalties).mean()\n        loss=classification if isolated else classification+shared_loss+lam*alignment\n        loss.backward()\n        if isolated:\n            for n in NAMES:torch.nn.utils.clip_grad_norm_(list(net.adapters[n].parameters())+list(net.heads[n].parameters()),5.)\n        else:torch.nn.utils.clip_grad_norm_(net.parameters(),5.)\n        optimizer.step()\n        history.append(dict(epoch=epoch,total_loss=float(loss.detach().cpu()),independent_loss=float(classification.detach().cpu()),shared_loss=float(shared_loss.detach().cpu()),prototype_loss=float(alignment.detach().cpu()),**details))\n    net.cpu()\n    bundle=dict(state={k:v.detach().cpu().clone() for k,v in net.state_dict().items()},dims=dims,transforms=transforms,lambda_alignment=lam,seed=seed,epochs=epochs,isolated=isolated,training_subjects={n:list(train_ids[n]) for n in NAMES},history=history,score_calibrated=False)\n    if cache_path is not None:\n        temporary=cache_path.with_suffix(\'.partial\');joblib.dump(bundle,temporary);temporary.replace(cache_path)\n    return bundle\n\ndef predict_alignment(bundle,name,x):\n    net=PrototypeNetwork(bundle[\'dims\'],bundle[\'isolated\']);net.load_state_dict(bundle[\'state\']);net.eval()\n    a=bundle[\'transforms\'][name].transform(x).astype(\'float32\')\n    with torch.inference_mode():\n        _,own,shared=net(name,torch.tensor(a));logits=own if bundle[\'isolated\'] else shared\n        return torch.softmax(logits,dim=1)[:,1].numpy()\n\ndef partitions(data,ids,seed):\n    output={}\n    common_folds=min(3,min(int(data[n][\'frame\'].set_index(\'subject_id\').loc[ids[n]].label.value_counts().min()) for n in NAMES))\n    assert common_folds>=2\n    for n in NAMES:\n        frame=data[n][\'frame\'].set_index(\'subject_id\').loc[ids[n]].reset_index()\n        k=common_folds;assert k>=2\n        output[n]=[(frame.subject_id.iloc[t].tolist(),frame.subject_id.iloc[v].tolist()) for t,v in StratifiedKFold(k,shuffle=True,random_state=seed).split(frame,frame.label)]\n    for n in NAMES:\n        covered=[sid for _,valid in output[n] for sid in valid]\n        assert len(covered)==len(set(covered))==len(ids[n]) and set(covered)==set(ids[n])\n    return output\n\ndef row_predictions(data,bundle,ids):\n    predictions={}\n    for n in NAMES:\n        d=data[n];ix=d[\'frame\'].set_index(\'subject_id\').index.get_indexer(ids[n]);assert (ix>=0).all()\n        assert set(ids[n]).isdisjoint(bundle[\'training_subjects\'][n])\n        frame=d[\'frame\'].iloc[ix][[\'subject_id\',\'label\']].copy()\n        frame[\'probability_pd\']=predict_alignment(bundle,n,d[\'x\'][ix]);predictions[n]=frame\n    return predictions\n\ndef tune_lambda(data,ids,grid,seed,epochs,device,cache_dir=None):\n    folds=partitions(data,ids,seed);rows=[]\n    for lam in grid:\n        scores=[]\n        for f in range(min(len(folds[n]) for n in NAMES)):\n            tr={n:folds[n][f][0] for n in NAMES};va={n:folds[n][f][1] for n in NAMES}\n            model=fit_alignment(data,tr,lam,seed+f,epochs,False,device,cache_dir)\n            predictions=row_predictions(data,model,va)\n            scores.extend(balanced_accuracy_score(p.label,p.probability_pd>=.5) for p in predictions.values())\n        rows.append(dict(lambda_alignment=lam,mean_modality_balanced_accuracy=float(np.mean(scores))))\n    # বাংলা: tie হলে কম alignment; holdout score দিয়ে lambda নির্বাচন নয়।\n    best=max(rows,key=lambda r:(r[\'mean_modality_balanced_accuracy\'],-r[\'lambda_alignment\']))\n    return best[\'lambda_alignment\'],rows\n\ndef run_alignment(specs,out,seed=42,epochs=120,grid=(0.,.01,.1),device=\'cpu\'):\n    import joblib\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    data={n:subject_inputs(specs[n]) for n in NAMES}\n    dev={n:data[n][\'frame\'].query("split==\'development\'").subject_id.tolist() for n in NAMES}\n    test={n:data[n][\'frame\'].query("split==\'test\'").subject_id.tolist() for n in NAMES}\n    folds=partitions(data,dev,seed);all_predictions=[];search=[]\n    for f in range(len(folds[NAMES[0]])):\n        tr={n:folds[n][f][0] for n in NAMES};va={n:folds[n][f][1] for n in NAMES}\n        lam,records=tune_lambda(data,tr,grid,seed+100+f,epochs,device,out/\'fit_cache\')\n        search.extend(dict(stage=f\'outer_{f+1}\',**r) for r in records)\n        for variant,weight,isolated in [(\'isolated\',0.,True),(\'shared_no_alignment\',0.,False),(\'aligned_selected\',lam,False)]:\n            print(f\'ALIGNMENT | outer {f+1}/3 | {variant} | started\',flush=True)\n            model=fit_alignment(data,tr,weight,seed+500+f,epochs,isolated,device,out/\'fit_cache\')\n            pd.DataFrame(model[\'history\']).to_csv(out/f\'history_outer{f+1}_{variant}.csv\',index=False)\n            for n,p in row_predictions(data,model,va).items():\n                p[\'modality\']=n;p[\'variant\']=variant;p[\'fold\']=f+1;all_predictions.append(p)\n            print(f\'ALIGNMENT | outer {f+1}/3 | {variant} | SUCCESSFULLY COMPLETED\',flush=True)\n    oof=pd.concat(all_predictions,ignore_index=True);oof.to_csv(out/\'outer_oof_predictions.csv\',index=False)\n    lam,records=tune_lambda(data,dev,grid,seed+900,epochs,device,out/\'fit_cache\')\n    search.extend(dict(stage=\'final_development\',**r) for r in records)\n    pd.DataFrame(search).to_csv(out/\'lambda_inner_search.csv\',index=False)\n    pd.DataFrame([dict(selected_lambda=lam,epochs=epochs,selection=\'Development inner CV only\',paired=False)]).to_csv(out/\'selection_lock.csv\',index=False)\n    tests=[]\n    for variant,weight,isolated in [(\'isolated\',0.,True),(\'shared_no_alignment\',0.,False),(\'aligned_selected\',lam,False)]:\n        model=fit_alignment(data,dev,weight,seed+1500,epochs,isolated,device,out/\'fit_cache\')\n        model[\'features\']={n:list(specs[n][\'features\']) for n in NAMES}\n        joblib.dump(model,out/f\'{variant}_bundle.joblib\')\n        pd.DataFrame(model[\'history\']).to_csv(out/f\'history_final_{variant}.csv\',index=False)\n        for n,p in row_predictions(data,model,test).items():\n            p[\'modality\']=n;p[\'variant\']=variant;tests.append(p)\n    test_predictions=pd.concat(tests,ignore_index=True);test_predictions.to_csv(out/\'retrospective_holdout_predictions.csv\',index=False)\n    return data,oof,test_predictions\n'
(OUTPUT/'pd_integrity_v4.py').write_text(INTEGRITY_SOURCE,encoding='utf-8')
(OUTPUT/'pd_unpaired_alignment.py').write_text(ALIGNMENT_SOURCE,encoding='utf-8')
import pd_integrity_v4,pd_unpaired_alignment
importlib.reload(pd_integrity_v4);importlib.reload(pd_unpaired_alignment)
(OUTPUT/'pd_spiral_features.py').write_text(SPIRAL_SOURCE,encoding='utf-8')
import pd_spiral_features
importlib.reload(pd_spiral_features)
from pd_integrity_v4 import clean_voice,numeric_candidates,enforce_review,mri_candidates,spiral_shape_candidates
from pd_unpaired_alignment import run_alignment,predict_alignment,fit_alignment,row_predictions
write_json(OUTPUT/'tables/duplicate_protocol.json',dict(relative_feature_tolerance=NEAR_VECTOR_RELATIVE_TOLERANCE,mri_correlation=MRI_NEAR_CORRELATION,mri_nrmse=MRI_NEAR_NRMSE,review_csv=str(DUPLICATE_REVIEW_CSV),exact_source_duplicates='stop',near_candidates='documented identity review',no_accuracy_driven_exclusion=True))

MRI_PREPROCESSING_SOURCE="import gc\nfixed=sitk.ReadImage(str(TEMPLATE_PATH),sitk.sitkFloat32)\nfixed_mask=sitk.ReadImage(str(TEMPLATE_MASK),sitk.sitkUInt8)\nref=sitk.GetArrayFromImage(fixed_mask)>0\nPROTOCOL=dict(version='9.0',n4_iterations=[30,20,10],n4_estimation_shrink=[2,2,2],bias_application='native full-resolution grid before registration',\n registration='masked rigid Euler3D; mask-moment initialization',metric='MattesMutualInformation',metric_bins=40,\n sampling='RANDOM',sampling_fraction=.25,seed=SEED,optimizer='RegularStepGradientDescent',learning_rate=1.,\n min_step=.0001,iterations=250,relaxation_factor=.6,optimizer_scales='physical_shift',\n pyramid_shrink=[4,2,1],pyramid_smoothing_mm=[2,1,0],interpolator='linear',mask_interpolator='nearest',\n normalization=dict(percentiles=[1,99],scope='subject own aligned brain mask',clip=[0,1],background=0),\n array_order='SimpleITK GetArrayFromImage: z,y,x; physical reference grid recorded separately',\n reference_grid=dict(size=list(fixed.GetSize()),spacing=list(fixed.GetSpacing()),origin=list(fixed.GetOrigin()),direction=list(fixed.GetDirection())),\n slice_sampling=dict(fractions=[.25,.75],frozen_per_axis=FROZEN_SLICES_PER_VIEW,cnn_per_axis=CNN_SLICES_PER_VIEW,\n resize=[224,224],cnn_resize=[160,160],imagenet_mean=[.485,.456,.406],imagenet_std=[.229,.224,.225]),\n synthstrip_command='python mri_synthstrip -i INPUT -o BRAIN -m MASK --model WEIGHTS --threads 2',\n assets=ASSET_HASHES,software={p:__import__(p).__version__ for p in ['numpy','nibabel','SimpleITK']})\nPROTOCOL['implementation_sha256']=sha256(OUT/'mri_preprocessing_cell.py')\nPROTOCOL_HASH=hashlib.sha256(json.dumps(PROTOCOL,sort_keys=True).encode()).hexdigest()\nwrite_json(OUT/'tables/mri_preprocessing_protocol.json',PROTOCOL)\n\ndef preprocess_one(row,reference_mask_array=ref):\n    deadline_check();folder=OUT/'mri_cache'/row.subject_id;folder.mkdir(exist_ok=True)\n    marker=folder/'completed.json'\n    if marker.exists():\n        record=json.loads(marker.read_text())\n        assert record['protocol_hash']==PROTOCOL_HASH and record['input_hash']==row.sha256,'Cache protocol/input mismatch; use a fresh output directory.'\n        assert sha256(folder/'normalized.npy')==record['array_hash'],'Cache array corrupted'\n        for filename,key in [('mask.nii.gz','native_mask_hash'),('registered_mask.nii.gz','registered_mask_hash'),('n4_corrected_native_brain.nii.gz','native_corrected_hash')]:\n            assert sha256(folder/filename)==record[key],'Review image cache corrupted: '+filename\n        return record\n    brain=folder/'brain.nii.gz';mask=folder/'mask.nii.gz'\n    # Unfinished cache is not trusted after a changed input/protocol.\n    for stale in [brain,mask]:\n        if stale.exists():stale.unlink()\n    if not (brain.exists() and mask.exists()):\n        cmd=[sys.executable,str(SCRIPT),'-i',row.path,'-o',str(brain),'-m',str(mask),'--model',str(WEIGHTS),'--threads','2']\n        result=subprocess.run(cmd,capture_output=True,text=True,timeout=1200)\n        (folder/'synthstrip.log').write_text(result.stdout+'\\n'+result.stderr)\n        if result.returncode:raise RuntimeError(f'SynthStrip failed for {row.subject_id}; see log: {result.stderr[-500:]}')\n    original=sitk.ReadImage(row.path,sitk.sitkFloat32);bm=sitk.ReadImage(str(mask),sitk.sitkUInt8)\n    assert original.GetSize()==bm.GetSize()\n    np.testing.assert_allclose(original.GetSpacing(),bm.GetSpacing(),atol=1e-4)\n    np.testing.assert_allclose(original.GetDirection(),bm.GetDirection(),atol=1e-4)\n    original_array=sitk.GetArrayFromImage(original)\n    assert np.isfinite(original_array).all() and np.any(original_array!=0),'Non-finite or empty MRI voxels'\n    del original_array\n    np.testing.assert_allclose(original.GetOrigin(),bm.GetOrigin(),atol=1e-4)\n    # বাংলা: ছোট image দিয়ে bias field estimate; corrected image full-resolution-এ থাকবে।\n    positive_mask=sitk.Cast((bm>0)&(original>0),sitk.sitkUInt8)\n    small=sitk.Shrink(original,[2,2,2]);small_mask=sitk.Shrink(positive_mask,[2,2,2])\n    n4=sitk.N4BiasFieldCorrectionImageFilter();n4.SetMaximumNumberOfIterations([30,20,10])\n    n4.Execute(small,small_mask)\n    log_bias=n4.GetLogBiasFieldAsImage(original)\n    corrected=sitk.Cast(original/sitk.Exp(log_bias),sitk.sitkFloat32)\n    ms=sitk.Cast(bm>0,sitk.sitkUInt8)\n    moving=sitk.Mask(corrected,ms)\n    assert moving.GetSize()==original.GetSize(),'Native resolution must be preserved.'\n    sitk.WriteImage(moving,str(folder/'n4_corrected_native_brain.nii.gz'))\n    native_mask_array=sitk.GetArrayFromImage(ms)>0\n    native_brain_volume_ml=float(native_mask_array.sum()*np.prod(original.GetSpacing())/1000.)\n    native_mask_fraction=float(native_mask_array.mean())\n    from scipy.ndimage import label as component_label\n    components,count=component_label(native_mask_array)\n    sizes=np.bincount(components.ravel())[1:]\n    largest_component_fraction=float(sizes.max()/sizes.sum()) if len(sizes) else 0.\n    del components,native_mask_array\n    initial=sitk.CenteredTransformInitializer(sitk.Cast(fixed_mask,sitk.sitkFloat32),sitk.Cast(ms,sitk.sitkFloat32),sitk.Euler3DTransform(),sitk.CenteredTransformInitializerFilter.MOMENTS)\n    reg=sitk.ImageRegistrationMethod();reg.SetMetricAsMattesMutualInformation(40)\n    reg.SetMetricFixedMask(fixed_mask);reg.SetMetricMovingMask(ms)\n    reg.SetMetricSamplingStrategy(reg.RANDOM);reg.SetMetricSamplingPercentage(.25,SEED)\n    reg.SetInterpolator(sitk.sitkLinear);reg.SetOptimizerAsRegularStepGradientDescent(learningRate=1.,minStep=.0001,numberOfIterations=250,relaxationFactor=.6)\n    reg.SetOptimizerScalesFromPhysicalShift();reg.SetShrinkFactorsPerLevel([4,2,1]);reg.SetSmoothingSigmasPerLevel([2,1,0]);reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()\n    reg.SetInitialTransform(initial,inPlace=False);transform=reg.Execute(fixed,moving)\n    aligned=sitk.Resample(moving,fixed,transform,sitk.sitkLinear,0.,sitk.sitkFloat32)\n    aligned_mask=sitk.Resample(ms,fixed,transform,sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)\n    optimizer_metric=float(reg.GetMetricValue())\n    if not np.isfinite(optimizer_metric):raise RuntimeError('Non-finite registration metric.')\n    a=sitk.GetArrayFromImage(aligned).astype('float32');m=sitk.GetArrayFromImage(aligned_mask)>0\n    assert np.isfinite(a).all() and m.sum()>100\n    # বাংলা: intensity normalization প্রতি subject-এর নিজের brain mask থেকে; cohort statistics নয়।\n    lo,hi=np.percentile(a[m],[1,99]);assert hi>lo\n    normalized=np.clip((a-lo)/(hi-lo),0,1);normalized[~m]=0\n    np.save(folder/'normalized.npy',normalized.astype('float32'))\n    sitk.WriteImage(aligned,str(folder/'registered_brain.nii.gz'));sitk.WriteImage(aligned_mask,str(folder/'registered_mask.nii.gz'))\n    sitk.WriteTransform(transform,str(folder/'rigid_transform.h5'))\n    intersection=(reference_mask_array&m).sum();dice=2*intersection/(reference_mask_array.sum()+m.sum());coverage=intersection/reference_mask_array.sum()\n    active=np.where(reference_mask_array.sum(axis=(1,2))>0)[0];edge=active[:max(1,len(active)//5)]\n    inferior=float((reference_mask_array[edge]&m[edge]).sum()/reference_mask_array[edge].sum())\n    fig,axes=plt.subplots(3,2,figsize=(8,10),constrained_layout=True)\n    for axidx,axis in enumerate([0,1,2]):\n        i=a.shape[axis]//2\n        axes[axidx,0].imshow(np.rot90(np.take(normalized,i,axis=axis)),cmap='gray',vmin=0,vmax=1)\n        axes[axidx,1].imshow(np.rot90(np.take(normalized,i,axis=axis)),cmap='gray',vmin=0,vmax=1)\n        axes[axidx,1].contour(np.rot90(np.take(reference_mask_array,i,axis=axis)),levels=[.5],colors=['red'],linewidths=.5)\n        for ax in axes[axidx]:ax.axis('off')\n    fig.suptitle(f'{row.subject_id} | anatomical QC pending | Dice={dice:.3f}')\n    fig.savefig(folder/'qc.png',dpi=150);plt.close(fig)\n    record=dict(subject_id=row.subject_id,protocol_hash=PROTOCOL_HASH,input_hash=row.sha256,\n       array_hash=sha256(folder/'normalized.npy'),dice=float(dice),coverage=float(coverage),inferior_coverage=inferior,\n       native_mask_hash=sha256(mask),registered_mask_hash=sha256(folder/'registered_mask.nii.gz'),native_corrected_hash=sha256(folder/'n4_corrected_native_brain.nii.gz'),\n       native_brain_volume_ml=native_brain_volume_ml,native_mask_fraction=native_mask_fraction,largest_component_fraction=largest_component_fraction,registration_metric=optimizer_metric,\n       priority_review=bool(dice<.82 or coverage<.75 or inferior<.5 or largest_component_fraction<.95),stop_condition=reg.GetOptimizerStopConditionDescription(),visual_qc='PENDING')\n    write_json(marker,record);gc.collect();return record\n\ncomplete('V9-09')\n"
(OUT/'mri_preprocessing_cell.py').write_text(MRI_PREPROCESSING_SOURCE,encoding='utf-8')
(OUTPUT/'pd_input_validation.py').write_text(VALIDATION_SOURCE,encoding='utf-8')
import pd_input_validation
importlib.reload(pd_input_validation)
from pd_input_validation import validate_qc_review,validate_mri_metadata,validate_feature_frame
write_json(OUT/'tables/software_versions.json',CFG['software_versions'])
complete('V9-02')


[V9-02] SUCCESSFULLY COMPLETED | elapsed=0.13 min


## Runtime self-checks
Checks synthetic malformed files, stale QC evidence and all-missing input before reading research data.


In [11]:
# Lightweight runtime input/provenance checks; synthetic fixtures are not study data.
import tempfile
from pd_spiral_features import shape_features
with tempfile.TemporaryDirectory() as temporary:
    bad=Path(temporary)/'invalid.txt';bad.write_text('1;2;3;4;5;6;7;0\n')
    try:
        load_trajectory(bad)
    except ValueError:pass
    else:raise AssertionError('Parser accepted eight columns.')
fixture=pd.DataFrame(dict(subject_id=['a'],input_hash=['r'],array_hash=['v'],protocol_hash=['p'],native_mask_hash=['mask'],registered_mask_hash=['regmask'],native_corrected_hash=['native']))
review_fixture=fixture.assign(decision='accept',reviewer='synthetic fixture',evidence='test only')
assert validate_qc_review(review_fixture,fixture)
review_fixture.loc[0,'array_hash']='changed'
try:validate_qc_review(review_fixture,fixture)
except ValueError:pass
else:raise AssertionError('Stale anatomical review was accepted.')
try:validate_feature_frame(pd.DataFrame({'f':[np.nan]}),['f'],'spiral')
except ValueError:pass
else:raise AssertionError('Entirely missing Spiral input was accepted.')
print('Runtime parser, review-hash and abstention self-checks PASSED. No research samples used.')
complete('V9-SELF-CHECKS')


Runtime parser, review-hash and abstention self-checks PASSED. No research samples used.
[V9-SELF-CHECKS] SUCCESSFULLY COMPLETED | elapsed=0.18 min


## Resolve exact raw-data paths

Known folder structures are checked directly. Missing paths stop with an actionable error; there is no whole-input recursive search.


In [12]:
# Download NEUROCON (public, no login): https://fcon_1000.projects.nitrc.org/indi/retro/parkinsons.html
import subprocess, tarfile
MRI_DL=Path('/kaggle/working/mri_data'); MRI_DL.mkdir(parents=True,exist_ok=True)
tgz=MRI_DL/'neurocon.tar.gz'
if not any(MRI_DL.rglob('neurocon_patients.tsv')):
    subprocess.run(['wget','-q','-O',str(tgz),'https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/umf_pd/neurocon.tar.gz'],check=True)
    with tarfile.open(tgz) as t:
        members=[m for m in t.getmembers() if m.name.endswith('neurocon_patients.tsv') or 'T1w' in m.name]
        t.extractall(MRI_DL,members=members)
    tgz.unlink()
t1=list(MRI_DL.rglob('*T1w.nii*')); print('NEUROCON T1 files:',len(t1)); assert len(t1)==43
complete('V19-NEUROCON-DOWNLOAD')


NEUROCON T1 files: 43
[V19-NEUROCON-DOWNLOAD] SUCCESSFULLY COMPLETED | elapsed=0.23 min


In [13]:
BASE=Path('/kaggle/input')
def _find(pattern,want_dir=False):
    hits=[p for root in [Path('/kaggle/input'),Path('/kaggle/working/mri_data')] if root.exists()
          for p in root.rglob(pattern) if p.is_dir()==want_dir]
    if not hits: raise FileNotFoundError(f'Not found: {pattern}. Attach the voice and spiral Kaggle datasets.')
    return sorted(hits,key=lambda p:len(p.parts))[0]
VOICE_PATH=_find('parkinsons.data')
MRI_ROOT=_find('neurocon_patients.tsv').parent
SPIRAL_ROOT=_find('hw_dataset',want_dir=True).parent
for k,v in dict(voice=VOICE_PATH,mri=MRI_ROOT,spiral=SPIRAL_ROOT).items(): print(k,'->',v)
assert (SPIRAL_ROOT/'hw_dataset/control').is_dir() and (SPIRAL_ROOT/'new_dataset/parkinson').is_dir()
write_json(OUT/'tables/input_locations.json',dict(base=str(BASE),voice=str(VOICE_PATH),mri=str(MRI_ROOT),spiral=str(SPIRAL_ROOT)))
complete('V9-03')


voice -> /kaggle/input/datasets/mdnazmulhasan07/voice-dataset/Voice/parkinsons.data
mri -> /kaggle/working/mri_data/neurocon
spiral -> /kaggle/input/datasets/mdnazmulhasan07/spiral-datasset/Spiral
[V9-03] SUCCESSFULLY COMPLETED | elapsed=0.38 min


## Resumable local file copying

Copy each raw file once while computing its SHA-256; all subsequent parsing and preprocessing use local files. Resume checks local integrity. Keep the dataset unchanged when resuming this project directory.


In [ ]:
def copy_verified(source,target):
    source=Path(source);target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)
    if not source.is_file() or source.stat().st_size==0:raise ValueError(f'Missing/empty source: {source}')
    marker=target.with_name(target.name+'.complete.json')
    current_hash=sha256(source)
    if target.exists() and marker.exists():
        old=json.loads(marker.read_text())
        if old.get('source')==str(source) and old.get('sha256')==current_hash and sha256(target)==current_hash:
            return target,current_hash
    temporary=target.with_name(target.name+'.partial')
    try:
        import shutil
        shutil.copyfile(source,temporary)
        if sha256(temporary)!=current_hash:raise RuntimeError('Source changed during copy or copy corrupted.')
        temporary.replace(target)
        write_json(marker,dict(source=str(source),bytes=target.stat().st_size,sha256=current_hash))
    finally:
        if temporary.exists():temporary.unlink()
    return target,current_hash
print('Source content and local copy are verified on every run.')
complete('V9-04')


## Voice validation and fixed subject split

Raw recordings are retained for later subject-grouped model fitting. PD=1, Control=0. The known previously evaluated subject holdout is preserved and labelled retrospective.


In [ ]:
VOICE_LOCAL,voice_hash=copy_verified(VOICE_PATH,OUT/'raw_local_cache/voice/parkinsons.data')
voice_raw=pd.read_csv(VOICE_LOCAL)
voice_raw['subject_id']=voice_raw['name'].str.extract(r'^(phon_R\d+_S\d+)_\d+$')[0]
assert voice_raw.subject_id.notna().all() and voice_raw.groupby('subject_id').status.nunique().max()==1
VOICE_FEATURES=[c for c in voice_raw if c not in ['name','status','subject_id']]
assert len(VOICE_FEATURES)==22 and np.isfinite(voice_raw[VOICE_FEATURES].to_numpy(float)).all()
voice_subjects=voice_raw.groupby('subject_id').status.first().rename('label').reset_index()
voice_subjects=assign_split(voice_subjects,'voice',OUT/'tables',SEED)
print('Voice recordings:',len(voice_raw),'| subjects:',len(voice_subjects));print(pd.crosstab(voice_subjects.split,voice_subjects.label))
print('Holdout status: previously evaluated subjects; retrospective reuse')

complete('V9-05')


## Spiral trajectory parsing, features and timestamp audit

The primary binary cohort is 15 controls + 25 PD subjects. The additional 37 PD-only subjects remain separate. Task transitions and nonpositive intervals are not used to compute speed; millisecond units follow the supplied original notebook and are not independently validated here.


In [ ]:
spiral_rows=[];acq_rows=[];hashes={}
groups=[('Healthy',SPIRAL_ROOT/'hw_dataset/control',0,'original'),
 ('Parkinson',SPIRAL_ROOT/'hw_dataset/parkinson',1,'original'),
 ('New_Parkinson',SPIRAL_ROOT/'new_dataset/parkinson',1,'pd_only')]
for group,folder,label,cohort in groups:
    if not folder.is_dir():raise FileNotFoundError(f'Missing Spiral group: {folder}')
    files=sorted(folder.glob('*.txt'));assert files,f'No TXT files in {folder}'
    for k,source in enumerate(files,1):
        deadline_check();print(f'{group} {k}/{len(files)} | {source.name}',flush=True)
        local,h=copy_verified(source,OUT/'raw_local_cache/spiral'/group/source.name)
        assert h not in hashes,f'Duplicate Spiral files: {source} and {hashes.get(h)}';hashes[h]=str(source)
        features,audit=trajectory_features(local);sid=f'{group}/{source.stem}'
        spiral_rows.append(dict(subject_id=sid,label=label,cohort=cohort,**features))
        acq_rows.append(dict(subject_id=sid,label=label,cohort=cohort,**audit))
        print('  Subject extraction COMPLETED',flush=True)
spiral_all=pd.DataFrame(spiral_rows);acq_all=pd.DataFrame(acq_rows)
assert spiral_all.subject_id.is_unique and acq_all.subject_id.is_unique
assert len(spiral_all.query("cohort=='original'"))==40 and len(spiral_all.query("cohort=='pd_only'"))==37
assert not np.isinf(spiral_all.filter(regex='^test_').to_numpy(float)).any()
spiral_all.to_csv(OUT/'tables/spiral_all_features.csv',index=False);acq_all.to_csv(OUT/'tables/spiral_acquisition_audit.csv',index=False)
spiral_df=assign_split(spiral_all.query("cohort=='original'").copy(),'spiral',OUT/'tables',SEED)
print(pd.crosstab(spiral_df.split,spiral_df.label));print('Acquisition confounding: UNRESOLVED until independently investigated')

complete('V9-06')


## MRI clinical matching, local copies and fixed split

Only one structural T1 volume per verified clinical subject is used. Header/affine checks are performed now; voxel integrity is checked during preprocessing. No directory-name-only clinical labels.


In [ ]:
import nibabel as nib
clinical=pd.read_csv(MRI_ROOT/'neurocon_patients.tsv',sep='\t',dtype=str,keep_default_na=False)
clinical.columns=clinical.columns.str.strip();assert {'code','status'}<=set(clinical.columns)
clinical['subject_id']='sub-'+clinical.code.str.strip()
clinical[['code','status','subject_id']].to_csv(OUT/'tables/clinical_source_mapping.csv',index=False)
write_json(OUT/'tables/clinical_source_provenance.json',dict(path=str(MRI_ROOT/'neurocon_patients.tsv'),sha256=sha256(MRI_ROOT/'neurocon_patients.tsv'),label_mapping={'NORMAL':0,'PD':1},labels_inferred_from_filename=False))
clinical['label']=clinical.status.str.strip().str.upper().map({'NORMAL':0,'PD':1})
assert len(clinical)==43 and clinical.subject_id.is_unique and clinical.label.notna().all()
mri_rows=[];hashes={}
for k,row in enumerate(clinical.itertuples(index=False),1):
    deadline_check();print(f'MRI input {k}/43 | {row.subject_id}',flush=True)
    subject_folder=MRI_ROOT/row.subject_id;candidates=[]
    for folder in [subject_folder/'anat',subject_folder]:
        if folder.is_dir():candidates.extend(p for p in folder.glob('*T1w.nii*') if p.is_file())
    candidates=sorted(set(candidates));assert len(candidates)==1,f'Expected one T1 MRI for {row.subject_id}; found {len(candidates)}'
    source=candidates[0];local,h=copy_verified(source,OUT/'raw_local_cache/mri'/row.subject_id/source.name)
    assert local.stat().st_size>1024,'MRI file too small'
    assert h not in hashes,f'Duplicate MRI subjects: {row.subject_id} and {hashes.get(h)}';hashes[h]=row.subject_id
    im=nib.load(str(local));assert len(im.shape)==3
    assert np.isfinite(im.affine).all() and abs(np.linalg.det(im.affine[:3,:3]))>1e-8
    if not source.name.startswith(row.subject_id+'_'):raise ValueError('T1 filename does not match clinical subject: '+row.subject_id)
    spatial_unit,temporal_unit=im.header.get_xyzt_units()
    spacing=tuple(float(v) for v in im.header.get_zooms()[:3]);axes=''.join(nib.aff2axcodes(im.affine))
    if not all(v>0 and np.isfinite(v) for v in spacing):raise ValueError('Invalid voxel spacing.')
    folder_hint='CONTROL' if row.subject_id.lower().startswith('sub-control') else ('PD' if row.subject_id.lower().startswith('sub-patient') else 'UNKNOWN')
    hint_conflict=bool(folder_hint!='UNKNOWN' and (folder_hint=='PD')!=bool(row.label))
    mri_rows.append(dict(subject_id=row.subject_id,label=int(row.label),path=str(local),sha256=h,source_filename=source.name,folder_hint=folder_hint,hint_conflict=hint_conflict,shape_x=int(im.shape[0]),shape_y=int(im.shape[1]),shape_z=int(im.shape[2]),spacing_x=spacing[0],spacing_y=spacing[1],spacing_z=spacing[2],orientation=axes,spatial_unit=spatial_unit,datatype=str(im.get_data_dtype())))
    pd.DataFrame(mri_rows).to_csv(OUT/'tables/mri_inventory_progress.csv',index=False)
    print('  Header, affine and local hash VERIFIED',flush=True)
mri_df=pd.DataFrame(mri_rows)
if MRI_SPLIT_OVERRIDE:
    split=pd.read_csv(MRI_SPLIT_OVERRIDE)[['subject_id','split']]
    assert split.subject_id.is_unique and set(split.subject_id)==set(mri_df.subject_id)
    mri_df=mri_df.merge(split,on='subject_id',validate='one_to_one')
    assert set(mri_df.split)=={'development','test'} and (mri_df.split=='test').sum()==11
else:mri_df=assign_split(mri_df,'mri',OUT/'tables',SEED)
mri_df.to_csv(OUT/'tables/mri_inventory.csv',index=False)
audit_cols=['subject_id','label','split','source_filename','folder_hint','hint_conflict','shape_x','shape_y','shape_z','spacing_x','spacing_y','spacing_z','orientation','spatial_unit','datatype','sha256']
mri_df[audit_cols].to_csv(OUT/'tables/mri_label_header_audit.csv',index=False)
geometry=['shape_x','shape_y','shape_z','spacing_x','spacing_y','spacing_z','orientation','spatial_unit','datatype']
mri_df.groupby(geometry+['label'],dropna=False).size().rename('subjects').reset_index().to_csv(OUT/'tables/mri_geometry_by_label.csv',index=False)
print(pd.crosstab(mri_df.split,mri_df.label))
print('Clinical/folder-hint conflicts (review, never auto-relabel):',int(mri_df.hint_conflict.sum()))

complete('V9-07')


## Official SynthStrip and MNI reference

Official pretrained SynthStrip weights download and SHA-256 verification; MNI152 fixed reference. Internet is required. Failure does not trigger an unvalidated skull-stripping fallback.


In [ ]:
import urllib.request
import SimpleITK as sitk
from nilearn.datasets import load_mni152_template,load_mni152_brain_mask
MRI_DETERMINISTIC=True   # single-threaded registration -> bit-identical MRI results across runs (slower)
sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(1 if MRI_DETERMINISTIC else 2)
SCRIPT=OUT/'tools/mri_synthstrip';WEIGHTS=OUT/'tools/synthstrip.1.pt'
assets=[(SCRIPT,'https://raw.githubusercontent.com/freesurfer/freesurfer/dev/mri_synthstrip/mri_synthstrip'),(WEIGHTS,'https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/requirements/synthstrip.1.pt')]
for path,url in assets:
    if not path.exists():
        tmp=path.with_suffix('.partial');urllib.request.urlretrieve(url,tmp);tmp.replace(path)
assert sha256(WEIGHTS)=='37417f802196186441aae3e7f385d94f8a98c64a88acaeaa2723af995c653e33','SynthStrip weight hash mismatch'
template=load_mni152_template(resolution=2);template_mask=load_mni152_brain_mask(resolution=2)
TEMPLATE_PATH=OUT/'tools/mni_t1_2mm.nii.gz';TEMPLATE_MASK=OUT/'tools/mni_mask_2mm.nii.gz'
mask_a=template_mask.get_fdata()>0
nib.save(nib.Nifti1Image((template.get_fdata()*mask_a).astype('float32'),template.affine),TEMPLATE_PATH)
nib.save(nib.Nifti1Image(mask_a.astype('uint8'),template.affine),TEMPLATE_MASK)
ASSET_HASHES={p.name:sha256(p) for p in [SCRIPT,WEIGHTS,TEMPLATE_PATH,TEMPLATE_MASK]}
write_json(OUT/'tables/asset_hashes.json',ASSET_HASHES)
print('SynthStrip checkpoint: VERIFIED');print('MNI152 2-mm reference: READY')

complete('V9-08')


## MRI preprocessing implementation

SynthStrip → brain-specific N4 → fixed masked rigid registration → MNI resampling → brain-mask percentile normalization. Fixed preprocessing is applied independently to each subject; no holdout label tunes the transforms. Rigid registration is not nonlinear atlas morphometry.


In [ ]:
import gc
fixed=sitk.ReadImage(str(TEMPLATE_PATH),sitk.sitkFloat32)
fixed_mask=sitk.ReadImage(str(TEMPLATE_MASK),sitk.sitkUInt8)
ref=sitk.GetArrayFromImage(fixed_mask)>0
PROTOCOL=dict(version='9.0',n4_iterations=[30,20,10],n4_estimation_shrink=[2,2,2],bias_application='native full-resolution grid before registration',
 registration='masked rigid Euler3D; mask-moment initialization',metric='MattesMutualInformation',metric_bins=40,
 sampling='RANDOM',sampling_fraction=.25,seed=SEED,optimizer='RegularStepGradientDescent',learning_rate=1.,
 min_step=.0001,iterations=250,relaxation_factor=.6,optimizer_scales='physical_shift',
 pyramid_shrink=[4,2,1],pyramid_smoothing_mm=[2,1,0],interpolator='linear',mask_interpolator='nearest',
 normalization=dict(percentiles=[1,99],scope='subject own aligned brain mask',clip=[0,1],background=0),
 array_order='SimpleITK GetArrayFromImage: z,y,x; physical reference grid recorded separately',
 reference_grid=dict(size=list(fixed.GetSize()),spacing=list(fixed.GetSpacing()),origin=list(fixed.GetOrigin()),direction=list(fixed.GetDirection())),
 slice_sampling=dict(fractions=[.25,.75],frozen_per_axis=FROZEN_SLICES_PER_VIEW,cnn_per_axis=CNN_SLICES_PER_VIEW,
 resize=[224,224],cnn_resize=[160,160],imagenet_mean=[.485,.456,.406],imagenet_std=[.229,.224,.225]),
 synthstrip_command='python mri_synthstrip -i INPUT -o BRAIN -m MASK --model WEIGHTS --threads 2',
 assets=ASSET_HASHES,software={p:__import__(p).__version__ for p in ['numpy','nibabel','SimpleITK']})
PROTOCOL['implementation_sha256']=sha256(OUT/'mri_preprocessing_cell.py')
PROTOCOL_HASH=hashlib.sha256(json.dumps(PROTOCOL,sort_keys=True).encode()).hexdigest()
write_json(OUT/'tables/mri_preprocessing_protocol.json',PROTOCOL)

def preprocess_one(row,reference_mask_array=ref):
    deadline_check();folder=OUT/'mri_cache'/row.subject_id;folder.mkdir(exist_ok=True)
    marker=folder/'completed.json'
    if marker.exists():
        record=json.loads(marker.read_text())
        assert record['protocol_hash']==PROTOCOL_HASH and record['input_hash']==row.sha256,'Cache protocol/input mismatch; use a fresh output directory.'
        assert sha256(folder/'normalized.npy')==record['array_hash'],'Cache array corrupted'
        for filename,key in [('mask.nii.gz','native_mask_hash'),('registered_mask.nii.gz','registered_mask_hash'),('n4_corrected_native_brain.nii.gz','native_corrected_hash')]:
            assert sha256(folder/filename)==record[key],'Review image cache corrupted: '+filename
        return record
    brain=folder/'brain.nii.gz';mask=folder/'mask.nii.gz'
    # Unfinished cache is not trusted after a changed input/protocol.
    for stale in [brain,mask]:
        if stale.exists():stale.unlink()
    if not (brain.exists() and mask.exists()):
        cmd=[sys.executable,str(SCRIPT),'-i',row.path,'-o',str(brain),'-m',str(mask),'--model',str(WEIGHTS),'--threads','2']
        result=subprocess.run(cmd,capture_output=True,text=True,timeout=1200)
        (folder/'synthstrip.log').write_text(result.stdout+'\n'+result.stderr)
        if result.returncode:raise RuntimeError(f'SynthStrip failed for {row.subject_id}; see log: {result.stderr[-500:]}')
    original=sitk.ReadImage(row.path,sitk.sitkFloat32);bm=sitk.ReadImage(str(mask),sitk.sitkUInt8)
    assert original.GetSize()==bm.GetSize()
    np.testing.assert_allclose(original.GetSpacing(),bm.GetSpacing(),atol=1e-4)
    np.testing.assert_allclose(original.GetDirection(),bm.GetDirection(),atol=1e-4)
    original_array=sitk.GetArrayFromImage(original)
    assert np.isfinite(original_array).all() and np.any(original_array!=0),'Non-finite or empty MRI voxels'
    del original_array
    np.testing.assert_allclose(original.GetOrigin(),bm.GetOrigin(),atol=1e-4)
    # বাংলা: ছোট image দিয়ে bias field estimate; corrected image full-resolution-এ থাকবে।
    positive_mask=sitk.Cast((bm>0)&(original>0),sitk.sitkUInt8)
    small=sitk.Shrink(original,[2,2,2]);small_mask=sitk.Shrink(positive_mask,[2,2,2])
    n4=sitk.N4BiasFieldCorrectionImageFilter();n4.SetMaximumNumberOfIterations([30,20,10])
    n4.Execute(small,small_mask)
    log_bias=n4.GetLogBiasFieldAsImage(original)
    corrected=sitk.Cast(original/sitk.Exp(log_bias),sitk.sitkFloat32)
    ms=sitk.Cast(bm>0,sitk.sitkUInt8)
    moving=sitk.Mask(corrected,ms)
    assert moving.GetSize()==original.GetSize(),'Native resolution must be preserved.'
    sitk.WriteImage(moving,str(folder/'n4_corrected_native_brain.nii.gz'))
    native_mask_array=sitk.GetArrayFromImage(ms)>0
    native_brain_volume_ml=float(native_mask_array.sum()*np.prod(original.GetSpacing())/1000.)
    native_mask_fraction=float(native_mask_array.mean())
    from scipy.ndimage import label as component_label
    components,count=component_label(native_mask_array)
    sizes=np.bincount(components.ravel())[1:]
    largest_component_fraction=float(sizes.max()/sizes.sum()) if len(sizes) else 0.
    del components,native_mask_array
    initial=sitk.CenteredTransformInitializer(sitk.Cast(fixed_mask,sitk.sitkFloat32),sitk.Cast(ms,sitk.sitkFloat32),sitk.Euler3DTransform(),sitk.CenteredTransformInitializerFilter.MOMENTS)
    reg=sitk.ImageRegistrationMethod();reg.SetMetricAsMattesMutualInformation(40)
    reg.SetMetricFixedMask(fixed_mask);reg.SetMetricMovingMask(ms)
    reg.SetMetricSamplingStrategy(reg.RANDOM);reg.SetMetricSamplingPercentage(.25,SEED)
    reg.SetInterpolator(sitk.sitkLinear);reg.SetOptimizerAsRegularStepGradientDescent(learningRate=1.,minStep=.0001,numberOfIterations=250,relaxationFactor=.6)
    reg.SetOptimizerScalesFromPhysicalShift();reg.SetShrinkFactorsPerLevel([4,2,1]);reg.SetSmoothingSigmasPerLevel([2,1,0]);reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    reg.SetInitialTransform(initial,inPlace=False);transform=reg.Execute(fixed,moving)
    aligned=sitk.Resample(moving,fixed,transform,sitk.sitkLinear,0.,sitk.sitkFloat32)
    aligned_mask=sitk.Resample(ms,fixed,transform,sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
    optimizer_metric=float(reg.GetMetricValue())
    if not np.isfinite(optimizer_metric):raise RuntimeError('Non-finite registration metric.')
    a=sitk.GetArrayFromImage(aligned).astype('float32');m=sitk.GetArrayFromImage(aligned_mask)>0
    assert np.isfinite(a).all() and m.sum()>100
    # বাংলা: intensity normalization প্রতি subject-এর নিজের brain mask থেকে; cohort statistics নয়।
    lo,hi=np.percentile(a[m],[1,99]);assert hi>lo
    normalized=np.clip((a-lo)/(hi-lo),0,1);normalized[~m]=0
    np.save(folder/'normalized.npy',normalized.astype('float32'))
    sitk.WriteImage(aligned,str(folder/'registered_brain.nii.gz'));sitk.WriteImage(aligned_mask,str(folder/'registered_mask.nii.gz'))
    sitk.WriteTransform(transform,str(folder/'rigid_transform.h5'))
    intersection=(reference_mask_array&m).sum();dice=2*intersection/(reference_mask_array.sum()+m.sum());coverage=intersection/reference_mask_array.sum()
    active=np.where(reference_mask_array.sum(axis=(1,2))>0)[0];edge=active[:max(1,len(active)//5)]
    inferior=float((reference_mask_array[edge]&m[edge]).sum()/reference_mask_array[edge].sum())
    fig,axes=plt.subplots(3,2,figsize=(8,10),constrained_layout=True)
    for axidx,axis in enumerate([0,1,2]):
        i=a.shape[axis]//2
        axes[axidx,0].imshow(np.rot90(np.take(normalized,i,axis=axis)),cmap='gray',vmin=0,vmax=1)
        axes[axidx,1].imshow(np.rot90(np.take(normalized,i,axis=axis)),cmap='gray',vmin=0,vmax=1)
        axes[axidx,1].contour(np.rot90(np.take(reference_mask_array,i,axis=axis)),levels=[.5],colors=['red'],linewidths=.5)
        for ax in axes[axidx]:ax.axis('off')
    fig.suptitle(f'{row.subject_id} | anatomical QC pending | Dice={dice:.3f}')
    fig.savefig(folder/'qc.png',dpi=150);plt.close(fig)
    record=dict(subject_id=row.subject_id,protocol_hash=PROTOCOL_HASH,input_hash=row.sha256,
       array_hash=sha256(folder/'normalized.npy'),dice=float(dice),coverage=float(coverage),inferior_coverage=inferior,
       native_mask_hash=sha256(mask),registered_mask_hash=sha256(folder/'registered_mask.nii.gz'),native_corrected_hash=sha256(folder/'n4_corrected_native_brain.nii.gz'),
       native_brain_volume_ml=native_brain_volume_ml,native_mask_fraction=native_mask_fraction,largest_component_fraction=largest_component_fraction,registration_metric=optimizer_metric,
       priority_review=bool(dice<.82 or coverage<.75 or inferior<.5 or largest_component_fraction<.95),stop_condition=reg.GetOptimizerStopConditionDescription(),visual_qc='PENDING')
    write_json(marker,record);gc.collect();return record

complete('V9-09')


## Run checkpointed MRI preprocessing

Each subject has a completion marker, normalized array, registered mask/volume, transform and QC image. An interruption preserves earlier completed subjects. Automated overlap scores flag review, not anatomical approval.


In [ ]:
mri_qc=[]
for k,row in enumerate(mri_df.itertuples(index=False),1):
    print(f'MRI preprocessing {k}/{len(mri_df)} | {row.subject_id}',flush=True)
    try:record=preprocess_one(row)
    except Exception:
        write_json(OUT/'logs/preprocessing_failure.json',dict(subject_id=row.subject_id,traceback=traceback.format_exc()))
        raise
    write_json(OUT/'mri_cache'/row.subject_id/'inference_metadata.json',dict(subject_id=row.subject_id,array_hash=record['array_hash'],protocol_hash=record['protocol_hash'],array_order='z,y,x',reference_grid=PROTOCOL['reference_grid'],normalization=PROTOCOL['normalization']))
    mri_qc.append(record);pd.DataFrame(mri_qc).to_csv(OUT/'tables/mri_qc.csv',index=False)
    print(f'Subject completed | Dice={record["dice"]:.3f} | review={record["priority_review"]}',flush=True)
print('All scans preprocessed:',len(mri_qc),'| anatomical QC approval: PENDING')

complete('V9-10')


## Inspect MRI QC before model training

Review all generated panels. To document anatomical acceptance, populate QC_REVIEW_CSV and rerun the following review-status cell. Without review the experiment remains exploratory.


In [ ]:
PROJECT=OUT
FIGURE_OUTPUT=OUT/'figure_review_v4'
FIGURE_OUTPUT.mkdir(exist_ok=True)
MRI_SUBJECT_LIMIT=None
MRI_PREVIEW_COUNT=2
FIGURE_MODULE='\nfrom pathlib import Path\nimport json,hashlib\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import MaxNLocator\nfrom sklearn.metrics import roc_curve,roc_auc_score,average_precision_score,confusion_matrix,PrecisionRecallDisplay,balanced_accuracy_score,accuracy_score,brier_score_loss\nCOLORS={\'voice\':\'#0072B2\',\'spiral\':\'#009E73\',\'mri\':\'#D55E00\'}\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for block in iter(lambda:f.read(2**20),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef style():\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':10,\'axes.titlesize\':11,\'axes.labelsize\':10,\n      \'axes.spines.top\':False,\'axes.spines.right\':False,\'axes.linewidth\':.8,\'lines.linewidth\':1.8,\n      \'figure.dpi\':110,\'savefig.dpi\':300,\'pdf.fonttype\':42,\'ps.fonttype\':42,\'svg.fonttype\':\'none\'})\n\ndef save(fig,stem,show=True):\n    stem=Path(stem);stem.parent.mkdir(parents=True,exist_ok=True)\n    for ext in [\'png\',\'pdf\',\'svg\']:fig.savefig(stem.with_suffix(\'.\'+ext),dpi=300,bbox_inches=\'tight\',facecolor=\'white\')\n    if show:plt.show()\n    plt.close(fig)\n\ndef validate(frame):\n    required={\'subject_id\',\'label\',\'probability_pd\'}\n    if not required<=set(frame):raise ValueError(\'Prediction CSV missing columns: \'+str(required-set(frame)))\n    if not frame.subject_id.is_unique:raise ValueError(\'Prediction rows must be unique subjects.\')\n    y=frame.label.to_numpy(int);p=frame.probability_pd.to_numpy(float)\n    if set(y)!={0,1} or not np.isfinite(p).all() or np.any((p<0)|(p>1)):raise ValueError(\'Invalid binary labels or scores.\')\n    return y,p\n\ndef roc_band(y,p,repeats=1000,seed=42):\n    # বাংলা: bootstrap band-এর interpolation শুধু uncertainty display; observed ROC বদলানো নয়।\n    rng=np.random.default_rng(seed);grid=np.linspace(0,1,101);samples=[];aucs=[]\n    neg=np.flatnonzero(y==0);pos=np.flatnonzero(y==1)\n    for _ in range(repeats):\n        ix=np.r_[rng.choice(neg,len(neg),replace=True),rng.choice(pos,len(pos),replace=True)]\n        f,t,_=roc_curve(y[ix],p[ix],drop_intermediate=False)\n        samples.append(np.interp(grid,f,t));aucs.append(roc_auc_score(y[ix],p[ix]))\n    return grid,np.percentile(samples,2.5,axis=0),np.percentile(samples,97.5,axis=0),np.percentile(aucs,[2.5,97.5])\n\ndef wilson(k,n):\n    z=1.95996398454;q=k/n;den=1+z*z/n\n    center=(q+z*z/(2*n))/den;delta=z*np.sqrt(q*(1-q)/n+z*z/(4*n*n))/den\n    return max(0,center-delta),min(1,center+delta)\n\ndef reliability_table(y,p,bins=3):\n    # বাংলা: equal-frequency bin; tied scores থাকলে bin জোর করে বাড়ানো হবে না।\n    edges=np.unique(np.quantile(p,np.linspace(0,1,bins+1)))\n    groups=np.zeros(len(p),int) if len(edges)<2 else np.searchsorted(edges[1:-1],p,side=\'right\')\n    rows=[]\n    for group in np.unique(groups):\n        ix=groups==group;n=int(ix.sum());k=int(y[ix].sum());lo,hi=wilson(k,n)\n        rows.append(dict(bin=int(group),n=n,mean_score=float(p[ix].mean()),observed_pd=float(k/n),ci_low=lo,ci_high=hi))\n    return pd.DataFrame(rows)\n\ndef performance_figures(frame,name,cohort,out,repeats=1000):\n    style();y,p=validate(frame);color=COLORS[name];out=Path(out)\n    cuts=frame.decision_threshold.to_numpy(float) if \'decision_threshold\' in frame else np.full(len(frame),.5)\n    decision=(p>=cuts).astype(int)\n    n=len(y);n0=int((y==0).sum());n1=int((y==1).sum());tag=f\'{name}_{cohort}\'\n    header=f\'{name.upper()} | {cohort.replace("_"," ")} | N={n} ({n0} Control, {n1} PD)\'\n    fpr,tpr,thresholds=roc_curve(y,p,drop_intermediate=False);auc=roc_auc_score(y,p)\n    grid,lo,hi,auc_ci=roc_band(y,p,repeats)\n    fig,ax=plt.subplots(figsize=(5.4,4.7),constrained_layout=True)\n    ax.fill_between(grid,lo,hi,color=color,alpha=.13,label=\'95% stratified subject-bootstrap band\')\n    ax.plot(fpr,tpr,\'o-\',color=color,markersize=3,label=f\'Observed ROC; AUC={auc:.3f}\')\n    ax.plot([0,1],[0,1],\'--\',color=\'.55\',linewidth=1,label=\'Chance\')\n    ax.set(xlim=(-.02,1.02),ylim=(-.02,1.02),xlabel=\'False positive rate\',ylabel=\'True positive rate\',title=header)\n    ax.legend(loc=\'lower right\',fontsize=8);ax.set_aspect(\'equal\',adjustable=\'box\')\n    save(fig,out/(tag+\'_roc\'))\n    pd.DataFrame({\'fpr\':fpr,\'tpr\':tpr,\'threshold\':thresholds}).to_csv(out/(tag+\'_roc_points.csv\'),index=False)\n    fig,ax=plt.subplots(figsize=(5.4,4.7),constrained_layout=True)\n    PrecisionRecallDisplay.from_predictions(y,p,ax=ax,color=color,name=\'Observed PR\',drawstyle=\'steps-post\')\n    ax.axhline(y.mean(),ls=\'--\',color=\'.55\',linewidth=1,label=f\'Prevalence={y.mean():.3f}\')\n    ax.set(xlim=(-.02,1.02),ylim=(-.02,1.02),xlabel=\'Recall\',ylabel=\'Precision\',title=header)\n    ax.legend(loc=\'lower left\',fontsize=8);save(fig,out/(tag+\'_pr\'))\n    tab=reliability_table(y,p,3);tab.to_csv(out/(tag+\'_reliability_bins.csv\'),index=False)\n    fig,axes=plt.subplots(1,2,figsize=(10.2,4.2),constrained_layout=True)\n    axes[0].plot([0,1],[0,1],\'--\',color=\'.55\',linewidth=1,label=\'Reference: observed = score\')\n    for i,row in enumerate(tab.itertuples(index=False)):\n        axes[0].errorbar([row.mean_score],[row.observed_pd],yerr=[[row.observed_pd-row.ci_low],[row.ci_high-row.observed_pd]],fmt=[\'o\',\'s\',\'^\'][i%3],color=[\'#0072B2\',\'#D55E00\',\'#009E73\'][i%3],markersize=6,capsize=4,label=f\'Quantile bin {i+1}: n={row.n}\')\n    axes[0].set(xlim=(-.02,1.02),ylim=(-.02,1.02),xlabel=\'Mean uncalibrated PD-class score\',ylabel=\'Observed PD fraction\',title=\'Reliability diagnostic; 95% binomial intervals\')\n    axes[0].legend(loc=\'lower right\',fontsize=8)\n    # বাংলা: সীমিত score range-এ সব subject আলাদাভাবে দেখানো; score value বদলানো নয়।\n    for label,offset in [(0,0),(1,1)]:\n        axes[1].scatter(p[y==label],np.full((y==label).sum(),offset),color=[\'#0072B2\',\'#D55E00\'][label],s=35,alpha=.8,label=[\'Control\',\'PD\'][label])\n    pad=max(.02,float(np.ptp(p))*.15);left=max(0,float(p.min())-pad);right=min(1,float(p.max())+pad)\n    if left==right:left=max(0,left-.01);right=min(1,right+.01)\n    axes[1].set(xlim=(left,right),ylim=(-.4,1.4),yticks=[0,1],yticklabels=[\'Control\',\'PD\'],xlabel=\'Actual PD-class score (zoomed axis)\',title=f\'Score range: {p.min():.4f}–{p.max():.4f}\')\n    fig.suptitle(header+\' | scores are not calibrated clinical risk\')\n    save(fig,out/(tag+\'_reliability_and_scores\'))\n    cm=confusion_matrix(y,decision,labels=[0,1]);normal=cm/cm.sum(axis=1,keepdims=True)\n    fig,axes=plt.subplots(1,2,figsize=(8.4,3.8),constrained_layout=True)\n    for ax,a,title in [(axes[0],cm,\'Counts at recorded cuts (default 0.50)\'),(axes[1],normal,\'Within-class fractions\')]:\n        ax.imshow(a,cmap=\'Blues\',vmin=0,vmax=cm.max() if a is cm else 1)\n        for i in range(2):\n            for j in range(2):ax.text(j,i,str(cm[i,j]) if a is cm else f\'{a[i,j]:.0%}\',ha=\'center\',va=\'center\',color=\'white\' if a[i,j]>(cm.max()/2 if a is cm else .5) else \'black\')\n        ax.set(xticks=[0,1],yticks=[0,1],xticklabels=[\'Control\',\'PD\'],yticklabels=[\'Control\',\'PD\'],xlabel=\'Predicted class\',ylabel=\'True class\',title=title)\n    fig.suptitle(header);save(fig,out/(tag+\'_confusion\'))\n    return dict(name=name,cohort=cohort,n=n,accuracy=float(accuracy_score(y,decision)),balanced_accuracy=float(balanced_accuracy_score(y,decision)),roc_auc=float(auc),auc_ci_low=float(auc_ci[0]),auc_ci_high=float(auc_ci[1]),average_precision=float(average_precision_score(y,p)),brier=float(brier_score_loss(y,p)),specificity=float(normal[0,0]),score_min=float(p.min()),score_max=float(p.max()),nonempty_reliability_bins=len(tab))\n\ndef loss_history_figures(table_dir,out,preview=2):\n    style();paths=sorted(Path(table_dir).glob(\'cnn_*_early_stopping_history.csv\'));records=[]\n    for k,path in enumerate(paths):\n        early=pd.read_csv(path);refit_path=path.with_name(path.name.replace(\'_early_stopping_history\',\'_refit_history\'))\n        if not refit_path.exists():continue\n        refit=pd.read_csv(refit_path);tag=path.name.removesuffix(\'_early_stopping_history.csv\')\n        fig,axes=plt.subplots(1,2,figsize=(10.5,4.1),constrained_layout=True)\n        axes[0].plot(early.epoch,early.train_loss,\'o-\',label=\'Training\',markersize=4)\n        axes[0].plot(early.epoch,early.validation_loss,\'s-\',label=\'Internal validation\',markersize=4)\n        selected=int(refit.epoch.max());axes[0].axvline(selected,color=\'.4\',ls=\'--\',lw=1,label=f\'Selected epoch={selected}\')\n        axes[0].set_title(\'Train-only early stopping\');axes[0].legend(fontsize=8)\n        axes[1].plot(refit.epoch,refit.train_loss,\'o-\',color=\'#0072B2\',markersize=6,label=\'Refit training\')\n        axes[1].set_title(\'Refit on all fit-input subjects\')\n        if len(refit)==1:axes[1].annotate(\'Only 1 measured epoch\',xy=(1,refit.train_loss.iloc[0]),xytext=(.5,.85),textcoords=\'axes fraction\',ha=\'center\',arrowprops={\'arrowstyle\':\'->\',\'color\':\'.4\'})\n        for ax,history in zip(axes,[early,refit]):\n            ax.set(xlim=(.5,max(1,int(history.epoch.max()))+.5),ylim=(0,None),xlabel=\'Epoch\',ylabel=\'Weighted BCE loss\')\n            last_epoch=max(1,int(history.epoch.max()));ax.set_xticks(np.arange(1,last_epoch+1,max(1,int(np.ceil(last_epoch/10)))));ax.grid(alpha=.15)\n        fig.suptitle(tag+\' | actual measured values; no smoothing\')\n        save(fig,Path(out)/tag,show=k<preview)\n        records.append(dict(fit=tag,early_stopping_epochs=len(early),refit_epochs=len(refit),selected_epoch=selected))\n    return pd.DataFrame(records)\n\ndef panel(a,axis,index):\n    # বাংলা: canonical RAS array; transpose + origin lower-এ neurological display।\n    return np.take(a,index,axis=axis).T\n\ndef mri_montage(array,mask,spacing,tag,stem,reference_mask=None,show=False):\n    style();a=np.asarray(array,np.float32);mask=np.asarray(mask,bool)\n    if a.ndim!=3 or a.shape!=mask.shape or not mask.any() or not np.isfinite(a).all():raise ValueError(\'Invalid MRI/mask for QC display.\')\n    lo,hi=np.percentile(a[mask],[1,99.5]);display=np.clip((a-lo)/max(float(hi-lo),1e-8),0,1);display[~mask]=0\n    fig,axes=plt.subplots(3,5,figsize=(13,8.8),layout=\'constrained\',facecolor=\'black\')\n    views=[(\'Axial\',2,(\'L\',\'R\')),(\'Coronal\',1,(\'L\',\'R\')),(\'Sagittal\',0,(\'P\',\'A\'))]\n    rows=[]\n    for r,(name,axis,sides) in enumerate(views):\n        active=np.flatnonzero(mask.sum(axis=tuple(i for i in range(3) if i!=axis))>0)\n        indices=np.rint(np.linspace(active[0],active[-1],7)[1:-1]).astype(int)\n        others=[i for i in range(3) if i!=axis];sx,sy=float(spacing[others[0]]),float(spacing[others[1]])\n        for c,index in enumerate(indices):\n            ax=axes[r,c];sl=panel(display,axis,index);ms=panel(mask,axis,index)\n            extent=(0,sl.shape[1]*sx,0,sl.shape[0]*sy)\n            ax.imshow(sl,cmap=\'gray\',vmin=0,vmax=1,origin=\'lower\',extent=extent,aspect=\'equal\',interpolation=\'nearest\')\n            xx=(np.arange(sl.shape[1])+.5)*sx;yy=(np.arange(sl.shape[0])+.5)*sy\n            if ms.any() and not ms.all():ax.contour(xx,yy,ms,levels=[.5],colors=[\'#64e572\'],linewidths=.45)\n            if reference_mask is not None:\n                rm=panel(reference_mask,axis,index)\n                if rm.any() and not rm.all():ax.contour(xx,yy,rm,levels=[.5],colors=[\'#ff6666\'],linewidths=.4)\n            ax.set_title(f\'{name} | slice {index}\',color=\'white\',fontsize=8);ax.axis(\'off\')\n            ax.text(.02,.5,sides[0],transform=ax.transAxes,color=\'white\',fontsize=8);ax.text(.98,.5,sides[1],transform=ax.transAxes,color=\'white\',fontsize=8,ha=\'right\')\n            rows.append(dict(view=name,array_axis=axis,slice_index=int(index)))\n    fig.suptitle(tag+\' | canonical RAS; neurological display; anatomical review pending\',color=\'white\',fontsize=11)\n    fig.text(.5,.005,\'Green: subject brain mask. Red (if present): fixed MNI brain mask. Display only; image data and model inputs unchanged.\',ha=\'center\',color=\'white\',fontsize=8)\n    stem=Path(stem);stem.parent.mkdir(parents=True,exist_ok=True)\n    for ext in [\'png\',\'pdf\',\'svg\']:fig.savefig(stem.with_suffix(\'.\'+ext),dpi=250,bbox_inches=\'tight\',facecolor=\'black\')\n    if show:plt.show()\n    plt.close(fig);return rows\n'
(FIGURE_OUTPUT/'pd_figure_review.py').write_text(FIGURE_MODULE,encoding='utf-8')
sys.path.insert(0,str(FIGURE_OUTPUT))
import pd_figure_review
importlib.reload(pd_figure_review)
from pd_figure_review import *
import nibabel as nib
qc_path=PROJECT/'tables/mri_qc.csv'
if not qc_path.is_file():raise FileNotFoundError('MRI QC CSV missing; restore the saved V3 project.')
qc=pd.read_csv(qc_path);subjects=np.random.default_rng(SEED+9009).permutation(qc.subject_id.astype(str).to_numpy()).tolist()
if MRI_SUBJECT_LIMIT is not None:subjects=subjects[:int(MRI_SUBJECT_LIMIT)]
reference_path=PROJECT/'tools/mni_mask_2mm.nii.gz'
reference=nib.as_closest_canonical(nib.load(reference_path)) if reference_path.is_file() else None
montage_records=[]
for k,sid in enumerate(subjects):
    folder=PROJECT/'mri_cache'/sid;image_path=folder/'registered_brain.nii.gz';mask_path=folder/'registered_mask.nii.gz'
    if not image_path.is_file() or not mask_path.is_file():raise FileNotFoundError(f'Registered MRI/mask missing for {sid}. Restore mri_cache, not just the compact results ZIP.')
    image=nib.as_closest_canonical(nib.load(image_path));mask_img=nib.as_closest_canonical(nib.load(mask_path))
    assert image.shape==mask_img.shape
    np.testing.assert_allclose(image.affine,mask_img.affine,atol=1e-4)
    array=np.asarray(image.dataobj,dtype=np.float32);mask=np.asarray(mask_img.dataobj)>0
    ref=None
    if reference is not None:
        assert reference.shape==image.shape
        np.testing.assert_allclose(reference.affine,image.affine,atol=1e-3)
        ref=np.asarray(reference.dataobj)>0
    details=mri_montage(array,mask,image.header.get_zooms()[:3],f'MRI QC {k+1:03d}',FIGURE_OUTPUT/'mri_qc'/f'QC_{k+1:03d}',reference_mask=ref,show=k<MRI_PREVIEW_COUNT)
    for row in details:montage_records.append(dict(qc_id=f'QC_{k+1:03d}',subject_id=sid,**row))
    print(f'MRI montage {k+1}/{len(subjects)} | COMPLETED',flush=True)
pd.DataFrame(montage_records).to_csv(FIGURE_OUTPUT/'mri_display_slice_manifest.csv',index=False)
print('Anatomical QC approval: still PENDING; no MRI preprocessing or training repeated.')

complete('V9-11')

# বাংলা: native scan + mask + native-resolution N4 correction দেখার panel।
from IPython.display import FileLink,display
import zipfile
NATIVE_REVIEW=FIGURE_OUTPUT/'mri_native_review';NATIVE_REVIEW.mkdir(exist_ok=True)
for k,sid in enumerate(subjects):
    raw_path=mri_df.set_index('subject_id').loc[sid,'path']
    folder=OUT/'mri_cache'/sid
    raw_img=nib.as_closest_canonical(nib.load(raw_path))
    native_mask=nib.as_closest_canonical(nib.load(folder/'mask.nii.gz'))
    corrected_img=nib.as_closest_canonical(nib.load(folder/'n4_corrected_native_brain.nii.gz'))
    np.testing.assert_allclose(raw_img.affine,native_mask.affine,atol=1e-4)
    np.testing.assert_allclose(raw_img.affine,corrected_img.affine,atol=1e-4)
    raw=np.asarray(raw_img.dataobj,dtype=np.float32);mask_arr=np.asarray(native_mask.dataobj)>0;corrected=np.asarray(corrected_img.dataobj,dtype=np.float32)
    if raw.shape!=corrected.shape or raw.shape!=mask_arr.shape or not mask_arr.any():raise ValueError('Native review geometry mismatch.')
    raw_lo,raw_hi=np.percentile(raw[mask_arr],[1,99.5]);n4_lo,n4_hi=np.percentile(corrected[mask_arr],[1,99.5])
    fig,axes=plt.subplots(3,6,figsize=(15,8.5),constrained_layout=True)
    for row,(axis,view) in enumerate([(2,'Axial'),(1,'Coronal'),(0,'Sagittal')]):
        active=np.flatnonzero(mask_arr.sum(axis=tuple(i for i in range(3) if i!=axis))>0)
        locations=np.rint(np.quantile(active,[.25,.5,.75])).astype(int)
        others=[i for i in range(3) if i!=axis];spacing=raw_img.header.get_zooms()
        for pair,index in enumerate(locations):
            raw_slice=np.take(raw,index,axis=axis).T;mask_slice=np.take(mask_arr,index,axis=axis).T;n4_slice=np.take(corrected,index,axis=axis).T
            extent=(0,raw_slice.shape[1]*spacing[others[0]],0,raw_slice.shape[0]*spacing[others[1]])
            a,b=axes[row,2*pair:2*pair+2]
            a.imshow(raw_slice,cmap='gray',origin='lower',extent=extent,vmin=raw_lo,vmax=raw_hi,aspect='equal')
            if mask_slice.any() and not mask_slice.all():a.contour(mask_slice,levels=[.5],colors=['lime'],linewidths=.5,origin='lower',extent=extent)
            b.imshow(n4_slice,cmap='gray',origin='lower',extent=extent,vmin=n4_lo,vmax=n4_hi,aspect='equal')
            a.set_title(f'{view} {index}: raw + mask',fontsize=8);b.set_title('Native N4 brain',fontsize=8)
            a.axis('off');b.axis('off')
    fig.suptitle(f'QC_{k+1:03d}: native anatomy and skull-strip review; diagnosis labels omitted')
    fig.savefig(NATIVE_REVIEW/f'QC_{k+1:03d}_native.png',dpi=160,bbox_inches='tight');plt.close(fig)
review_hash_fields=['subject_id','input_hash','array_hash','protocol_hash','native_mask_hash','registered_mask_hash','native_corrected_hash']
review_table=pd.DataFrame(mri_qc)[review_hash_fields+['dice','coverage','inferior_coverage','priority_review','native_brain_volume_ml','largest_component_fraction']].copy()
review_ids=pd.DataFrame({'subject_id':subjects,'qc_id':[f'QC_{k+1:03d}' for k in range(len(subjects))]})
review_table=review_ids.merge(review_table,on='subject_id',validate='one_to_one')
for field in ['decision','reviewer','evidence']:review_table[field]=''
review_table.to_csv(OUT/'tables/mri_qc_review_template.csv',index=False)
QC_PACKET=OUT/'MRI_Diagnostics_Review.zip'
with zipfile.ZipFile(QC_PACKET,'w',zipfile.ZIP_DEFLATED) as z:
    for directory in [FIGURE_OUTPUT/'mri_qc',NATIVE_REVIEW]:
        for path in sorted(directory.glob('*.png')):z.write(path,'anatomical_panels/'+path.name)
    for filename in ['mri_qc_review_template.csv','mri_qc.csv','mri_label_header_audit.csv','mri_geometry_by_label.csv','clinical_source_mapping.csv','clinical_source_provenance.json','mri_preprocessing_protocol.json']:
        z.write(OUT/'tables'/filename,'tables/'+filename)
    z.writestr('README.txt','Panel QC IDs omit diagnosis labels. The label/header audit tables contain labels for separate identity verification; this is not a formally blinded review package. Review brain inclusion, extracranial removal, brainstem/cerebellum coverage, registration and source clinical matching. Fill decision=accept/reject with reviewer and evidence, preserving every hash. Rejects require preprocessing correction, not accuracy-driven deletion. No automatic anatomical approval.\n')
print('DOWNLOAD THE DIAGNOSTIC PACKET BEFORE TRAINING:')
display(FileLink(str(QC_PACKET)))
print('This packet is sufficient for visual screening; complete cached volumes/raw data are needed for deeper investigation.')
complete('V9-DIAGNOSTIC-PACKET')


## Anatomical review status

Review all QC images and registered volumes. Supply QC_REVIEW_CSV with subject_id,decision (accept/reject) to document review. Rejected scans stop for protocol review; no performance-driven exclusions. Without documented review, downstream results remain exploratory.


In [ ]:
# বাংলা: review pending রেখেই exploratory training চালানো
REQUIRE_APPROVED_MRI_QC = False
print("Exploratory training enabled. MRI QC remains PENDING.")

In [ ]:
qc_current=pd.DataFrame(mri_qc).copy()
QC_APPROVED=False
if QC_REVIEW_CSV:
    review=pd.read_csv(QC_REVIEW_CSV,dtype=str).fillna('')
    QC_APPROVED=validate_qc_review(review,qc_current)
    review.to_csv(OUT/'tables/mri_qc_completed_review.csv',index=False)
source_conflicts=bool(mri_df.hint_conflict.any())
if source_conflicts:raise RuntimeError('Clinical/folder-hint discrepancy requires source verification; diagnoses are never automatically flipped.')
write_json(OUT/'tables/anatomical_qc_status.json',dict(approved=bool(QC_APPROVED),status='APPROVED' if QC_APPROVED else 'PENDING',publication_ready=False,review_matches_current_hashes=bool(QC_APPROVED)))
print('MRI anatomical review:', 'APPROVED' if QC_APPROVED else 'PENDING')
print('Diagnostic packet:',QC_PACKET)
if REQUIRE_APPROVED_MRI_QC and not QC_APPROVED:
    raise MRIReviewRequired('REVIEW REQUIRED, NOT A TRAINING FAILURE. Download MRI_Diagnostics_Review.zip, inspect every scan and source mapping, complete mri_qc_review_template.csv, set QC_REVIEW_CSV, then rerun this review cell and subsequent cells. All preprocessing caches are preserved. Training will not be presented as approved without review.')
complete('V9-12')


## Frozen MRI features (CPU/GPU)

ImageNet ResNet18 transfer baseline; three views × eight slices. Aggregate within each patient before model fitting. The same source encoder is the initialization for the later fine-tuning candidate. No slice-level train/test split.


In [ ]:
require_mri_review()
from torchvision.models import resnet18,ResNet18_Weights
encoder=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1);encoder.fc=torch.nn.Identity();encoder=encoder.to(DEVICE).eval()
encoder=encoder.cpu();encoder_path=OUT/'models/mri_encoder.pt'
if encoder_path.exists():
    saved=torch.load(encoder_path,map_location='cpu',weights_only=True)
    if set(saved)!=set(encoder.state_dict()) or any(not torch.equal(saved[k],v) for k,v in encoder.state_dict().items()):raise RuntimeError('Pretrained encoder cache changed; use a new versioned output folder.')
else:torch.save(encoder.state_dict(),encoder_path)
encoder=encoder.to(DEVICE)
feature_rows=[]
for k,row in enumerate(mri_df.itertuples(index=False),1):
    deadline_check();cache=OUT/'mri_cache'/row.subject_id/'embedding.npz'
    spec=PROTOCOL_HASH+f'|resnet18_imagenet1k_v1|{N_SLICES_PER_VIEW}|'+sha256(OUT/'mri_cache'/row.subject_id/'normalized.npy')+'|'+sha256(OUT/'models/mri_encoder.pt')+'|'+hashlib.sha256(PREPROCESSING_SOURCE.encode()).hexdigest()
    if cache.exists():
        with np.load(cache,allow_pickle=False) as z:
            valid=str(z['spec'])==spec
            if valid:f=z['features'].copy()
        if not valid:
            a=np.load(OUT/'mri_cache'/row.subject_id/'normalized.npy',allow_pickle=False)
            f=encode_volume(a,encoder,DEVICE,N_SLICES_PER_VIEW);np.savez_compressed(cache,features=f,spec=spec)
    else:
        a=np.load(OUT/'mri_cache'/row.subject_id/'normalized.npy')
        f=encode_volume(a,encoder,DEVICE,N_SLICES_PER_VIEW);np.savez_compressed(cache,features=f,spec=spec)
    feature_rows.append(f)
    print(f'MRI feature extraction {k}/{len(mri_df)} | completed',flush=True)
MRI_FEATURES=[f'f_{i:04d}' for i in range(len(feature_rows[0]))]
mri_features=pd.concat([mri_df[['subject_id','label','split']].reset_index(drop=True),pd.DataFrame(feature_rows,columns=MRI_FEATURES)],axis=1)
mri_features.to_csv(OUT/'tables/mri_subject_features.csv',index=False)
print('MRI features:',mri_features.shape,'| patient count:',len(mri_features))

complete('V9-13')


## Voice candidate inputs

Retain recording-level observations. Every fold splits unique subject IDs; predictions are averaged per held-out subject. Training uses equal-subject, class-balanced weights. No subject ID or recording name is a predictor.


In [ ]:
VOICE_PATH=VOICE_LOCAL
# বাংলা: exact path; recursive scan নয়।
if not VOICE_PATH.is_file():raise FileNotFoundError('Set VOICE_PATH to the original parkinsons.data classification file.')
voice_obs=pd.read_csv(VOICE_PATH)
voice_obs['subject_id']=voice_obs.name.str.extract(r'^(phon_R[0-9]+_S[0-9]+)_[0-9]+$')[0]
voice_obs=voice_obs.rename(columns={'status':'label'})
assert voice_obs.subject_id.notna().all() and voice_obs.groupby('subject_id').label.nunique().max()==1
VOICE_FEATURES=[c for c in voice_obs.columns if c not in ['name','label','subject_id']]
assert len(VOICE_FEATURES)==22 and np.isfinite(voice_obs[VOICE_FEATURES].to_numpy(float)).all()
voice_subjects=pd.read_csv(PREVIOUS_OUTPUT/'tables/voice_split.csv')
assert voice_subjects.subject_id.is_unique and set(voice_subjects.subject_id)==set(voice_obs.subject_id)
observed=voice_obs.groupby('subject_id').label.first()
assert (voice_subjects.set_index('subject_id').label==observed.reindex(voice_subjects.subject_id)).all()
VOICE_SPEC=dict(name='voice',subjects=voice_subjects,observations=voice_obs,features=VOICE_FEATURES,signature=sha256(VOICE_PATH)+sha256(PREVIOUS_OUTPUT/'tables/voice_split.csv'))
print('Voice:',len(voice_obs),'recordings;',len(voice_subjects),'subjects')
print(pd.crosstab(voice_subjects.split,voice_subjects.label))
voice_obs=clean_voice(voice_obs,VOICE_FEATURES,OUTPUT/'tables/voice_within_subject_repeated_rows.csv')
VOICE_SPEC['observations']=voice_obs
VOICE_SPEC['signature']+='|within-subject-exact-feature-dedup-v4'

complete('V9-14')


## Shape-only Spiral inputs and conservative-feature audit


In [ ]:
from pd_spiral_features import shape_features,SHAPE_FEATURE_NAMES
spiral_all=pd.read_csv(OUT/'tables/spiral_all_features.csv')
spiral_split=pd.read_csv(OUT/'tables/spiral_split.csv')
SPIRAL_CONSERVATIVE_FEATURES=[f'test_{task}_{feature}' for task in (0,1) for feature in ['x_range','y_range','median_speed','p90_speed','grip_mean','grip_std']]
SPIRAL_FEATURES=[f'shape_{task}_{feature}' for task in (0,1) for feature in SHAPE_FEATURE_NAMES]
shape_rows=[];qc_rows=[];raw_files=[]
for group,folder,label,cohort in [('Healthy',SPIRAL_ROOT/'hw_dataset/control',0,'original'),('Parkinson',SPIRAL_ROOT/'hw_dataset/parkinson',1,'original'),('New_Parkinson',SPIRAL_ROOT/'new_dataset/parkinson',1,'pd_only')]:
    for path in sorted(folder.glob('*.txt')):
        sid=f'{group}/{path.stem}'
        # Use verified local bytes, not a different unverified copy.
        local=OUT/'raw_local_cache/spiral'/group/path.name
        frame=load_trajectory(local);features,quality=shape_features(frame,RESAMPLING_POINTS)
        shape_rows.append(dict(subject_id=sid,**features));raw_files.append(dict(subject_id=sid,path=str(local)))
        qc_rows.extend(dict(subject_id=sid,cohort=cohort,**r) for r in quality)
shape_table=pd.DataFrame(shape_rows)
assert shape_table.subject_id.is_unique and set(shape_table.subject_id)==set(spiral_all.subject_id)
shape_table.to_csv(OUTPUT/'tables/spiral_shape_features_all.csv',index=False)
spiral_all=spiral_all.merge(shape_table,on='subject_id',validate='one_to_one')
task_qc=pd.DataFrame(qc_rows).merge(spiral_all[['subject_id','label']],on='subject_id',validate='many_to_one')
task_qc.to_csv(OUTPUT/'tables/spiral_task_quality.csv',index=False)
# Optional independent spiral_identity_map manifest. Labels must never define person spiral_identity_map.
spiral_identity_map=pd.DataFrame({'file_subject_id':spiral_all.subject_id,'person_id':spiral_all.subject_id})
if SPIRAL_IDENTITY_CSV:
    spiral_identity_map=pd.read_csv(SPIRAL_IDENTITY_CSV,dtype=str)
    if not {'file_subject_id','person_id'}<=set(spiral_identity_map) or spiral_identity_map[['file_subject_id','person_id']].isna().any().any():raise ValueError('Invalid Spiral spiral_identity_map manifest.')
    if not spiral_identity_map.file_subject_id.is_unique or set(spiral_identity_map.file_subject_id)!=set(spiral_all.subject_id):raise ValueError('Identity manifest must map every file exactly once.')
    if spiral_identity_map.person_id.str.strip().eq('').any():raise ValueError('Empty person spiral_identity_map.')
if SPIRAL_IDENTITY_VERIFIED and not SPIRAL_IDENTITY_EVIDENCE.strip():raise ValueError('Independent spiral_identity_map evidence is required.')
spiral_identity_map.to_csv(OUTPUT/'tables/spiral_file_person_manifest.csv',index=False)
spiral_all=spiral_all.merge(spiral_identity_map,left_on='subject_id',right_on='file_subject_id',validate='one_to_one')
if spiral_all.groupby('person_id').label.nunique().gt(1).any():raise ValueError('Conflicting labels within mapped person.')
if spiral_all.groupby('person_id').cohort.nunique().gt(1).any():raise ValueError('Mapped person overlaps original/transport cohort; resolve manifest first.')
# Aggregate file features by documented person BEFORE constructing train/test partitions.
original_split=spiral_split.set_index('subject_id').split
spiral_all['split']=spiral_all.subject_id.map(original_split).fillna('transport')
if SPIRAL_PERSON_SPLIT_CSV:
    person_split=pd.read_csv(SPIRAL_PERSON_SPLIT_CSV,dtype=str)
    if not {'person_id','split'}<=set(person_split) or not person_split.person_id.is_unique:raise ValueError('Invalid person split manifest.')
    original_people=set(spiral_all.query("cohort=='original'").person_id)
    if set(person_split.person_id)!=original_people or set(person_split.split)!={'development','test'}:raise ValueError('Person split must cover original people with development/test only.')
    mask=spiral_all.cohort.eq('original')
    spiral_all.loc[mask,'split']=spiral_all.loc[mask,'person_id'].map(person_split.set_index('person_id').split)
if spiral_all.groupby('person_id').split.nunique().gt(1).any():raise ValueError('Same person crosses fixed partitions. Provide a prespecified person-level split override; no silent split repair.')
feature_cols=[c for c in spiral_all if c.startswith(('test_','shape_'))]
files_table=spiral_all.copy()
spiral_all=spiral_all.groupby('person_id',sort=True).agg({**{c:'mean' for c in feature_cols},'label':'first','cohort':'first','split':'first'}).reset_index().rename(columns={'person_id':'subject_id'})
spiral_subjects=spiral_all.query("cohort=='original'").copy().reset_index(drop=True)
if spiral_subjects[SPIRAL_FEATURES].isna().all(axis=1).any():raise RuntimeError('Original-cohort subject has no usable geometry; correct source before training.')
if spiral_subjects.query("split=='development'").label.value_counts().min()<6:raise ValueError('Too few development persons per class for nested model comparison.')
shape_hash=sha256(OUTPUT/'tables/spiral_shape_features_all.csv')
spiral_subjects[['subject_id','label','split']].to_csv(OUTPUT/'tables/spiral_person_split.csv',index=False)
SPIRAL_SPEC=dict(name='spiral',subjects=spiral_subjects,features=SPIRAL_FEATURES,signature=shape_hash+sha256(OUTPUT/'tables/spiral_file_person_manifest.csv')+sha256(OUTPUT/'tables/spiral_person_split.csv')+hashlib.sha256(SPIRAL_SOURCE.encode()).hexdigest())
write_json(OUTPUT/'tables/spiral_identity_status.json',dict(verified=SPIRAL_IDENTITY_VERIFIED,evidence=SPIRAL_IDENTITY_EVIDENCE,person_mapping_supplied=bool(SPIRAL_IDENTITY_CSV),unverified_results_exploratory=not SPIRAL_IDENTITY_VERIFIED))
print(pd.crosstab(spiral_subjects.split,spiral_subjects.label))
print(task_qc.groupby(['cohort','task','status']).size())
print('Arc-length resampled shape features; acquisition confounding remains unresolved.')
complete('V9-15')


## MRI candidate inputs

Load this run’s normalized arrays and frozen embeddings. Create a fixed downsampled voxel baseline. All PCA/scaling is fitted inside training folds; no cohort-wide learned preprocessing.


In [ ]:
mri_subjects=pd.read_csv(PREVIOUS_OUTPUT/'tables/mri_subject_features.csv')
MRI_FEATURES=[c for c in mri_subjects if c.startswith('f_')]
assert len(mri_subjects)==43 and mri_subjects.subject_id.is_unique and len(MRI_FEATURES)==3072
arrays=[];voxel_rows=[];array_hashes={}
for row in mri_subjects.itertuples(index=False):
    path=PREVIOUS_OUTPUT/'mri_cache'/row.subject_id/'normalized.npy'
    if not path.is_file():raise FileNotFoundError(f'Preprocessed MRI array missing: {path}. Restore old mri_cache; do not rerun preprocessing unnecessarily.')
    a=np.load(path,allow_pickle=False);assert np.isfinite(a).all() and a.min()>=0 and a.max()<=1
    arrays.append(str(path));voxel_rows.append(voxel_vector(a));array_hashes[row.subject_id]=sha256(path)
    print('MRI cache verified:',row.subject_id,flush=True)
mri_subjects['array_path']=arrays
VOXEL_FEATURES=[f'voxel_{i:04d}' for i in range(len(voxel_rows[0]))]
mri_subjects=pd.concat([mri_subjects,pd.DataFrame(voxel_rows,columns=VOXEL_FEATURES)],axis=1)
write_json(OUTPUT/'tables/mri_array_hashes.json',array_hashes)
MRI_SPEC=dict(name='mri',subjects=mri_subjects,features=MRI_FEATURES,voxel_features=VOXEL_FEATURES,checkpoint=str(PREVIOUS_OUTPUT/'models/mri_encoder.pt'),signature=sha256(PREVIOUS_OUTPUT/'tables/mri_subject_features.csv')+hashlib.sha256(json.dumps(array_hashes,sort_keys=True).encode()).hexdigest()+sha256(PREVIOUS_OUTPUT/'models/mri_encoder.pt'))
SPATIAL_FEATURES=[f'spatial_{i:04d}' for i in range(len(spatial_vector(np.load(mri_subjects.array_path.iloc[0],allow_pickle=False))))]
spatial_rows=[spatial_vector(np.load(path,allow_pickle=False)) for path in mri_subjects.array_path]
mri_subjects=pd.concat([mri_subjects,pd.DataFrame(spatial_rows,columns=SPATIAL_FEATURES)],axis=1)
MRI_SPEC['subjects']=mri_subjects;MRI_SPEC['spatial_features']=SPATIAL_FEATURES
MRI_SPEC['signature']+='|atlas-free-spatial-v7|'+hashlib.sha256(np.asarray(spatial_rows,dtype=np.float32).tobytes()).hexdigest()
mri_subjects[['subject_id']+SPATIAL_FEATURES].to_csv(OUTPUT/'tables/mri_spatial_descriptors.csv',index=False)
write_json(OUTPUT/'tables/mri_representation_notes.json',dict(spatial_grid=[3,4,3],n_features=len(SPATIAL_FEATURES),anatomical_roi_volumes=False,atlas_used=False,foreground='normalized intensity>0',limitations='Contrast, brain size, age/sex and acquisition confounding unresolved; use only as candidate comparison.'))
qc=pd.read_csv(PREVIOUS_OUTPUT/'tables/mri_qc.csv');qc.to_csv(OUTPUT/'tables/mri_qc_previous.csv',index=False)
print('MRI QC approval: PENDING unless a documented independent review has been completed.')
print(pd.crosstab(mri_subjects.split,mri_subjects.label))

complete('V9-16')


## Frozen MedicalNet 3D medical-image encoder
Pinned checkpoint/source, strict backbone loading and no random-weight fallback. This is general medical pretraining, not a Parkinson-specific model. Pretraining overlap with Neurocon is not independently verified.


In [ ]:
require_mri_review()
MEDICAL_SOURCE='"""Frozen MedicalNet features: exploratory transfer, not a Parkinson diagnostic.\nMedicalNet source/weights attribution: Tencent/MedicalNet, MIT; Chen et al. Med3D (2019).\n"""\nfrom pathlib import Path\nimport hashlib,json,importlib.util\nimport numpy as np\n\ndef digest(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(2**20),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\ndef canonical_xyz(volume,grid):\n    import nibabel as nib\n    a=np.asarray(volume,np.float32)\n    if a.ndim!=3 or not np.isfinite(a).all() or a.min()<0 or a.max()>1:raise ValueError(\'Expected finite [0,1] normalized MRI.\')\n    if tuple(a.shape[::-1])!=tuple(grid[\'size\']):raise ValueError(\'Reference grid size mismatch.\')\n    affine=np.eye(4);affine[:3,:3]=np.asarray(grid[\'direction\']).reshape(3,3)@np.diag(grid[\'spacing\']);affine[:3,3]=grid[\'origin\']\n    affine=np.diag([-1,-1,1,1])@affine  # ITK physical LPS -> NIfTI RAS\n    transform=nib.orientations.ornt_transform(nib.orientations.io_orientation(affine),nib.orientations.axcodes2ornt((\'R\',\'A\',\'S\')))\n    return nib.orientations.apply_orientation(a.transpose(2,1,0),transform).copy()\n\ndef medical_input(volume,grid,shape=(96,112,96)):\n    import torch\n    from torch.nn import functional as F\n    xyz=canonical_xyz(volume,grid);mask=xyz>0\n    if mask.sum()<100:raise ValueError(\'Insufficient MRI foreground.\')\n    foreground=xyz[mask];sd=float(foreground.std())\n    if sd<1e-6:raise ValueError(\'Degenerate MRI contrast.\')\n    normalized=np.zeros_like(xyz);normalized[mask]=np.clip((foreground-foreground.mean())/sd,-5,5)\n    tensor=torch.from_numpy(normalized)[None,None]\n    return F.interpolate(tensor,size=shape,mode=\'trilinear\',align_corners=False)\n\ndef load_encoder(source,weights,device=\'cpu\'):\n    import torch\n    spec=importlib.util.spec_from_file_location(\'medicalnet_resnet_source\',str(source));module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)\n    encoder=module.resnet18(sample_input_D=96,sample_input_H=112,sample_input_W=96,num_seg_classes=2,shortcut_type=\'A\',no_cuda=(device==\'cpu\'))\n    encoder.conv_seg=torch.nn.Identity()\n    checkpoint=torch.load(weights,map_location=\'cpu\',weights_only=True)\n    raw=checkpoint.get(\'state_dict\',checkpoint)\n    if not isinstance(raw,dict):raise ValueError(\'Checkpoint has no tensor state dictionary.\')\n    state={k.removeprefix(\'module.\'):v for k,v in raw.items() if not k.removeprefix(\'module.\').startswith(\'conv_seg.\')}\n    expected=encoder.state_dict()\n    unexpected=set(state)-set(expected);missing=set(expected)-set(state)\n    critical_missing=[k for k in missing if not k.endswith(\'num_batches_tracked\')]\n    mismatched=[k for k in state if k in expected and tuple(state[k].shape)!=tuple(expected[k].shape)]\n    if unexpected or critical_missing or mismatched:raise RuntimeError(f\'Pretrained backbone mismatch: unexpected={unexpected}, missing={critical_missing}, shape={mismatched}\')\n    result=encoder.load_state_dict(state,strict=False)\n    if any(not k.endswith(\'num_batches_tracked\') for k in result.missing_keys) or result.unexpected_keys:raise RuntimeError(\'Incomplete backbone load.\')\n    for parameter in encoder.parameters():parameter.requires_grad=False\n    encoder.to(device).eval()\n    return encoder,dict(loaded_tensors=len(state),missing_only_bn_counters=sorted(missing),source_hash=digest(source),weights_hash=digest(weights))\n\ndef extract(volume,grid,encoder,device=\'cpu\'):\n    import torch\n    from torch.nn import functional as F\n    x=medical_input(volume,grid).to(device)\n    with torch.inference_mode():\n        z=encoder(x)\n        if z.ndim!=5 or z.shape[1]!=512 or not torch.isfinite(z).all():raise RuntimeError(\'Unexpected pretrained backbone output.\')\n        global_features=torch.cat([F.adaptive_avg_pool3d(z,1).flatten(1),F.adaptive_max_pool3d(z,1).flatten(1)],1)\n        regional_features=F.adaptive_avg_pool3d(z,(2,2,2)).flatten(1)\n    return global_features.cpu().numpy()[0].astype(\'float32\'),regional_features.cpu().numpy()[0].astype(\'float32\')\n'
# বাংলা: pretrained medical backbone; random initialization fallback নয়।
(OUTPUT/'pd_medicalnet.py').write_text(MEDICAL_SOURCE,encoding='utf-8')
import pd_medicalnet
importlib.reload(pd_medicalnet)
import urllib.request,shutil
MEDICAL_ASSETS=OUT/'tools/medicalnet';MEDICAL_ASSETS.mkdir(parents=True,exist_ok=True)
MEDICAL_SOURCE_PATH=MEDICAL_ASSETS/'medicalnet_resnet.py'
MEDICAL_WEIGHTS=MEDICAL_ASSETS/'resnet_18_23dataset.pth'
MEDICAL_LICENSE=MEDICAL_ASSETS/'MedicalNet_LICENSE'
MEDICAL_COMMIT='18c8bb6cd564eb1b964bffef1f4c2283f1ae6e7b'
MEDICAL_REVISION='758fe285bc8ab565eb4f9f965810f1d1a3f79491'
MEDICAL_WEIGHT_HASH='61224f9317fcce873366deb3703183e92cc47325b726b69691b33536244e10f4'
assets=[(MEDICAL_SOURCE_PATH,f'https://raw.githubusercontent.com/Tencent/MedicalNet/{MEDICAL_COMMIT}/models/resnet.py'),(MEDICAL_LICENSE,f'https://raw.githubusercontent.com/Tencent/MedicalNet/{MEDICAL_COMMIT}/LICENSE'),(MEDICAL_WEIGHTS,f'https://huggingface.co/TencentMedicalNet/MedicalNet-Resnet18/resolve/{MEDICAL_REVISION}/resnet_18_23dataset.pth')]
for path,url in assets:
    deadline_check()
    if not path.exists():
        temporary=path.with_name(path.name+'.partial')
        try:
            print('Downloading',path.name,flush=True)
            with urllib.request.urlopen(url,timeout=120) as src,temporary.open('wb') as dst:shutil.copyfileobj(src,dst)
            temporary.replace(path)
        finally:
            if temporary.exists():temporary.unlink()
if pd_medicalnet.digest(MEDICAL_WEIGHTS)!=MEDICAL_WEIGHT_HASH:raise RuntimeError('MedicalNet checkpoint hash mismatch; no unverified fallback.')
medical_encoder,medical_load_report=pd_medicalnet.load_encoder(MEDICAL_SOURCE_PATH,MEDICAL_WEIGHTS,DEVICE)
MEDICAL_PROVENANCE=dict(source_commit=MEDICAL_COMMIT,checkpoint_revision=MEDICAL_REVISION,license='MIT',PD_specific=False,pretraining_overlap_independently_verified=False,**medical_load_report)
write_json(OUTPUT/'tables/medicalnet_pretrained_provenance.json',MEDICAL_PROVENANCE)
print('MedicalNet backbone tensors loaded:',medical_load_report['loaded_tensors'])
complete('V9-MEDICALNET-ASSETS')


## Global and regional 3D representations
Full-volume 3D features supply candidates for the configured compact or expanded MRI comparison. Frozen extraction is label-independent; all supervised selection/PCA/scaling/threshold learning stays inside subject CV. Regional pooling bins are not anatomical ROI volumes.


In [ ]:
require_mri_review()
# বাংলা: সব patient-এর frozen embedding; কোনো diagnosis label encoder fit-এ নয়।
MEDICAL_PROTOCOL=dict(version='MRI8-integrated-1',reference_grid=PROTOCOL['reference_grid'],orientation='canonical RAS XYZ network spatial axes',input_shape=[96,112,96],normalization='own nonzero foreground z-score, clip [-5,5], background zero',global_pool='mean and max',regional_pool=[2,2,2],module_hash=sha256(OUTPUT/'pd_medicalnet.py'),assets=MEDICAL_PROVENANCE)
MEDICAL_PROTOCOL_HASH=hashlib.sha256(json.dumps(MEDICAL_PROTOCOL,sort_keys=True).encode()).hexdigest()
MEDICAL_CACHE=OUT/'mri_medicalnet_cache';MEDICAL_CACHE.mkdir(exist_ok=True)
global_rows=[];regional_rows=[]
for i,row in enumerate(MRI_SPEC['subjects'].itertuples(index=False),1):
    deadline_check();array_path=Path(row.array_path)
    medical_cache_key=hashlib.sha256((sha256(array_path)+MEDICAL_PROTOCOL_HASH).encode()).hexdigest()
    cache=MEDICAL_CACHE/(medical_cache_key+'.npz')
    if cache.exists():
        with np.load(cache,allow_pickle=False) as saved:g=saved['global_features'].copy();r=saved['regional_features'].copy()
    else:
        g,r=pd_medicalnet.extract(np.load(array_path,allow_pickle=False),PROTOCOL['reference_grid'],medical_encoder,DEVICE)
        np.savez_compressed(cache,global_features=g,regional_features=r)
    if g.shape!=(1024,) or r.shape!=(4096,) or not np.isfinite(g).all() or not np.isfinite(r).all():raise ValueError('Invalid MedicalNet feature vector.')
    global_rows.append(g);regional_rows.append(r)
    print(f'MedicalNet 3D MRI features {i}/{len(MRI_SPEC["subjects"])} completed',flush=True)
medical_encoder.cpu();del medical_encoder
if torch.cuda.is_available():torch.cuda.empty_cache()
MEDICAL_GLOBAL_FEATURES=[f'medical_global_{i:04d}' for i in range(1024)]
MEDICAL_REGIONAL_FEATURES=[f'medical_regional_{i:04d}' for i in range(4096)]
medical_table=pd.concat([MRI_SPEC['subjects'][['subject_id']].reset_index(drop=True),pd.DataFrame(global_rows,columns=MEDICAL_GLOBAL_FEATURES),pd.DataFrame(regional_rows,columns=MEDICAL_REGIONAL_FEATURES)],axis=1)
if not medical_table.subject_id.is_unique:raise ValueError('Duplicate MRI subjects.')
MRI_SPEC['subjects']=MRI_SPEC['subjects'].merge(medical_table,on='subject_id',validate='one_to_one')
MRI_SPEC['medical_global_features']=MEDICAL_GLOBAL_FEATURES;MRI_SPEC['medical_regional_features']=MEDICAL_REGIONAL_FEATURES
medical_table.to_csv(OUTPUT/'tables/mri_medicalnet_features.csv',index=False)
write_json(OUTPUT/'tables/medicalnet_feature_protocol.json',MEDICAL_PROTOCOL)
MRI_SPEC['signature']+='|'+MEDICAL_PROTOCOL_HASH+'|'+sha256(OUTPUT/'tables/mri_medicalnet_features.csv')
print('MRI candidate families:',list(grids('mri')))
complete('V9-MEDICALNET-FEATURES')


## Lock selection and evaluation protocol

Identical outer subject folds for all four candidates. Inner CV tunes each family and selects the family inside each outer training fold. A fresh development-only inner search locks the final model; no test-based model selection. Report per-family OOF comparisons as descriptive and the selection-procedure OOF estimate separately.


In [ ]:
SPECS={'voice':VOICE_SPEC,'spiral':SPIRAL_SPEC,'mri':MRI_SPEC}
for name,spec in SPECS.items():
    frame=spec['subjects'];assert set(frame.split)=={'development','test'}
    assert set(frame.query("split=='development'").subject_id).isdisjoint(frame.query("split=='test'").subject_id)
    frame[['subject_id','label','split']].to_csv(OUTPUT/'tables'/f'{name}_fixed_split.csv',index=False)
    print(name.upper(),'candidates:',', '.join(grids(name)))
write_json(OUTPUT/'tables/protocol_lock.json',dict(selection='Nested subject CV; inner balanced accuracy, ROC-AUC tie-break',test_status='Previously inspected holdouts; retrospective V8 integrated representation/threshold experiment',families={name:grids(name) for name in SPECS},threshold='Selected inside development CV; saved per subject/fold',calibrated=False,paired_patients=False,external_validation=False,publication_ready=False))

complete('V9-17')


## Duplicate and near-duplicate gate

Feature-space similarity is a screen only. Candidate pairs across either cohort partition stop for identity review. Confirmed duplicates must be corrected at source; do not authorize them as distinct.


In [ ]:
voice_pairs=numeric_candidates('voice',voice_obs.subject_id.tolist(),voice_obs.label.tolist(),voice_obs[VOICE_FEATURES].to_numpy(float),NEAR_VECTOR_RELATIVE_TOLERANCE)
voice_pairs=voice_pairs.query('subject_a != subject_b').drop_duplicates('pair_id')
spiral_feature_pairs=numeric_candidates('spiral',SPIRAL_SPEC['subjects'].subject_id.tolist(),SPIRAL_SPEC['subjects'].label.tolist(),SPIRAL_SPEC['subjects'][SPIRAL_FEATURES].to_numpy(float),NEAR_VECTOR_RELATIVE_TOLERANCE)
spiral_pairs=spiral_shape_candidates(SPIRAL_ROOT,load_trajectory)
mri_pairs=mri_candidates(MRI_SPEC['subjects'],MRI_NEAR_CORRELATION,MRI_NEAR_NRMSE)
pairs=pd.concat([voice_pairs,spiral_feature_pairs,spiral_pairs,mri_pairs],ignore_index=True).drop_duplicates('pair_id')
enforce_review(pairs,OUTPUT/'tables/duplicate_candidates.csv',DUPLICATE_REVIEW_CSV)
for name,spec in SPECS.items():
    subjects=spec['subjects'];x=voice_obs[VOICE_FEATURES] if name=='voice' else subjects[spec['features']]
    report=pd.DataFrame({'feature':x.columns,'missing_count':x.isna().sum().to_numpy(),'unique_values':x.nunique().to_numpy()})
    report.to_csv(OUTPUT/'tables'/f'{name}_feature_quality.csv',index=False)
    print(name.upper(),'| all-missing columns:',int((report.missing_count==len(x)).sum()),'| constant columns:',int((report.unique_values<=1).sum()))
print('Duplicate review gate PASSED. Similar values alone were not deleted.')

complete('V9-18')


## Five Voice model families

Logistic Regression, RBF-SVM, Random Forest and Gradient Boosting. Primary selection criterion is subject-level balanced accuracy; ROC-AUC is a tie-break. SVM uses a monotonic decision score, not a test-driven probability reversal.


In [ ]:
VOICE_RESULT=compare_development(VOICE_SPEC,CFG)

complete('V9-19')


## Five Spiral model families

Logistic Regression, RBF-SVM, Random Forest and Gradient Boosting. Shared subjects and folds. A high score does not resolve the separately audited acquisition confound.


In [ ]:
SPIRAL_RESULT=compare_development(SPIRAL_SPEC,CFG)

complete('V9-20')


## MRI model comparison after human QC

Voxel-SVM, frozen-feature LR, frozen-feature SVM and ResNet18 layer4 fine-tuning. CNN early stopping sees only an internal subject split of its current fit-input; after selecting epochs it reinitializes and refits all fit-input subjects. BatchNorm running statistics remain frozen. Fine-tuning may take longer than classical fitting.


In [ ]:
require_mri_review()
MRI_RESULT=compare_development(MRI_SPEC,CFG)

complete('V9-21')


## Selected model holdout evaluation

Only each development-locked winner is evaluated. These source holdouts were previously viewed, so results remain retrospective. Do not change families, thresholds or seeds after inspecting this output.


In [ ]:
RESULTS={'voice':VOICE_RESULT,'spiral':SPIRAL_RESULT,'mri':MRI_RESULT};TESTS={};METRICS={}
for name in ['voice','spiral','mri']:
    TESTS[name],METRICS[name]=holdout_evaluate(SPECS[name],RESULTS[name],CFG)
    publication_plots(TESTS[name],f'{name.upper()} | selected model | retrospective holdout | uncalibrated scores',OUTPUT/'figures'/f'{name}_selected_holdout')

complete('V9-22')


## Majority baseline and degenerate-prediction checks

Compare every selected model with a train-prior constant baseline. One-class predictions are flagged explicitly; high class-imbalance accuracy is not treated as successful discrimination.


In [ ]:
checks=[]
for name,spec in SPECS.items():
    prior=float(spec['subjects'].query("split=='development'").label.mean())
    pred=TESTS[name];base=score(pred.label,np.full(len(pred),prior))
    observed=METRICS[name];one_class=np.unique(predicted_labels(pred)).size==1
    checks.append(dict(modality=name,majority_baseline_accuracy=base['accuracy'],majority_baseline_balanced_accuracy=base['balanced_accuracy'],selected_accuracy=observed['accuracy'],selected_balanced_accuracy=observed['balanced_accuracy'],one_class_predictions=bool(one_class),status='DEGENERATE_CLASSIFICATION' if one_class else 'TWO_CLASS_PREDICTIONS'))
    if one_class:print(name.upper(),'| WARNING: all predictions belong to one class. This is not successful disease discrimination.')
pd.DataFrame(checks).to_csv(OUTPUT/'tables/majority_and_degeneracy_audit.csv',index=False)
print(pd.DataFrame(checks).round(4).to_string(index=False))

complete('V9-23')

# No threshold/family is changed using these inspected holdout results.
quality_rows=[]
for name,result in RESULTS.items():
    oof=result['oof'];test=TESTS[name]
    dev_metrics=score_frame(oof)
    test_metrics=METRICS[name]
    issues=[]
    if dev_metrics['balanced_accuracy']<=.5:issues.append('DEVELOPMENT_NOT_ABOVE_CHANCE')
    if np.unique(predicted_labels(test)).size==1:issues.append('HOLDOUT_CONSTANT_PREDICTION')
    if name=='mri' and not QC_APPROVED:issues.append('ANATOMICAL_QC_PENDING')
    if name=='spiral':
        issues.append('ACQUISITION_CONFOUNDING_UNRESOLVED')
        if not SPIRAL_IDENTITY_VERIFIED:issues.append('PERSON_IDENTITY_UNVERIFIED')
    quality_rows.append(dict(modality=name,selected_family=result['choice']['family'],development_selection_procedure_balanced_accuracy=dev_metrics['balanced_accuracy'],retrospective_holdout_balanced_accuracy=test_metrics['balanced_accuracy'],issues=issues,research_only=True,clinically_validated=False))
write_json(OUTPUT/'tables/model_quality_flags.json',quality_rows)
for row in quality_rows:print(row['modality'].upper(),':',', '.join(row['issues']) or 'No listed automatic flags; external validation still required.')
complete('V9-MODEL-QUALITY-FLAGS')


## Acquisition-only negative control

A fixed depth-1 tree tests whether collection-related variables alone predict labels. It is fitted on development subjects and evaluated once on the old holdout. Strong control performance is a warning; weak performance does not prove the handwriting features are confound-free.


In [ ]:
from sklearn.tree import DecisionTreeClassifier,export_text
# Reload the saved mapping: independent of mutable notebook globals.
spiral_identity_map=pd.read_csv(OUTPUT/'tables/spiral_file_person_manifest.csv',dtype=str)
required={'file_subject_id','person_id'}
if not required<=set(spiral_identity_map) or spiral_identity_map[list(required)].isna().any().any():raise ValueError('Invalid saved Spiral identity manifest.')
if not spiral_identity_map.file_subject_id.is_unique or set(spiral_identity_map.file_subject_id)!=set(acq_all.subject_id):raise ValueError('Saved identity mapping does not match acquisition records.')
mapped=acq_all.merge(spiral_identity_map,left_on='subject_id',right_on='file_subject_id',validate='one_to_one')
ACQ_FEATURES=['n_points','median_dt','zero_dt_fraction','negative_dt_count']
acq=mapped.query("cohort=='original'").groupby('person_id')[ACQ_FEATURES].mean().reset_index().rename(columns={'person_id':'subject_id'})
acq=acq.merge(spiral_subjects[['subject_id','label','split']],on='subject_id',validate='one_to_one')
control=Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),('model',DecisionTreeClassifier(max_depth=1,class_weight='balanced',random_state=SEED))])
dev=acq.query("split=='development'");test=acq.query("split=='test'")
control.fit(dev[ACQ_FEATURES],dev.label)
p=model_score(control,test[ACQ_FEATURES]);metrics=score(test.label,p)
pred=test[['subject_id','label']].copy();pred['probability_pd']=p
pred.to_csv(OUTPUT/'tables/spiral_acquisition_control_predictions.csv',index=False)
write_json(OUTPUT/'tables/spiral_acquisition_control_metrics.json',metrics)
rule=export_text(control.named_steps['model'],feature_names=ACQ_FEATURES)
(OUTPUT/'tables/spiral_acquisition_control_rule.txt').write_text(rule)
print(rule);print('Acquisition-only retrospective metrics:',metrics)
print('Association may include disease effects or protocol effects; no causal attribution established.')
joblib.dump(dict(model=control,features=ACQ_FEATURES),OUTPUT/'models/acquisition_negative_control.joblib')
publication_plots(pred,'Acquisition-only diagnostic | retrospective holdout',OUTPUT/'figures/spiral_acquisition_control')
complete('V9-24')


## Additional PD-only transport check

The extra cohort cannot provide specificity, balanced accuracy or binary ROC-AUC. Cross-cohort identity overlap is not independently verified; this is a sensitivity-only transport check, not external validation.


In [ ]:
extra=spiral_all.query("cohort=='pd_only'").copy()
bundle=SPIRAL_RESULT['bundle']
usable=usable_input_rows(bundle,extra[bundle['features']].to_numpy(float))
usable=pd.Series(usable,index=extra.index)
extra['score_pd']=np.nan;extra['prediction']=pd.Series(pd.NA,index=extra.index,dtype='Int64')
extra['status']=np.where(usable,'SCORED','ABSTAIN_NO_USABLE_GEOMETRY')
if usable.any():
    p=model_score(bundle['model'],extra.loc[usable,bundle['features']].to_numpy(float))
    extra.loc[usable,'score_pd']=p;extra.loc[usable,'prediction']=(p>=bundle['threshold']).astype(int)
extra[['subject_id','score_pd','prediction','status']].to_csv(OUTPUT/'tables/spiral_pd_only_transport.csv',index=False)
transport=dict(total_persons=len(extra),usable_persons=int(usable.sum()),abstained_persons=int((~usable).sum()),coverage=float(usable.mean()),
 usable_pd_decision_fraction=float(extra.loc[usable,'prediction'].astype(float).mean()) if usable.any() else None,
 decision_threshold=float(bundle['threshold']),excluded_ids=extra.loc[~usable,'subject_id'].tolist(),external_validation=False,identity_verified=SPIRAL_IDENTITY_VERIFIED,
 interpretation='PD-only usable-case decision fraction; no control specificity, no case-control accuracy')
write_json(OUTPUT/'tables/spiral_pd_only_transport_metrics.json',transport)
print(json.dumps(transport,indent=2));complete('V9-25')


## Standardized figures and actual CNN histories

ROC, PR, calibration diagnostics, confusion matrices, thresholds and four-model bars are generated from measured scores. CNN loss histories come from real training; no smoothing or fabricated epochs.


In [ ]:
for name,result in RESULTS.items():
    publication_plots(result['oof'],f'{name.upper()} | nested selection-procedure OOF | uncalibrated scores',OUTPUT/'figures'/f'{name}_selection_oof')
    frame=result['comparison'];fig,ax=plt.subplots(figsize=(max(8,1.35*len(frame)),5));x=np.arange(len(frame))
    ax.bar(x-.18,frame.accuracy,.36,label='OOF accuracy');ax.bar(x+.18,frame.balanced_accuracy,.36,label='OOF balanced accuracy')
    ax.set_xticks(x,frame.family,rotation=35,ha='right');ax.set_ylim(0,1.03);ax.set_title(name.upper()+' | per-family nested OOF (descriptive)');ax.legend();fig.tight_layout()
    for ext in ['png','pdf','svg']:fig.savefig(OUTPUT/'figures'/f'{name}_four_models.{ext}',dpi=300,bbox_inches='tight')
    plt.show();plt.close(fig)
# বাংলা: selected MRI family CNN না হলেও cached CNN fit-এর বাস্তব history সংরক্ষণ।
history_count=0
for path in sorted((OUTPUT/'fit_cache').glob('*.joblib')):
    item=joblib.load(path)
    if item['family']!='FineTune_ResNet18':continue
    model=item['model'];tag=path.stem[:12]
    early=pd.DataFrame(model.early_stopping_history_);refit=pd.DataFrame(model.refit_history_)
    early.to_csv(OUTPUT/'tables'/f'cnn_{tag}_early_stopping_history.csv',index=False)
    refit.to_csv(OUTPUT/'tables'/f'cnn_{tag}_refit_history.csv',index=False)
    history_count+=1
    if history_count==1:
        fig,axes=plt.subplots(1,2,figsize=(10,4));axes[0].plot(early.epoch,early.train_loss,label='Training');axes[0].plot(early.epoch,early.validation_loss,label='Internal validation');axes[0].legend();axes[0].set_title('First predefined CNN fit: early stopping')
        axes[1].plot(refit.epoch,refit.train_loss);axes[1].set_title('Refit on all fit-input subjects')
        for ax in axes:ax.set(xlabel='Epoch',ylabel='Weighted BCE loss')
        fig.tight_layout()
        for ext in ['png','pdf','svg']:fig.savefig(OUTPUT/'figures'/f'cnn_actual_history.{ext}',dpi=300)
        plt.show();plt.close(fig)
print('CNN fit histories exported:',history_count)

complete('V9-26')


## Actual classical learning curves

Fixed LR baseline learning curves use increasing development-subject subsets and fixed outer folds. Voice subsets include all recordings of selected training subjects, with subject-level validation. These are baseline curves, not CNN epoch curves or tuned-winner performance estimates.


In [ ]:
for name,spec in SPECS.items():
    dev=spec['subjects'].query("split=='development'").reset_index(drop=True);records=[]
    family='Frozen_ResNet_LR' if name=='mri' else 'LogisticRegression'
    for fold,(tr,va) in enumerate(StratifiedKFold(3,shuffle=True,random_state=SEED).split(dev,dev.label),1):
        pool=dev.iloc[tr].copy();val=dev.iloc[va].copy()
        for fraction in [.5,.75,1.]:
            if fraction==1.:subset=pool
            else:
                _,chosen=train_test_split(np.arange(len(pool)),test_size=fraction,random_state=SEED+fold,stratify=pool.label)
                subset=pool.iloc[chosen]
            fit=fit_candidate(spec,subset.subject_id.tolist(),family,{'C':1.},CFG,SEED+2000+fold)
            p_train=predict_subjects(spec,subset.subject_id.tolist(),fit);p_val=predict_subjects(spec,val.subject_id.tolist(),fit)
            records.append(dict(fold=fold,fraction=fraction,n=len(subset),train_balanced_accuracy=balanced_accuracy_score(subset.label,p_train>=.5),validation_balanced_accuracy=balanced_accuracy_score(val.label,p_val>=.5)))
    frame=pd.DataFrame(records);frame.to_csv(OUTPUT/'tables'/f'{name}_baseline_learning_curve.csv',index=False)
    mean=frame.groupby('fraction').mean(numeric_only=True);std=frame.groupby('fraction').std(numeric_only=True)
    fig,ax=plt.subplots(figsize=(6,4));x=mean['n'].to_numpy()
    ax.plot(x,mean.train_balanced_accuracy,'o-',label='Training');ax.plot(x,mean.validation_balanced_accuracy,'o-',label='Validation')
    ax.fill_between(x,mean.validation_balanced_accuracy-std.validation_balanced_accuracy,mean.validation_balanced_accuracy+std.validation_balanced_accuracy,alpha=.15)
    ax.set(xlabel='Mean training subjects',ylabel='Balanced accuracy',ylim=(0,1.03),title=name.upper()+' | fixed LR baseline');ax.legend();fig.tight_layout()
    for ext in ['png','pdf','svg']:fig.savefig(OUTPUT/'figures'/f'{name}_baseline_learning_curve.{ext}',dpi=300)
    plt.show();plt.close(fig);print(name.upper(),'| learning curve completed')

complete('V9-27')


## Spiral feature-set ablation

Same outer subject folds, fixed LR C=1. This is a descriptive development comparison, not an additional tuned winner or external validation.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
ablation=[]
frame=SPIRAL_SPEC['subjects'].query("split=='development'").reset_index(drop=True)
for fold,(tr,va) in enumerate(StratifiedKFold(3,shuffle=True,random_state=SEED).split(frame,frame.label),1):
    for kind,columns in [('shape_only',SPIRAL_FEATURES),('acquisition_sensitive_original',SPIRAL_CONSERVATIVE_FEATURES)]:
        model=make_pipeline('LogisticRegression',{'C':1.},SEED)
        y=frame.label.iloc[tr].to_numpy(int);counts=np.bincount(y,minlength=2);w=len(y)/(2*counts[y])
        model.fit(frame.iloc[tr][columns],y,model__sample_weight=w)
        p=model_score(model,frame.iloc[va][columns])
        ablation.append(dict(fold=fold,feature_set=kind,**score(frame.label.iloc[va],p)))
pd.DataFrame(ablation).to_csv(OUTPUT/'tables/spiral_feature_ablation.csv',index=False)
print(pd.DataFrame(ablation).groupby('feature_set')[['accuracy','balanced_accuracy','roc_auc']].mean().round(4))

complete('V9-28')


## Spiral task ablations
Static-only, dynamic-only and shape-without-step-CV feature sets, fixed LR, development folds only (descriptive).

In [ ]:
from sklearn.model_selection import StratifiedKFold
ablations=[]
dev=SPIRAL_SPEC['subjects'].query("split=='development'").reset_index(drop=True)
for fold,(tr,va) in enumerate(StratifiedKFold(3,shuffle=True,random_state=SEED).split(dev,dev.label),1):
    for variant,columns in [('static_only',[c for c in SPIRAL_FEATURES if c.startswith('shape_0_')]),('dynamic_only',[c for c in SPIRAL_FEATURES if c.startswith('shape_1_')]),('shape_without_step_cv',[c for c in SPIRAL_FEATURES if not c.endswith('step_cv')])]:
        model=make_pipeline('LogisticRegression',{'C':1.},SEED)
        y=dev.label.iloc[tr].to_numpy(int);counts=np.bincount(y,minlength=2);w=len(y)/(2*counts[y])
        model.fit(dev.iloc[tr][columns].to_numpy(float),y,model__sample_weight=w)
        prediction=model_score(model,dev.iloc[va][columns].to_numpy(float))
        ablations.append(dict(variant=variant,fold=fold,**score(dev.label.iloc[va],prediction)))
pd.DataFrame(ablations).to_csv(OUTPUT/'tables/spiral_task_ablations.csv',index=False)
print(pd.DataFrame(ablations).groupby('variant')[['balanced_accuracy','roc_auc']].mean())
complete('V9-SPIRAL-TASK-ABLATIONS')


## Uncertainty, model reload and target checks

Subject bootstrap intervals and exact binomial accuracy intervals accompany retrospective holdout results. Verify saved-model predictions. A fresh session does not reset the fact that holdout outcomes were already viewed. Report measured results even if below target.


In [ ]:
rows=[]
for name,result in RESULTS.items():
    bundle=joblib.load(OUTPUT/'models'/f'{name}_selected_bundle.joblib')
    ids=TESTS[name].subject_id.tolist();reproduced=predict_subjects(SPECS[name],ids,bundle)
    np.testing.assert_allclose(reproduced,TESTS[name].probability_pd.to_numpy(),rtol=1e-6,atol=1e-6)
    rows.append(dict(dataset=name,selected_family=result['choice']['family'],**METRICS[name],retrospective=True))
    print(name.upper(),'| model reload prediction verification PASSED')
    # বাংলা: patient bootstrap; recording/slice/triplet-কে independent sample ধরা নয়।
    rng=np.random.default_rng(SEED);bootstrap=[]
    y=TESTS[name].label.to_numpy(int);p=TESTS[name].probability_pd.to_numpy(float)
    for _ in range(BOOTSTRAP_REPEATS):
        ix=rng.integers(0,len(y),len(y))
        if np.unique(y[ix]).size<2:continue
        bootstrap.append(score(y[ix],p[ix],decision_thresholds(TESTS[name])[ix]))
    intervals={}
    for metric in ['accuracy','balanced_accuracy','sensitivity','specificity','roc_auc','average_precision','brier']:
        lo,hi=np.percentile([record[metric] for record in bootstrap],[2.5,97.5])
        intervals[metric]={'low':float(lo),'high':float(hi)}
    write_json(OUTPUT/'tables'/f'{name}_subject_bootstrap_intervals.json',dict(intervals=intervals,valid_resamples=len(bootstrap),status='Retrospective small-sample uncertainty; not external validation'))
results_table=pd.DataFrame(rows);results_table.to_csv(OUTPUT/'tables/selected_models_retrospective_summary.csv',index=False)
print(results_table[['dataset','selected_family','accuracy','balanced_accuracy','sensitivity','specificity','roc_auc','retrospective']].round(4).to_string(index=False))
write_json(OUTPUT/'tables/limitations.json',dict(publication_ready=False,clinical_validated=False,external_validation=False,paired_fusion=False,
 anatomical_qc='APPROVED' if QC_APPROVED else 'PENDING independently documented review',spiral_acquisition_confounding='UNRESOLVED',test_previously_viewed=True,
 scores_calibrated=False,accuracy_guaranteed=False,spiral_identity_verified=SPIRAL_IDENTITY_VERIFIED))

complete('V9-32')


## Publication figure style and empirical curves

All curves use measured predictions. Small samples produce stepped ROC curves and wide uncertainty. No invented epochs, smoothed scores or forced calibration points.


In [ ]:
from pathlib import Path
import sys,json,importlib.util,subprocess
PROJECT=Path('/kaggle/working/PD_Unpaired_Research_v9')
COMPARISON=PROJECT/'model_comparison'
TABLES=COMPARISON/'tables'
FIGURE_OUTPUT=PROJECT/'figure_review_v4'
FIGURE_OUTPUT.mkdir(parents=True,exist_ok=True)
BOOTSTRAP_REPEATS=1000
MRI_SUBJECT_LIMIT=None
MRI_PREVIEW_COUNT=2
for package in ['nibabel']:
    if importlib.util.find_spec(package) is None:subprocess.run([sys.executable,'-m','pip','install','--quiet',package],check=True)
if not TABLES.is_dir():raise FileNotFoundError('Saved V3 outputs missing. Set PROJECT to the restored project folder; the notebook file alone does not contain MRI arrays or prediction CSVs.')


FIGURE_MODULE='\nfrom pathlib import Path\nimport json,hashlib\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import MaxNLocator\nfrom sklearn.metrics import roc_curve,roc_auc_score,average_precision_score,confusion_matrix,PrecisionRecallDisplay,balanced_accuracy_score,accuracy_score,brier_score_loss\nCOLORS={\'voice\':\'#0072B2\',\'spiral\':\'#009E73\',\'mri\':\'#D55E00\'}\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for block in iter(lambda:f.read(2**20),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef style():\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':10,\'axes.titlesize\':11,\'axes.labelsize\':10,\n      \'axes.spines.top\':False,\'axes.spines.right\':False,\'axes.linewidth\':.8,\'lines.linewidth\':1.8,\n      \'figure.dpi\':110,\'savefig.dpi\':300,\'pdf.fonttype\':42,\'ps.fonttype\':42,\'svg.fonttype\':\'none\'})\n\ndef save(fig,stem,show=True):\n    stem=Path(stem);stem.parent.mkdir(parents=True,exist_ok=True)\n    for ext in [\'png\',\'pdf\',\'svg\']:fig.savefig(stem.with_suffix(\'.\'+ext),dpi=300,bbox_inches=\'tight\',facecolor=\'white\')\n    if show:plt.show()\n    plt.close(fig)\n\ndef validate(frame):\n    required={\'subject_id\',\'label\',\'probability_pd\'}\n    if not required<=set(frame):raise ValueError(\'Prediction CSV missing columns: \'+str(required-set(frame)))\n    if not frame.subject_id.is_unique:raise ValueError(\'Prediction rows must be unique subjects.\')\n    y=frame.label.to_numpy(int);p=frame.probability_pd.to_numpy(float)\n    if set(y)!={0,1} or not np.isfinite(p).all() or np.any((p<0)|(p>1)):raise ValueError(\'Invalid binary labels or scores.\')\n    return y,p\n\ndef roc_band(y,p,repeats=1000,seed=42):\n    # বাংলা: bootstrap band-এর interpolation শুধু uncertainty display; observed ROC বদলানো নয়।\n    rng=np.random.default_rng(seed);grid=np.linspace(0,1,101);samples=[];aucs=[]\n    neg=np.flatnonzero(y==0);pos=np.flatnonzero(y==1)\n    for _ in range(repeats):\n        ix=np.r_[rng.choice(neg,len(neg),replace=True),rng.choice(pos,len(pos),replace=True)]\n        f,t,_=roc_curve(y[ix],p[ix],drop_intermediate=False)\n        samples.append(np.interp(grid,f,t));aucs.append(roc_auc_score(y[ix],p[ix]))\n    return grid,np.percentile(samples,2.5,axis=0),np.percentile(samples,97.5,axis=0),np.percentile(aucs,[2.5,97.5])\n\ndef wilson(k,n):\n    z=1.95996398454;q=k/n;den=1+z*z/n\n    center=(q+z*z/(2*n))/den;delta=z*np.sqrt(q*(1-q)/n+z*z/(4*n*n))/den\n    return max(0,center-delta),min(1,center+delta)\n\ndef reliability_table(y,p,bins=3):\n    # বাংলা: equal-frequency bin; tied scores থাকলে bin জোর করে বাড়ানো হবে না।\n    edges=np.unique(np.quantile(p,np.linspace(0,1,bins+1)))\n    groups=np.zeros(len(p),int) if len(edges)<2 else np.searchsorted(edges[1:-1],p,side=\'right\')\n    rows=[]\n    for group in np.unique(groups):\n        ix=groups==group;n=int(ix.sum());k=int(y[ix].sum());lo,hi=wilson(k,n)\n        rows.append(dict(bin=int(group),n=n,mean_score=float(p[ix].mean()),observed_pd=float(k/n),ci_low=lo,ci_high=hi))\n    return pd.DataFrame(rows)\n\ndef performance_figures(frame,name,cohort,out,repeats=1000):\n    style();y,p=validate(frame);color=COLORS[name];out=Path(out)\n    cuts=frame.decision_threshold.to_numpy(float) if \'decision_threshold\' in frame else np.full(len(frame),.5)\n    decision=(p>=cuts).astype(int)\n    n=len(y);n0=int((y==0).sum());n1=int((y==1).sum());tag=f\'{name}_{cohort}\'\n    header=f\'{name.upper()} | {cohort.replace("_"," ")} | N={n} ({n0} Control, {n1} PD)\'\n    fpr,tpr,thresholds=roc_curve(y,p,drop_intermediate=False);auc=roc_auc_score(y,p)\n    grid,lo,hi,auc_ci=roc_band(y,p,repeats)\n    fig,ax=plt.subplots(figsize=(5.4,4.7),constrained_layout=True)\n    ax.fill_between(grid,lo,hi,color=color,alpha=.13,label=\'95% stratified subject-bootstrap band\')\n    ax.plot(fpr,tpr,\'o-\',color=color,markersize=3,label=f\'Observed ROC; AUC={auc:.3f}\')\n    ax.plot([0,1],[0,1],\'--\',color=\'.55\',linewidth=1,label=\'Chance\')\n    ax.set(xlim=(-.02,1.02),ylim=(-.02,1.02),xlabel=\'False positive rate\',ylabel=\'True positive rate\',title=header)\n    ax.legend(loc=\'lower right\',fontsize=8);ax.set_aspect(\'equal\',adjustable=\'box\')\n    save(fig,out/(tag+\'_roc\'))\n    pd.DataFrame({\'fpr\':fpr,\'tpr\':tpr,\'threshold\':thresholds}).to_csv(out/(tag+\'_roc_points.csv\'),index=False)\n    fig,ax=plt.subplots(figsize=(5.4,4.7),constrained_layout=True)\n    PrecisionRecallDisplay.from_predictions(y,p,ax=ax,color=color,name=\'Observed PR\',drawstyle=\'steps-post\')\n    ax.axhline(y.mean(),ls=\'--\',color=\'.55\',linewidth=1,label=f\'Prevalence={y.mean():.3f}\')\n    ax.set(xlim=(-.02,1.02),ylim=(-.02,1.02),xlabel=\'Recall\',ylabel=\'Precision\',title=header)\n    ax.legend(loc=\'lower left\',fontsize=8);save(fig,out/(tag+\'_pr\'))\n    tab=reliability_table(y,p,3);tab.to_csv(out/(tag+\'_reliability_bins.csv\'),index=False)\n    fig,axes=plt.subplots(1,2,figsize=(10.2,4.2),constrained_layout=True)\n    axes[0].plot([0,1],[0,1],\'--\',color=\'.55\',linewidth=1,label=\'Reference: observed = score\')\n    for i,row in enumerate(tab.itertuples(index=False)):\n        axes[0].errorbar([row.mean_score],[row.observed_pd],yerr=[[row.observed_pd-row.ci_low],[row.ci_high-row.observed_pd]],fmt=[\'o\',\'s\',\'^\'][i%3],color=[\'#0072B2\',\'#D55E00\',\'#009E73\'][i%3],markersize=6,capsize=4,label=f\'Quantile bin {i+1}: n={row.n}\')\n    axes[0].set(xlim=(-.02,1.02),ylim=(-.02,1.02),xlabel=\'Mean uncalibrated PD-class score\',ylabel=\'Observed PD fraction\',title=\'Reliability diagnostic; 95% binomial intervals\')\n    axes[0].legend(loc=\'lower right\',fontsize=8)\n    # বাংলা: সীমিত score range-এ সব subject আলাদাভাবে দেখানো; score value বদলানো নয়।\n    for label,offset in [(0,0),(1,1)]:\n        axes[1].scatter(p[y==label],np.full((y==label).sum(),offset),color=[\'#0072B2\',\'#D55E00\'][label],s=35,alpha=.8,label=[\'Control\',\'PD\'][label])\n    pad=max(.02,float(np.ptp(p))*.15);left=max(0,float(p.min())-pad);right=min(1,float(p.max())+pad)\n    if left==right:left=max(0,left-.01);right=min(1,right+.01)\n    axes[1].set(xlim=(left,right),ylim=(-.4,1.4),yticks=[0,1],yticklabels=[\'Control\',\'PD\'],xlabel=\'Actual PD-class score (zoomed axis)\',title=f\'Score range: {p.min():.4f}–{p.max():.4f}\')\n    fig.suptitle(header+\' | scores are not calibrated clinical risk\')\n    save(fig,out/(tag+\'_reliability_and_scores\'))\n    cm=confusion_matrix(y,decision,labels=[0,1]);normal=cm/cm.sum(axis=1,keepdims=True)\n    fig,axes=plt.subplots(1,2,figsize=(8.4,3.8),constrained_layout=True)\n    for ax,a,title in [(axes[0],cm,\'Counts at recorded cuts (default 0.50)\'),(axes[1],normal,\'Within-class fractions\')]:\n        ax.imshow(a,cmap=\'Blues\',vmin=0,vmax=cm.max() if a is cm else 1)\n        for i in range(2):\n            for j in range(2):ax.text(j,i,str(cm[i,j]) if a is cm else f\'{a[i,j]:.0%}\',ha=\'center\',va=\'center\',color=\'white\' if a[i,j]>(cm.max()/2 if a is cm else .5) else \'black\')\n        ax.set(xticks=[0,1],yticks=[0,1],xticklabels=[\'Control\',\'PD\'],yticklabels=[\'Control\',\'PD\'],xlabel=\'Predicted class\',ylabel=\'True class\',title=title)\n    fig.suptitle(header);save(fig,out/(tag+\'_confusion\'))\n    return dict(name=name,cohort=cohort,n=n,accuracy=float(accuracy_score(y,decision)),balanced_accuracy=float(balanced_accuracy_score(y,decision)),roc_auc=float(auc),auc_ci_low=float(auc_ci[0]),auc_ci_high=float(auc_ci[1]),average_precision=float(average_precision_score(y,p)),brier=float(brier_score_loss(y,p)),specificity=float(normal[0,0]),score_min=float(p.min()),score_max=float(p.max()),nonempty_reliability_bins=len(tab))\n\ndef loss_history_figures(table_dir,out,preview=2):\n    style();paths=sorted(Path(table_dir).glob(\'cnn_*_early_stopping_history.csv\'));records=[]\n    for k,path in enumerate(paths):\n        early=pd.read_csv(path);refit_path=path.with_name(path.name.replace(\'_early_stopping_history\',\'_refit_history\'))\n        if not refit_path.exists():continue\n        refit=pd.read_csv(refit_path);tag=path.name.removesuffix(\'_early_stopping_history.csv\')\n        fig,axes=plt.subplots(1,2,figsize=(10.5,4.1),constrained_layout=True)\n        axes[0].plot(early.epoch,early.train_loss,\'o-\',label=\'Training\',markersize=4)\n        axes[0].plot(early.epoch,early.validation_loss,\'s-\',label=\'Internal validation\',markersize=4)\n        selected=int(refit.epoch.max());axes[0].axvline(selected,color=\'.4\',ls=\'--\',lw=1,label=f\'Selected epoch={selected}\')\n        axes[0].set_title(\'Train-only early stopping\');axes[0].legend(fontsize=8)\n        axes[1].plot(refit.epoch,refit.train_loss,\'o-\',color=\'#0072B2\',markersize=6,label=\'Refit training\')\n        axes[1].set_title(\'Refit on all fit-input subjects\')\n        if len(refit)==1:axes[1].annotate(\'Only 1 measured epoch\',xy=(1,refit.train_loss.iloc[0]),xytext=(.5,.85),textcoords=\'axes fraction\',ha=\'center\',arrowprops={\'arrowstyle\':\'->\',\'color\':\'.4\'})\n        for ax,history in zip(axes,[early,refit]):\n            ax.set(xlim=(.5,max(1,int(history.epoch.max()))+.5),ylim=(0,None),xlabel=\'Epoch\',ylabel=\'Weighted BCE loss\')\n            last_epoch=max(1,int(history.epoch.max()));ax.set_xticks(np.arange(1,last_epoch+1,max(1,int(np.ceil(last_epoch/10)))));ax.grid(alpha=.15)\n        fig.suptitle(tag+\' | actual measured values; no smoothing\')\n        save(fig,Path(out)/tag,show=k<preview)\n        records.append(dict(fit=tag,early_stopping_epochs=len(early),refit_epochs=len(refit),selected_epoch=selected))\n    return pd.DataFrame(records)\n\ndef panel(a,axis,index):\n    # বাংলা: canonical RAS array; transpose + origin lower-এ neurological display।\n    return np.take(a,index,axis=axis).T\n\ndef mri_montage(array,mask,spacing,tag,stem,reference_mask=None,show=False):\n    style();a=np.asarray(array,np.float32);mask=np.asarray(mask,bool)\n    if a.ndim!=3 or a.shape!=mask.shape or not mask.any() or not np.isfinite(a).all():raise ValueError(\'Invalid MRI/mask for QC display.\')\n    lo,hi=np.percentile(a[mask],[1,99.5]);display=np.clip((a-lo)/max(float(hi-lo),1e-8),0,1);display[~mask]=0\n    fig,axes=plt.subplots(3,5,figsize=(13,8.8),layout=\'constrained\',facecolor=\'black\')\n    views=[(\'Axial\',2,(\'L\',\'R\')),(\'Coronal\',1,(\'L\',\'R\')),(\'Sagittal\',0,(\'P\',\'A\'))]\n    rows=[]\n    for r,(name,axis,sides) in enumerate(views):\n        active=np.flatnonzero(mask.sum(axis=tuple(i for i in range(3) if i!=axis))>0)\n        indices=np.rint(np.linspace(active[0],active[-1],7)[1:-1]).astype(int)\n        others=[i for i in range(3) if i!=axis];sx,sy=float(spacing[others[0]]),float(spacing[others[1]])\n        for c,index in enumerate(indices):\n            ax=axes[r,c];sl=panel(display,axis,index);ms=panel(mask,axis,index)\n            extent=(0,sl.shape[1]*sx,0,sl.shape[0]*sy)\n            ax.imshow(sl,cmap=\'gray\',vmin=0,vmax=1,origin=\'lower\',extent=extent,aspect=\'equal\',interpolation=\'nearest\')\n            xx=(np.arange(sl.shape[1])+.5)*sx;yy=(np.arange(sl.shape[0])+.5)*sy\n            if ms.any() and not ms.all():ax.contour(xx,yy,ms,levels=[.5],colors=[\'#64e572\'],linewidths=.45)\n            if reference_mask is not None:\n                rm=panel(reference_mask,axis,index)\n                if rm.any() and not rm.all():ax.contour(xx,yy,rm,levels=[.5],colors=[\'#ff6666\'],linewidths=.4)\n            ax.set_title(f\'{name} | slice {index}\',color=\'white\',fontsize=8);ax.axis(\'off\')\n            ax.text(.02,.5,sides[0],transform=ax.transAxes,color=\'white\',fontsize=8);ax.text(.98,.5,sides[1],transform=ax.transAxes,color=\'white\',fontsize=8,ha=\'right\')\n            rows.append(dict(view=name,array_axis=axis,slice_index=int(index)))\n    fig.suptitle(tag+\' | canonical RAS; neurological display; anatomical review pending\',color=\'white\',fontsize=11)\n    fig.text(.5,.005,\'Green: subject brain mask. Red (if present): fixed MNI brain mask. Display only; image data and model inputs unchanged.\',ha=\'center\',color=\'white\',fontsize=8)\n    stem=Path(stem);stem.parent.mkdir(parents=True,exist_ok=True)\n    for ext in [\'png\',\'pdf\',\'svg\']:fig.savefig(stem.with_suffix(\'.\'+ext),dpi=250,bbox_inches=\'tight\',facecolor=\'black\')\n    if show:plt.show()\n    plt.close(fig);return rows\n'
(FIGURE_OUTPUT/'pd_figure_review.py').write_text(FIGURE_MODULE,encoding='utf-8')
sys.path.insert(0,str(FIGURE_OUTPUT))
import pd_figure_review
importlib.reload(pd_figure_review)
from pd_figure_review import *

complete('V9-33')


In [ ]:
input_hashes={};summaries=[]
for name in ['voice','spiral','mri']:
    for cohort,suffix in [('retrospective_holdout','selected_retrospective_holdout'),('nested_selection_oof','selection_procedure_oof')]:
        path=TABLES/f'{name}_{suffix}.csv'
        if not path.is_file():raise FileNotFoundError(f'Missing prediction table: {path}')
        input_hashes[str(path)]=sha256(path)
        frame=pd.read_csv(path)
        result=performance_figures(frame,name,cohort,FIGURE_OUTPUT/'performance',BOOTSTRAP_REPEATS)
        summaries.append(result)
        print(name.upper(),cohort,'| AUC:',round(result['roc_auc'],4),'| specificity:',round(result['specificity'],4),'| score range:',round(result['score_min'],4),round(result['score_max'],4))
pd.DataFrame(summaries).to_csv(FIGURE_OUTPUT/'figure_metrics_and_support.csv',index=False)

complete('V9-34')


## MRI anatomical review panels

QC montages were generated before model training and remain in the figure folder.


In [ ]:
print('MRI QC panels already exported:',FIGURE_OUTPUT/'mri_qc')

complete('V9-36')


## Classical learning curves

Measured fold means and standard deviations; markers preserve single points.


In [ ]:
style()
for name in ['voice','spiral','mri']:
    path=TABLES/f'{name}_baseline_learning_curve.csv'
    if not path.is_file():raise FileNotFoundError(f'Missing baseline learning curve: {path}')
    d=pd.read_csv(path);mean=d.groupby('fraction').mean(numeric_only=True);sd=d.groupby('fraction').std(numeric_only=True)
    fig,ax=plt.subplots(figsize=(6.2,4.4),constrained_layout=True);x=mean['n'].to_numpy()
    for column,label,marker,color in [('train_balanced_accuracy','Training','o','#0072B2'),('validation_balanced_accuracy','Validation','s','#D55E00')]:
        ax.errorbar(x,mean[column],yerr=sd[column].fillna(0),fmt=marker+'-',color=color,capsize=4,markersize=6,label=label+' (fold mean ± SD)')
    ax.set(xlabel='Mean training subjects across folds',ylabel='Balanced accuracy',ylim=(-.03,1.03),title=name.upper()+' | measured fixed-LR baseline learning curve')
    ax.set_xticks(x,[f'{v:.1f}' for v in x]);ax.grid(alpha=.15);ax.legend(fontsize=8)
    save(fig,FIGURE_OUTPUT/'learning_curves'/f'{name}_baseline')

complete('V9-37')


## Candidate families: training vs outer-validation balanced accuracy

In [ ]:
for name in SPECS:
    folds=pd.read_csv(OUTPUT/'tables'/f'{name}_candidate_outer_folds.csv')
    mean=folds.groupby('family')[['training_balanced_accuracy','balanced_accuracy']].mean()
    sd=folds.groupby('family')[['training_balanced_accuracy','balanced_accuracy']].std().fillna(0)
    fig,ax=plt.subplots(figsize=(max(8,1.35*len(mean)),5.2),constrained_layout=True);x=np.arange(len(mean))
    ax.bar(x-.18,mean.training_balanced_accuracy,.36,yerr=sd.training_balanced_accuracy,capsize=3,label='Training',color='#0072B2')
    ax.bar(x+.18,mean.balanced_accuracy,.36,yerr=sd.balanced_accuracy,capsize=3,label='Outer validation',color='#D55E00')
    ax.set(xticks=x,xticklabels=[v.replace('_',' ') for v in mean.index],ylabel='Balanced accuracy',ylim=(0,1.15),title=name.upper()+' | candidate families; fold mean ± SD')
    ax.tick_params(axis='x',labelrotation=35);ax.legend();save(fig,FIGURE_OUTPUT/'candidate_comparison'/name)

complete('V19-CANDIDATE-FIGURES')


# Part R — Revision analyses (reviewer response)

Everything below re-evaluates the **complete** selection procedure; nothing reuses earlier exported predictions.

| Cell | What it does | Reviewer points |
|---|---|---|
| R0 | Configuration (FULL / FAST mode, parallel workers, time budgets) | — |
| R1 | Library: nested selection, permutation engine, bootstrap, progress, checkpoints | — |
| R2 | Repeated nested cross-validation of the full pipeline | 1, 21, 22 |
| R3 | Full-pipeline permutation test (parallel, resumable) | 1, 25, 38 |
| R4 | Uncertainty, baselines, Brier reference | 8, 24, 31, 39 |
| R5 | Training vs validation gap | 40 |
| R6 | Spiral acquisition-controlled analysis | 3, 35 |
| R7 | MRI QC-pass sensitivity + QC-vs-diagnosis test | 4, 33, 34 |
| R8 | Voice identifier audit + exclusion sensitivity | 2, 12, 13 |
| R9 | Recording- vs subject-level split (leakage demonstration) | 49 |
| R10 | Publication figures | — |
| R11 | Tables, `paper_numbers.md`, final archive | — |
| R12 | Research demo app (Streamlit, independent models) + reload check | 46–47 |

**Run with _Save Version → Save & Run All (Commit)_** so that power or network interruptions on your side cannot stop it.

In [ ]:
# R0 configuration
import os, time, copy, shutil, tempfile, zipfile, json
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, brier_score_loss
from scipy.stats import binom, fisher_exact, mannwhitneyu

MODE = 'FULL'                                   # 'FULL' (paper quality) or 'FAST' (quick check)
N_REPEATS = 10 if MODE == 'FULL' else 5         # repeated outer partitions
N_PERM = 200 if MODE == 'FULL' else 60          # full-pipeline permutations per modality
PERM_BUDGET_MIN = ({'voice': 90, 'spiral': 150, 'mri': 60} if MODE == 'FULL'
                   else {'voice': 15, 'spiral': 20, 'mri': 10})   # hard wall-clock caps
N_JOBS = max(1, min(4, os.cpu_count() or 1))    # parallel workers for permutations
N_BOOT = 2000
REPEAT_SEEDS = [SEED + 7000 + r for r in range(N_REPEATS)]
PERM_SEED0 = SEED + 20000                       # permutation b uses seed PERM_SEED0 + b (resumable, order-free)

REV = OUTPUT / 'revision'; FIG = REV / 'figures'
for p in (REV, FIG): p.mkdir(parents=True, exist_ok=True)
SCRATCH = Path('/kaggle/working/_scratch') if Path('/kaggle/working').exists() else Path(tempfile.gettempdir()) / '_scratch'
SCRATCH.mkdir(parents=True, exist_ok=True)
print(f'MODE={MODE} | repeats={N_REPEATS} | permutations={N_PERM} | workers={N_JOBS} | output={REV}')


In [ ]:
# R1 library
from IPython.display import display

def _scratch_cfg(seed):
    """CFG copy with a new outer seed and a throw-away fit cache (keeps the disk clean)."""
    cfg = dict(CFG); cfg['seed'] = int(seed)
    tmp = Path(tempfile.mkdtemp(prefix='rev_', dir=SCRATCH))
    for sub in ('fit_cache', 'tables', 'models'): (tmp / sub).mkdir(parents=True, exist_ok=True)
    cfg['output'] = str(tmp)
    return cfg, tmp

def nested_selection_oof(spec, seed):
    """Outer 3-fold subject-grouped CV. Inside every outer-training set each family is tuned
    (parameters + threshold) by inner CV and the family is chosen by inner BA.
    Returns out-of-fold scores of the *whole selection procedure*."""
    cfg, tmp = _scratch_cfg(seed)
    try:
        dev = spec['subjects'].query("split=='development'").reset_index(drop=True)
        cands = grids(spec['name'])
        p = np.full(len(dev), np.nan); t = np.full(len(dev), np.nan); chosen = []
        for fold, (tr, va) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(dev, dev.label), 1):
            tr_ids, va_ids = dev.subject_id.iloc[tr].tolist(), dev.subject_id.iloc[va].tolist()
            assert set(tr_ids).isdisjoint(va_ids), 'participant leakage'
            winners = [tune_family(spec, tr_ids, fam, prm, cfg, seed + fold)[0] for fam, prm in cands.items()]
            win = max(winners, key=choose_key)
            bundle = fit_candidate(spec, tr_ids, win['family'], win['param'], cfg, seed + fold + 500)
            p[va] = predict_subjects(spec, va_ids, bundle); t[va] = win['threshold']; chosen.append(win['family'])
        y = dev.label.to_numpy(int)
        return dict(y=y, p=p, t=t, chosen=chosen, acc=float(np.mean((p >= t) == y)),
                    ba=float(balanced_accuracy_score(y, p >= t)), auc=float(roc_auc_score(y, p)))
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

def permuted_spec(spec, perm_seed):
    """Shuffle development labels at participant level (voice recordings inherit their identifier's label)."""
    rng = np.random.default_rng(perm_seed); s = copy.deepcopy(spec)
    m = s['subjects'].split.eq('development')
    s['subjects'].loc[m, 'label'] = rng.permutation(s['subjects'].loc[m, 'label'].to_numpy())
    if 'observations' in s:
        lab = s['subjects'].set_index('subject_id').label
        s['observations']['label'] = s['observations'].subject_id.map(lab).astype(int)
    s['signature'] = f"{s['signature']}|perm{perm_seed}"
    return s

def _one_perm(spec, b, seed):
    r = nested_selection_oof(permuted_spec(spec, PERM_SEED0 + b), seed)
    return dict(b=b, ba=r['ba'], auc=r['auc'])

class Progress:
    """Live one-line progress bar (interactive) + periodic log lines (commit logs)."""
    def __init__(self, label, total, budget_min):
        self.label, self.total, self.budget, self.t0 = label, total, budget_min, time.time()
        self.h = display(f'{label}: starting…', display_id=True); self.last_log = 0
    def update(self, done):
        el = (time.time() - self.t0) / 60; rate = el / max(done, 1)
        eta = min(self.budget - el, rate * (self.total - done)); pct = 100 * done / self.total
        bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
        msg = f'[{bar}] {pct:5.1f}% | {self.label} {done}/{self.total} | {el:.1f} min elapsed | ~{max(eta, 0):.0f} min left'
        if self.h is not None: self.h.update(msg)
        if done - self.last_log >= 10 or done == self.total: print(msg, flush=True); self.last_log = done

def run_permutations(name, spec, seed, n_perm, budget_min, ckpt):
    """Parallel, resumable full-pipeline permutations. Each permutation b has its own seed,
    so results do not depend on order, number of workers or restarts."""
    done = pd.read_csv(ckpt) if Path(ckpt).exists() else pd.DataFrame(columns=['b', 'ba', 'auc'])
    todo = [b for b in range(n_perm) if b not in set(done.b.astype(int))]
    prog = Progress(f'{name.upper()} permutations', n_perm, budget_min); prog.update(len(done)); t0 = time.time()
    rows = done.to_dict('records')
    try:
        from joblib import Parallel, delayed
        batch = N_JOBS * 2
        while todo and (time.time() - t0) / 60 < budget_min:
            chunk, todo = todo[:batch], todo[batch:]
            res = Parallel(n_jobs=N_JOBS, backend='loky')(delayed(_one_perm)(spec, b, seed) for b in chunk)
            rows += res; pd.DataFrame(rows).to_csv(ckpt, index=False); prog.update(len(rows))
    except Exception as e:                                   # fall back to sequential if workers fail
        print('Parallel execution unavailable, continuing sequentially:', repr(e)[:200])
        for b in todo:
            if (time.time() - t0) / 60 >= budget_min: break
            rows.append(_one_perm(spec, b, seed)); pd.DataFrame(rows).to_csv(ckpt, index=False); prog.update(len(rows))
    out = pd.DataFrame(rows).drop_duplicates('b').sort_values('b')
    if len(out) < n_perm: print(f'{name.upper()}: time budget reached at B={len(out)} (reported honestly).')
    return out

def hanley_mcneil(auc, n_pos, n_neg):
    q1, q2 = auc / (2 - auc), 2 * auc ** 2 / (1 + auc)
    se = np.sqrt((auc * (1 - auc) + (n_pos - 1) * (q1 - auc ** 2) + (n_neg - 1) * (q2 - auc ** 2)) / (n_pos * n_neg))
    return max(0., auc - 1.96 * se), min(1., auc + 1.96 * se)

def stratified_bootstrap(y, p, t, B=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); y, p = np.asarray(y), np.asarray(p); t = np.broadcast_to(t, p.shape)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]; a, b = [], []
    for _ in range(B):
        i = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]
        a.append(roc_auc_score(y[i], p[i])); b.append(balanced_accuracy_score(y[i], p[i] >= t[i]))
    return np.percentile(a, [2.5, 97.5]).tolist(), np.percentile(b, [2.5, 97.5]).tolist()

def cached(path, fn):
    """Return a cached DataFrame if it exists, otherwise compute and store it (resume support)."""
    path = Path(path)
    if path.exists(): print('loaded from checkpoint:', path.name); return pd.read_csv(path)
    df = fn(); df.to_csv(path, index=False); return df

print('Revision library ready.')


In [ ]:
# R2 repeated nested cross-validation of the complete selection procedure
rep_oof = {}
def _repeat_rows():
    rows = []
    for name, spec in SPECS.items():
        prog = Progress(f'{name.upper()} repeated CV', len(REPEAT_SEEDS), 999)
        for k, seed in enumerate(REPEAT_SEEDS, 1):
            r = nested_selection_oof(spec, seed); rep_oof[(name, seed)] = r
            rows.append(dict(modality=name, seed=seed, accuracy=r['acc'], balanced_accuracy=r['ba'],
                             roc_auc=r['auc'], chosen_families='|'.join(r['chosen'])))
            prog.update(k)
    return pd.DataFrame(rows)
rep = cached(REV / 'repeated_nested_cv.csv', _repeat_rows)
# OOF of the first partition is needed for permutation / bootstrap even when resuming
for name, spec in SPECS.items():
    if (name, REPEAT_SEEDS[0]) not in rep_oof: rep_oof[(name, REPEAT_SEEDS[0])] = nested_selection_oof(spec, REPEAT_SEEDS[0])

rep_summary = rep.groupby('modality').agg(n_repeats=('seed', 'size'),
    BA_mean=('balanced_accuracy', 'mean'), BA_sd=('balanced_accuracy', 'std'), BA_min=('balanced_accuracy', 'min'), BA_max=('balanced_accuracy', 'max'),
    AUC_mean=('roc_auc', 'mean'), AUC_sd=('roc_auc', 'std'), AUC_min=('roc_auc', 'min'), AUC_max=('roc_auc', 'max')).reset_index()
fam = (rep.assign(family=rep.chosen_families.str.split('|')).explode('family')
          .groupby(['modality', 'family']).size().rename('times_selected').reset_index())
rep_summary.to_csv(REV / 'repeated_nested_cv_summary.csv', index=False); fam.to_csv(REV / 'family_selection_frequency.csv', index=False)
display(rep_summary.round(3)); display(fam)


In [ ]:
# R3 full-pipeline permutation test (parallel + resumable)
perm_all, perm_rows = [], []
for name, spec in SPECS.items():
    null = run_permutations(name, spec, REPEAT_SEEDS[0], N_PERM, PERM_BUDGET_MIN[name], REV / f'perm_{name}.csv')
    obs = rep_oof[(name, REPEAT_SEEDS[0])]; B = len(null)
    perm_all.append(null.assign(modality=name))
    perm_rows.append(dict(modality=name, permutations=B, min_attainable_p=1 / (1 + B),
                          observed_BA=obs['ba'], observed_AUC=obs['auc'],
                          null_BA_mean=null.ba.mean(), null_BA_sd=null.ba.std(), null_AUC_mean=null.auc.mean(),
                          p_BA=(1 + (null.ba >= obs['ba']).sum()) / (1 + B), p_AUC=(1 + (null.auc >= obs['auc']).sum()) / (1 + B)))
perm_all = pd.concat(perm_all); perm_all.to_csv(REV / 'full_pipeline_permutations.csv', index=False)
perm_summary = pd.DataFrame(perm_rows); perm_summary.to_csv(REV / 'full_pipeline_permutation_summary.csv', index=False)
display(perm_summary.round(4))


In [ ]:
# R4 uncertainty, baselines and Brier reference
unc = []
for name, spec in SPECS.items():
    r = rep_oof[(name, REPEAT_SEEDS[0])]; y, p, t = r['y'], r['p'], r['t']
    auc_ci, ba_ci = stratified_bootstrap(y, p, t); hm = hanley_mcneil(r['auc'], int(y.sum()), int((1 - y).sum())); prev = float(y.mean())
    unc.append(dict(set='development_nested_OOF', modality=name, n=len(y), AUC=r['auc'], AUC_boot_lo=auc_ci[0], AUC_boot_hi=auc_ci[1],
                    AUC_HM_lo=hm[0], AUC_HM_hi=hm[1], BA=r['ba'], BA_boot_lo=ba_ci[0], BA_boot_hi=ba_ci[1],
                    Brier=float(brier_score_loss(y, np.clip(p, 0, 1))), Brier_prevalence_reference=prev * (1 - prev)))
    h = TESTS[name]; yh = h.label.to_numpy(int); m = METRICS[name]
    hmh = hanley_mcneil(m['roc_auc'], int(yh.sum()), int((1 - yh).sum()))
    prev_dev = float(spec['subjects'].query("split=='development'").label.mean())
    maj = max(yh.mean(), 1 - yh.mean()); k = int(round(m['accuracy'] * len(yh)))
    unc.append(dict(set='retrospective_holdout', modality=name, n=len(yh), AUC=m['roc_auc'], AUC_HM_lo=hmh[0], AUC_HM_hi=hmh[1],
                    BA=m['balanced_accuracy'], Brier=m['brier'], Brier_prevalence_reference=prev_dev * (1 - prev_dev),
                    majority_baseline_accuracy=maj, P_ge_observed_if_majority_rate=float(binom.sf(k - 1, len(yh), maj))))
unc = pd.DataFrame(unc); unc.to_csv(REV / 'uncertainty_auc_ba_brier.csv', index=False); display(unc.round(3))


In [ ]:
# R5 training vs validation gap
train_gap = pd.concat([pd.read_csv(OUTPUT / 'tables' / f'{n}_candidate_outer_folds.csv')
                         .groupby('family')[['training_accuracy', 'accuracy', 'training_balanced_accuracy', 'balanced_accuracy']]
                         .mean().assign(modality=n).reset_index() for n in SPECS])
train_gap.to_csv(REV / 'training_vs_outer_accuracy.csv', index=False); display(train_gap.round(3))


In [ ]:
# R6 spiral: acquisition-controlled analysis
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

sp = (SPIRAL_SPEC['subjects'].query("split=='development'")[['subject_id', 'label'] + SPIRAL_FEATURES]
      .merge(acq[['subject_id'] + ACQ_FEATURES], on='subject_id', validate='one_to_one').reset_index(drop=True))
sp['acq_stratum'] = np.where(sp.median_dt <= 8, 'median_dt<=8', 'median_dt>8')
strata = pd.crosstab(sp.acq_stratum, sp.label).rename(columns={0: 'control', 1: 'PD'}); strata.to_csv(REV / 'spiral_acquisition_strata_development.csv')
strata_all = pd.crosstab(np.where(acq.median_dt <= 8, '<=8 ms', '>8 ms'), acq.label).rename(columns={0: 'control', 1: 'PD'})
strata_all.to_csv(REV / 'spiral_acquisition_strata_all40.csv'); display(strata, strata_all)

def _lr(): return Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler()), ('m', LogisticRegression(max_iter=4000, class_weight='balanced'))])
rows = []
for seed in REPEAT_SEEDS:
    for fold, (tr, va) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(sp, sp.label), 1):
        a, b = sp.iloc[tr], sp.iloc[va]; med = a[SPIRAL_FEATURES].median()
        imp = SimpleImputer(strategy='median').fit(a[ACQ_FEATURES]); A_tr, A_va = imp.transform(a[ACQ_FEATURES]), imp.transform(b[ACQ_FEATURES])
        reg = LinearRegression().fit(A_tr, a[SPIRAL_FEATURES].fillna(med))       # fitted on the training fold only
        R_tr = a[SPIRAL_FEATURES].fillna(med).to_numpy() - reg.predict(A_tr); R_va = b[SPIRAL_FEATURES].fillna(med).to_numpy() - reg.predict(A_va)
        for variant, Xtr, Xva in [('acquisition_only', a[ACQ_FEATURES], b[ACQ_FEATURES]), ('shape_only', a[SPIRAL_FEATURES], b[SPIRAL_FEATURES]),
                                  ('shape_plus_acquisition', a[SPIRAL_FEATURES + ACQ_FEATURES], b[SPIRAL_FEATURES + ACQ_FEATURES]),
                                  ('shape_residualised_on_acquisition', R_tr, R_va)]:
            pr = _lr().fit(Xtr, a.label).predict_proba(Xva)[:, 1]
            rows.append(dict(seed=seed, fold=fold, variant=variant, ba=balanced_accuracy_score(b.label, pr >= .5),
                             auc=roc_auc_score(b.label, pr) if b.label.nunique() == 2 else np.nan))
acq_cv = pd.DataFrame(rows); acq_cv.to_csv(REV / 'spiral_acquisition_controlled_cv.csv', index=False)
acq_sum = acq_cv.groupby('variant')[['ba', 'auc']].agg(['mean', 'std']); acq_sum.to_csv(REV / 'spiral_acquisition_controlled_summary.csv'); display(acq_sum.round(3))


In [ ]:
# R7 MRI: QC-pass sensitivity and QC-vs-diagnosis test
qc = pd.read_csv(OUT / 'tables/mri_qc.csv').merge(MRI_SPEC['subjects'][['subject_id', 'label']], on='subject_id')
qc['flag'] = qc['priority_review'].astype(str).str.lower().isin(['true', '1'])
qc_tab = pd.crosstab(qc.flag, qc.label).rename(columns={0: 'control', 1: 'PD'}); qc_tab.to_csv(REV / 'mri_qc_flag_by_label.csv')
qc_tests = dict(fisher_p=float(fisher_exact(qc_tab.to_numpy())[1]),
                dice_mannwhitney_p=float(mannwhitneyu(qc.query('label==1').dice, qc.query('label==0').dice).pvalue) if 'dice' in qc else np.nan)
mri_pass = copy.deepcopy(MRI_SPEC); mri_pass['subjects'] = mri_pass['subjects'][mri_pass['subjects'].subject_id.isin(qc.loc[~qc.flag, 'subject_id'])].reset_index(drop=True)
mri_pass['signature'] += '|qc_pass_only'
dev_counts = mri_pass['subjects'].query("split=='development'").label.value_counts().to_dict()
sens = []
if min(dev_counts.values()) >= 3:
    for seed in REPEAT_SEEDS:
        r = nested_selection_oof(mri_pass, seed); sens.append(dict(analysis='MRI_QC_pass_only', seed=seed, n=len(r['y']), balanced_accuracy=r['ba'], roc_auc=r['auc']))
sens = pd.DataFrame(sens)
print('QC flag x diagnosis:\n', qc_tab, '\n', qc_tests, '\nQC-pass development counts:', dev_counts)


In [ ]:
# R8 voice: identifier audit and exclusion sensitivity
vid = voice_raw.groupby('subject_id').agg(recordings=('name', 'size'), status=('status', 'first')).reset_index()
vid.to_csv(REV / 'voice_identifier_audit.csv', index=False)
EXCL = vid.query('recordings != 6').subject_id.tolist()       # identifiers with an atypical number of recordings
print(f'{len(vid)} identifiers (PD {int(vid.status.sum())}, control {int((1 - vid.status).sum())}); atypical: {EXCL}')
v2 = copy.deepcopy(VOICE_SPEC)
v2['subjects'] = v2['subjects'][~v2['subjects'].subject_id.isin(EXCL)].reset_index(drop=True)
v2['observations'] = v2['observations'][~v2['observations'].subject_id.isin(EXCL)].reset_index(drop=True)
v2['signature'] += '|excl_atypical'
for seed in REPEAT_SEEDS:
    r = nested_selection_oof(v2, seed); sens = pd.concat([sens, pd.DataFrame([dict(analysis='voice_excluding_' + '+'.join(EXCL), seed=seed, n=len(r['y']), balanced_accuracy=r['ba'], roc_auc=r['auc'])])])
sens.to_csv(REV / 'sensitivity_analyses.csv', index=False)
sens_sum = sens.groupby('analysis')[['balanced_accuracy', 'roc_auc']].agg(['mean', 'std', 'count']); display(sens_sum.round(3))


In [ ]:
# R9 recording- vs subject-level splitting (voice leakage demonstration)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
obs = VOICE_SPEC['observations'].reset_index(drop=True)
Xv, yv, gv = obs[VOICE_FEATURES].to_numpy(float), obs.label.to_numpy(int), obs.subject_id.to_numpy()
MODELS = {'RBF-SVM': lambda s: Pipeline([('s', StandardScaler()), ('m', SVC(C=10, probability=True, random_state=s))]),
          'Logistic regression': lambda s: Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=4000))]),
          'Random forest': lambda s: RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=s, n_jobs=-1)}
rows = []
for seed in REPEAT_SEEDS:
    schemes = {'Recording-level 80/20 split': [train_test_split(np.arange(len(yv)), test_size=.2, stratify=yv, random_state=seed)],
               'Recording-level 3-fold CV': list(StratifiedKFold(3, shuffle=True, random_state=seed).split(Xv, yv)),
               'Subject-level 3-fold CV': list(StratifiedGroupKFold(3, shuffle=True, random_state=seed).split(Xv, yv, gv))}
    for scheme, splits in schemes.items():
        for mname, mk in MODELS.items():
            p = np.full(len(yv), np.nan)
            for tr, te in splits: p[te] = mk(seed).fit(Xv[tr], yv[tr]).predict_proba(Xv[te])[:, 1]
            ok = ~np.isnan(p)
            rows.append(dict(seed=seed, scheme=scheme, model=mname, accuracy=np.mean((p[ok] >= .5) == yv[ok]),
                             BA=balanced_accuracy_score(yv[ok], p[ok] >= .5), AUC=roc_auc_score(yv[ok], p[ok]),
                             shared_identifiers=len(set(gv[splits[0][0]]) & set(gv[splits[0][1]]))))
leak = pd.DataFrame(rows); leak.to_csv(REV / 'voice_leakage_demonstration.csv', index=False)
leak_sum = leak.groupby(['scheme', 'model'])[['accuracy', 'BA', 'AUC', 'shared_identifiers']].agg(['mean', 'std']).round(3)
leak_sum.to_csv(REV / 'voice_leakage_summary.csv'); display(leak_sum)


In [ ]:
# R10 publication figures
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42})
COL = {'voice': '#0072B2', 'spiral': '#009E73', 'mri': '#D55E00'}; LBL = {'voice': 'Voice', 'spiral': 'Spiral drawing', 'mri': 'T1 MRI'}
def save(fig, stem):
    for ext in ('png', 'pdf'): fig.savefig(FIG / f'{stem}.{ext}', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), constrained_layout=True)
for ax, metric, title in [(axes[0], 'balanced_accuracy', 'Balanced accuracy'), (axes[1], 'roc_auc', 'ROC-AUC')]:
    for k, n in enumerate(SPECS):
        v = rep.query('modality==@n')[metric].to_numpy()
        ax.scatter(k + np.random.default_rng(k).uniform(-.12, .12, len(v)), v, color=COL[n], alpha=.75, s=22); ax.hlines(v.mean(), k - .25, k + .25, color='k', lw=2)
    ax.axhline(.5, ls='--', color='grey', lw=1); ax.set_xticks(range(3), [LBL[n] for n in SPECS]); ax.set_ylim(0, 1.02)
    ax.set_title(f'{title} — {N_REPEATS} repeated nested CV partitions')
save(fig, 'fig2_repeated_nested_cv')

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), constrained_layout=True)
for ax, n in zip(axes, SPECS):
    row = perm_summary.query('modality==@n').iloc[0]
    ax.hist(perm_all.query('modality==@n').ba, bins=20, color=COL[n], alpha=.6)
    ax.axvline(row.observed_BA, color='k', lw=2, label=f'observed {row.observed_BA:.2f}')
    ax.set_title(f'{LBL[n]}: p = {row.p_BA:.3f} (B = {int(row.permutations)})'); ax.set_xlabel('BA under permuted labels'); ax.legend(fontsize=8)
save(fig, 'fig3_permutation_null')

fig, ax = plt.subplots(figsize=(5, 3.4), constrained_layout=True)
for lab, nm, c in [(0, 'Control', '#555555'), (1, 'PD', '#009E73')]:
    v = acq.query('label==@lab').median_dt
    ax.scatter(v, lab + np.random.default_rng(lab).uniform(-.15, .15, len(v)), color=c, s=24, label=f'{nm} (n={len(v)})')
ax.axvline(8, ls='--', color='k', lw=1); ax.set_yticks([0, 1], ['Control', 'PD']); ax.set_xlabel('Median inter-sample interval (ms)')
ax.set_title('Spiral: acquisition timing vs diagnosis'); ax.legend(fontsize=8)
save(fig, 'fig4_spiral_acquisition_confound')

order = ['acquisition_only', 'shape_only', 'shape_plus_acquisition', 'shape_residualised_on_acquisition']
fig, ax = plt.subplots(figsize=(6.5, 3.4), constrained_layout=True); x = np.arange(4)
ax.bar(x - .18, acq_sum.loc[order, ('ba', 'mean')], .36, yerr=acq_sum.loc[order, ('ba', 'std')], capsize=3, label='BA', color='#009E73')
ax.bar(x + .18, acq_sum.loc[order, ('auc', 'mean')], .36, yerr=acq_sum.loc[order, ('auc', 'std')], capsize=3, label='AUC', color='#7FC4AE')
ax.axhline(.5, ls='--', color='grey'); ax.set_xticks(x, ['Acquisition\nonly', 'Shape\nonly', 'Shape +\nacquisition', 'Shape resid.\non acquisition'])
ax.set_ylim(0, 1.1); ax.legend(); ax.set_title('Spiral: acquisition-controlled repeated CV')
save(fig, 'fig5_spiral_acquisition_controlled')

fig, ax = plt.subplots(figsize=(8, 3.8), constrained_layout=True); schemes = list(dict.fromkeys(leak.scheme)); w = .26
for k, m in enumerate(MODELS):
    g = leak.query('model==@m').groupby('scheme').accuracy; v = [g.mean()[s] for s in schemes]; e = [g.std()[s] for s in schemes]
    bars = ax.bar(np.arange(3) + (k - 1) * w, v, w, yerr=e, capsize=3, label=m)
    for xx, yy in zip(np.arange(3) + (k - 1) * w, v): ax.text(xx, yy + .012, f'{yy:.2f}', ha='center', fontsize=8)
ax.axhline(float(yv.mean()), ls='--', color='grey', lw=1, label=f'majority rate ({yv.mean():.3f})')
ax.set_xticks(range(3), [s.replace(' 3-fold', '\n3-fold').replace(' 80/20', '\n80/20') for s in schemes]); ax.set_ylim(.5, 1.02)
ax.set_ylabel(f'Accuracy (mean ± SD, {N_REPEATS} seeds)'); ax.set_title('Voice: same features and models, different splitting unit'); ax.legend(fontsize=8)
save(fig, 'fig6_voice_leakage')


In [ ]:
# R11 tables, numbers file and archive
def tex(df, name, caption):
    df.to_csv(REV / f'{name}.csv', index=False)
    try: (REV / f'{name}.tex').write_text(df.to_latex(index=False, float_format='%.3f', caption=caption, label='tab:' + name))
    except Exception as e: print('LaTeX export skipped:', name, e)
t3 = rep_summary.merge(perm_summary[['modality', 'permutations', 'p_BA', 'p_AUC', 'null_BA_mean']], on='modality') \
     .merge(unc.query("set=='development_nested_OOF'")[['modality', 'AUC_boot_lo', 'AUC_boot_hi', 'BA_boot_lo', 'BA_boot_hi']], on='modality')
tex(t3, 'table3_repeated_cv_permutation', 'Repeated nested CV and full-pipeline permutation test.')
tex(unc.query("set=='retrospective_holdout'").dropna(axis=1, how='all'), 'table5_holdout', 'Retrospective holdout (descriptive).')
tex(acq_sum.reset_index().set_axis(['variant', 'BA_mean', 'BA_sd', 'AUC_mean', 'AUC_sd'], axis=1), 'table7_spiral_acquisition', 'Spiral acquisition-controlled CV.')
f3 = lambda x: 'NA' if pd.isna(x) else f'{x:.3f}'
L = [f'# Numbers for the manuscript ({MODE} mode, {N_REPEATS} repeats, generated {time.strftime("%Y-%m-%d %H:%M")})\n']
for _, r in t3.iterrows():
    L.append(f"- **{LBL[r.modality]}**: BA {f3(r.BA_mean)} ± {f3(r.BA_sd)} (range {f3(r.BA_min)}–{f3(r.BA_max)}); AUC {f3(r.AUC_mean)} ± {f3(r.AUC_sd)} "
             f"(range {f3(r.AUC_min)}–{f3(r.AUC_max)}); bootstrap AUC CI {f3(r.AUC_boot_lo)}–{f3(r.AUC_boot_hi)}; permutation B={int(r.permutations)}, "
             f"null BA {f3(r.null_BA_mean)}, p_BA={f3(r.p_BA)}, p_AUC={f3(r.p_AUC)} → {'beyond chance' if r.p_BA < .05 else 'NOT beyond chance'}")
for title, df in [('Family selection', fam), ('Holdout', unc.query("set=='retrospective_holdout'").dropna(axis=1, how='all').round(3)),
                  ('Training vs outer', train_gap.round(3)), ('Spiral strata (dev)', strata), ('Spiral strata (all 40)', strata_all),
                  ('Spiral acquisition-controlled', acq_sum.round(3)), ('Sensitivity', sens_sum.round(3)), ('Voice leakage', leak_sum)]:
    L.append(f'\n## {title}\n' + df.to_string())
L.append(f"\n## MRI QC\n{qc_tab.to_string()}\nFisher p={qc_tests['fisher_p']:.3f}; Dice Mann-Whitney p={qc_tests['dice_mannwhitney_p']:.3f}; QC-pass dev counts {dev_counts}")
(REV / 'paper_numbers.md').write_text('\n'.join(L)); print('\n'.join(L))

ARCH = OUTPUT / 'PD_revision_package.zip'
with zipfile.ZipFile(ARCH, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in [REV, OUTPUT / 'tables', OUTPUT / 'figures']:
        for p in Path(folder).rglob('*'):
            if p.is_file() and p.suffix in {'.csv', '.json', '.md', '.tex', '.png', '.pdf', '.txt'}: z.write(p, p.relative_to(OUT))
shutil.rmtree(SCRATCH, ignore_errors=True)
print(f'Archive: {ARCH} ({ARCH.stat().st_size / 1e6:.1f} MB)')
complete('REVISION-COMPLETE')


## R12. Research demo app (Streamlit) — independent models only
Packages the three locked, development-selected models (voice, spiral, MRI) with a small Streamlit app.
The prototype-alignment model was removed from the study and is **not** included.
Output: `PD_Research_App.zip` (run locally with `pip install -r requirements.txt` and `streamlit run app.py`).

In [ ]:
# R12 export research app
import shutil, importlib.metadata, ast
APP = OUTPUT / 'research_app'; shutil.rmtree(APP, ignore_errors=True); APP.mkdir(parents=True)
def _copy(src, name=None):
    src = Path(src)
    if src.exists(): shutil.copy2(src, APP / (name or src.name)); return True
    print('  (skipped, not found):', src); return False
for f in ['pd_four_model_core.py', 'pd_spiral_features.py', 'pd_input_validation.py', 'pd_medicalnet.py']: _copy(OUTPUT / f)
_copy(OUT / 'pd_preprocessing_core.py'); _copy(OUT / 'mri_preprocessing_cell.py')
_copy(MEDICAL_SOURCE_PATH, 'medicalnet_resnet.py'); _copy(MEDICAL_WEIGHTS, 'resnet_18_23dataset.pth'); _copy(MEDICAL_LICENSE, 'MedicalNet_LICENSE')
for f in ['medicalnet_pretrained_provenance.json', 'medicalnet_feature_protocol.json', 'model_quality_flags.json',
          'mri_preprocessing_protocol.json', 'asset_hashes.json']: _copy(OUTPUT / 'tables' / f) or _copy(OUT / 'tables' / f)
for n in SPECS: _copy(OUTPUT / 'models' / f'{n}_selected_bundle.joblib')
write_json(APP / 'workflow_status.json', dict(
    research_only=True, scores_calibrated=False, external_validation=False, patient_fusion=False,
    mri_anatomical_qc_approved=bool(QC_APPROVED), spiral_identity_verified=bool(SPIRAL_IDENTITY_VERIFIED),
    spiral_acquisition_confounded=True, holdout='retrospective, previously inspected',
    revision_findings=dict(voice='not beyond permutation null', spiral='explained by acquisition timing', mri='no signal')))
write_json(APP / 'input_config.json', dict(mri_shape=list(np.load(MRI_SPEC['subjects'].array_path.iloc[0]).shape),
    spiral_resampling_points=RESAMPLING_POINTS, protocol_hash=PROTOCOL_HASH, reference_grid=PROTOCOL['reference_grid']))
write_json(APP / 'feature_schema.json', {n: list(SPECS[n]['features']) for n in SPECS})
for n, spec in SPECS.items():
    pd.DataFrame(columns=(['subject_id'] if n == 'voice' else []) + list(spec['features'])).to_csv(APP / f'{n}_feature_template.csv', index=False)

app_source = r'''"""Parkinson's research demo — independent voice / spiral / MRI models (research use only)."""
from pathlib import Path
import io, json, tempfile
import numpy as np, pandas as pd, joblib, streamlit as st
from pd_four_model_core import model_score, voxel_vector, spatial_vector, usable_input_rows
from pd_preprocessing_core import load_trajectory
from pd_spiral_features import shape_features
from pd_input_validation import validate_mri_metadata, validate_feature_frame

ROOT = Path(__file__).parent
cfg = json.loads((ROOT / "input_config.json").read_text())
schema = json.loads((ROOT / "feature_schema.json").read_text())
status = json.loads((ROOT / "workflow_status.json").read_text())

st.set_page_config(page_title="Parkinson Research Demo", page_icon="🧠", layout="centered")
st.title("Parkinson's disease — research demo")
st.error("RESEARCH ONLY — not a medical device. Scores are uncalibrated. In the accompanying study, voice was not "
         "distinguishable from chance, spiral discrimination was explained by acquisition timing, and MRI showed no signal.")
with st.expander("Study status flags"):
    st.json(status)

name = st.selectbox("Modality", ["voice", "spiral", "mri"],
                    format_func={"voice": "Voice (acoustic-feature CSV)", "spiral": "Spiral drawing (raw TXT or feature CSV)",
                                 "mri": "T1 MRI (normalised NPY + metadata JSON)"}.get)
template = ROOT / f"{name}_feature_template.csv"
if name != "mri" and template.exists():
    st.download_button("Download input template", template.read_bytes(), file_name=template.name)
upload = st.file_uploader("Upload input", type=["npy"] if name == "mri" else (["csv", "txt"] if name == "spiral" else ["csv"]))
meta = st.file_uploader("Matching inference_metadata.json", type=["json"]) if name == "mri" else None

@st.cache_resource
def bundle_for(n):
    return joblib.load(ROOT / f"{n}_selected_bundle.joblib")

@st.cache_resource
def medicalnet():
    from pd_medicalnet import load_encoder
    enc, _ = load_encoder(ROOT / "medicalnet_resnet.py", ROOT / "resnet_18_23dataset.pth", "cpu")
    return enc

if upload is not None and st.button("Run research prediction", type="primary"):
    try:
        bundle = bundle_for(name); model = bundle["model"]
        st.caption(f"Locked model: **{bundle['family']}**, threshold {float(bundle['threshold']):.2f}")
        if name == "mri":
            if meta is None: raise ValueError("MRI requires the matching inference_metadata.json.")
            validate_mri_metadata(json.loads(meta.getvalue()), upload.getvalue(), cfg["protocol_hash"])
            a = np.load(io.BytesIO(upload.getvalue()), allow_pickle=False)
            if list(a.shape) != cfg["mri_shape"] or not np.isfinite(a).all() or a.min() < 0 or a.max() > 1:
                raise ValueError("Expected the trained MRI grid with finite normalised values in [0, 1].")
            fam = bundle["family"]
            if fam.startswith("MedicalNet_"):
                from pd_medicalnet import extract
                g, r = extract(a, cfg["reference_grid"], medicalnet(), "cpu")
                p = float(model_score(model, (g if "Global" in fam else r)[None, :])[0])
            elif fam.startswith("Spatial_"): p = float(model_score(model, spatial_vector(a)[None, :])[0])
            else: p = float(model_score(model, voxel_vector(a)[None, :])[0])
        else:
            if name == "spiral" and upload.name.lower().endswith(".txt"):
                with tempfile.TemporaryDirectory() as d:
                    path = Path(d) / "trajectory.txt"; path.write_bytes(upload.getvalue())
                    feats, quality = shape_features(load_trajectory(path), cfg["spiral_resampling_points"])
                frame = pd.DataFrame([feats]); st.write("Task quality", quality)
            else:
                frame = pd.read_csv(upload)
            missing = set(schema[name]) - set(frame)
            if missing: raise ValueError(f"Missing features: {sorted(missing)}")
            if name == "voice" and ("subject_id" not in frame or frame.subject_id.nunique() != 1):
                raise ValueError("Voice CSV needs one subject_id shared by all of that person's recordings.")
            frame, values = validate_feature_frame(frame, schema[name], name)
            if name == "spiral":
                if len(values) != 1: raise ValueError("Provide exactly one subject feature row.")
                if not usable_input_rows(bundle, values).all(): raise ValueError("ABSTAIN: selected task has no usable geometry.")
                p = float(model_score(model, values)[0])
            else:
                p = float(model_score(model, values).mean())     # mean over the person's recordings
        thr = float(bundle["threshold"])
        c1, c2 = st.columns(2)
        c1.metric("Uncalibrated PD score", f"{p:.3f}")
        c2.metric("Research class", "PD" if p >= thr else "Control")
        st.caption("Not a clinical probability. No combined multimodal score is produced.")
    except Exception as e:
        st.error(str(e))
'''
ast.parse(app_source)
(APP / 'app.py').write_text(app_source, encoding='utf-8')
req = [f"{p}=={importlib.metadata.version(p).split('+')[0]}" for p in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'joblib', 'torch']]
(APP / 'requirements.txt').write_text('\n'.join(req + ['streamlit>=1.40,<2', 'nibabel>=5,<6', 'SimpleITK>=2.3']) + '\n')
(APP / 'README.md').write_text(
    "# Parkinson's research demo (independent models)\n\nResearch use only; not a medical device; scores are uncalibrated.\n\n"
    "```\npip install -r requirements.txt\nstreamlit run app.py\n```\n\n"
    "- Voice: CSV with `subject_id` + 22 acoustic features (template included); recordings of one person are averaged.\n"
    "- Spiral: raw semicolon TXT from the UCI tablet protocol, or one feature row.\n"
    "- MRI: `normalized.npy` + `inference_metadata.json` produced by the notebook's preprocessing protocol.\n")

# reload check: the exported bundles reproduce the notebook's holdout scores
for n in SPECS:
    b = joblib.load(APP / f'{n}_selected_bundle.joblib'); ids = TESTS[n].subject_id.tolist()
    np.testing.assert_allclose(predict_subjects(SPECS[n], ids, b), TESTS[n].probability_pd, atol=1e-6)
print('Exported model bundles reproduce stored holdout scores: PASSED')
shutil.make_archive(str(OUTPUT / 'PD_Research_App'), 'zip', APP)
with zipfile.ZipFile(ARCH, 'a', zipfile.ZIP_DEFLATED) as z: z.write(OUTPUT / 'PD_Research_App.zip', 'PD_Research_App.zip')
print('App:', OUTPUT / 'PD_Research_App.zip', '| added to', ARCH.name)
print('\nALL DONE. Download from the version Output tab:\n  -', ARCH.relative_to(Path('/kaggle/working')) if str(ARCH).startswith('/kaggle/working') else ARCH,
      '\n  -', (OUTPUT / 'PD_Research_App.zip'))
complete('R12-APP-EXPORTED')
